# 逐行学习代码

从原始文件中良性和恶性各抽取100张图用于测试和学习流程代码

In [2]:
import os
import random
import shutil

random.seed(300)

source_root = r"G:\04PhD\02Project\03Urine\rawdata\UR_SingleCell"
target_root = r"G:\04PhD\03Study\05SelfLearn\shilab-binary-classifier\examples\data\UR_SingleCell_mini"

classes = ["benign", "malignant"]
n_per_class = 50  # 每类抽取50张

for cls in classes:
    src_dir = os.path.join(source_root, cls)
    dst_dir = os.path.join(target_root, cls)
    os.makedirs(dst_dir, exist_ok=True)

    files = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]
    selected = random.sample(files, n_per_class)

    for f in selected:
        shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))

    print(f"{cls}: 复制了 {len(selected)} 张")


benign: 复制了 50 张
malignant: 复制了 50 张


# 学习参数设定

In [1]:
import os
import torch
from datetime import datetime
from shilab_classifier import (
    split_dataset,
    set_seed,
    prepare_cross_validation,
    train_all_models,
    evaluate_all_models,
    SUPPORTED_MODELS
)
# ========================================================================
# 第0步：配置参数
# ========================================================================
# 路径配置
seed = 42
set_seed(seed)  # 设置随机种子以确保结果可复现
RAW_DATA_DIR = r"G:\04PhD\03Study\05SelfLearn\shilab-binary-classifier\examples\data\UR_SingleCell_mini"  # 原始数据目录
BASE_OUTPUT_DIR = r"G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell"  # 基础输出目录    
# 原始良性测试集路径
ORIGINAL_BENIGN_TEST_DIR = r"H:\limr\project\urine\result\classification_model\UR_SingleCell_source\split_data\test\benign"

# 创建时间戳文件夹（可选，用于区分不同的实验）
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# EXPERIMENT_DIR = os.path.join(BASE_OUTPUT_DIR, f"experiment_{timestamp}")
EXPERIMENT_DIR = os.path.join(BASE_OUTPUT_DIR, f"")
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
target_benign_test_dir = os.path.join(EXPERIMENT_DIR, "split_data", "test", "benign")
# 各阶段路径
benign_path = os.path.join(RAW_DATA_DIR, "benign")
malignant_path = os.path.join(RAW_DATA_DIR, "malignant")
SPLIT_DATA_DIR = os.path.join(EXPERIMENT_DIR, "split_data")
CV_DATA_DIR = os.path.join(EXPERIMENT_DIR, "cross_validation_data")
MODELS_DIR = os.path.join(EXPERIMENT_DIR, "train_val_models")
RESULTS_DIR = os.path.join(EXPERIMENT_DIR, "evaluation_results")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# 使用第1步划分的训练集和验证集
TRAIN_VAL_DIR = os.path.join(SPLIT_DATA_DIR, 'train_val')    
TEST_DIR = os.path.join(SPLIT_DATA_DIR, 'test')

# # 正式训练参数
# NUM_FOLDS = 5
# NUM_EPOCHS = 50
# BATCH_SIZE = 16
# INITIAL_LR = 0.001
# PATIENCE = 10
# NUM_WORKERS = 0
# NUM_CLASSES = 2

# 测试训练参数
NUM_FOLDS = 3
NUM_EPOCHS = 3
BATCH_SIZE = 6
INITIAL_LR = 0.001
PATIENCE = 6
NUM_WORKERS = 0
NUM_CLASSES = 2
# 设备配置
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 选择要训练的模型（可以选择部分或全部）    
MODELS_TO_TRAIN = [
'MobileNetV2'
]
# # 或者训练所有模型：
# MODELS_TO_TRAIN = SUPPORTED_MODELS       

# 检查原始数据目录是否存在
if not os.path.exists(RAW_DATA_DIR):
    print(f"\n❌ 错误: 原始数据目录不存在: {RAW_DATA_DIR}")
    raise FileNotFoundError(f"原始数据目录不存在: {RAW_DATA_DIR}")

if not os.path.exists(benign_path):
    print(f"\n❌ 错误: 良性数据目录不存在: {benign_path}")
    raise FileNotFoundError(f"原始数据目录不存在: {benign_path}")

if not os.path.exists(malignant_path):
    print(f"\n❌ 错误: 恶性数据目录不存在: {malignant_path}")
    raise FileNotFoundError(f"原始数据目录不存在: {malignant_path}")

print(f"\n✅ 数据目录检查通过")


✅ 数据目录检查通过


# 分割代码：data\dataset_splitter.py

In [7]:
"""
数据集分割模块（修复版 v3）
提供数据集分割功能，支持分层抽样
# #	改动点	原代码	修改后代码
# 1	随机性控制	全局 random.seed()	局部 random.Random(seed) 实例
# 2	目录清空	❌ 无，仅 makedirs	✅ 新增 clear_directory()
# 3	文件名处理	.replace(".svs","") 篡改文件名	✅ 直接保留原始文件名
# 4	冲突检测	❌ 无，静默覆盖	✅ os.path.exists() 检测后报错
# 5	文件过滤	❌ 无，处理所有文件	✅ 白名单过滤非图像文件
# 6	统计日志	第二次循环重新计算	✅ 同一循环内记录，与实际一致
"""

import os
import shutil
import random
from pathlib import Path
import pandas as pd
from datetime import datetime

# 图像文件扩展名白名单
VALID_IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
# ✅ 移除 '.svs'：您的源文件全是png，svs是病理切片原始格式，不应出现在源目录


def clear_directory(directory: str):
    """
    清空目录中的所有内容（保留目录本身）
    若目录不存在则创建
    """
    if os.path.exists(directory):
        for filename in os.listdir(directory):
            file_path = os.path.join(directory, filename)
            if os.path.isfile(file_path):
                os.remove(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        print(f"  已清空目录: {directory}")
    else:
        os.makedirs(directory)
        print(f"  已创建目录: {directory}")


def split_dataset(benign_path, malignant_path, output_dir,
                  split_ratio=0.9, random_seed=42):
    """
    将数据集分成训练验证集(train_val)和测试集(test)，使用分层抽样

    规则：
        - 良性图像：100% 进入训练验证集，测试集为空
        - 恶性图像：按 split_ratio 划分训练验证集和测试集

    参数:
        benign_path (str):    良性图像的路径
        malignant_path (str): 恶性图像的路径
        output_dir (str):     输出目录
        split_ratio (float):  恶性图像训练验证集的比例，默认为 0.9
        random_seed (int):    随机种子，默认为 42（保证完美复现）

    返回:
        dict: 包含分割统计信息的字典
    """
    # 使用局部 Random 实例，完全隔离全局随机状态，保证复现性
    rng = random.Random(random_seed)

    # 验证输入路径
    if not os.path.exists(benign_path):
        raise ValueError(f"良性图像路径不存在: {benign_path}")
    if not os.path.exists(malignant_path):
        raise ValueError(f"恶性图像路径不存在: {malignant_path}")

    split_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    split_logs = []

    # 定义目录结构
    train_val_benign_dir    = os.path.join(output_dir, "train_val", "benign")
    train_val_malignant_dir = os.path.join(output_dir, "train_val", "malignant")
    test_benign_dir         = os.path.join(output_dir, "test",      "benign")
    test_malignant_dir      = os.path.join(output_dir, "test",      "malignant")

    # 划分前清空所有目标目录，杜绝历史残留
    print("=" * 50)
    print("Step 1: 清空目标目录...")
    for directory in [train_val_benign_dir, train_val_malignant_dir,
                      test_benign_dir, test_malignant_dir]:
        clear_directory(directory)

    # 处理良性图像（split_ratio=1.0，全部进训练验证集）
    print("\n" + "=" * 50)
    print("Step 2: 处理良性图像（全部放入训练验证集）...")
    benign_log = _process_images(
        benign_path, train_val_benign_dir, test_benign_dir,
        split_ratio=1.0, rng=rng
    )
    split_logs.append({
        '划分时间':       split_time,
        '类别':          '良性',
        '输入路径':       benign_path,
        '输出路径':       output_dir,
        '分割比例':       '100% → 训练验证集',
        '训练验证集数量': benign_log['train_val'],
        '测试集数量':     benign_log['test'],
        '总数':          benign_log['total']
    })

    # 处理恶性图像（按 split_ratio 划分）
    print("\n" + "=" * 50)
    print(f"Step 3: 处理恶性图像（训练验证集比例={split_ratio*100:.1f}%）...")
    malignant_log = _process_images(
        malignant_path, train_val_malignant_dir, test_malignant_dir,
        split_ratio=split_ratio, rng=rng
    )
    split_logs.append({
        '划分时间':       split_time,
        '类别':          '恶性',
        '输入路径':       malignant_path,
        '输出路径':       output_dir,
        '分割比例':       f'{split_ratio*100:.1f}% → 训练验证集',
        '训练验证集数量': malignant_log['train_val'],
        '测试集数量':     malignant_log['test'],
        '总数':          malignant_log['total']
    })

    # 保存日志
    print("\n" + "=" * 50)
    train_val_dir = os.path.join(output_dir, "train_val")
    test_dir      = os.path.join(output_dir, "test")
    print(f"数据集分割完成！")
    print(f"  训练验证集保存在: {train_val_dir}")
    print(f"  测试集保存在:     {test_dir}")

    df = pd.DataFrame(split_logs)
    excel_path = os.path.join(output_dir, "dataset_split_log.xlsx")
    df.to_excel(excel_path, index=False, engine='openpyxl')
    print(f"  日志已保存到:     {excel_path}")

    return {
        'benign':    benign_log,
        'malignant': malignant_log,
        'log_path':  excel_path
    }


def _process_images(source_dir, train_val_dir, test_dir, split_ratio, rng):
    """
    处理图像：分层抽样分割并复制到目标目录

    参数:
        source_dir (str):    源目录
        train_val_dir (str): 训练验证集目标目录
        test_dir (str):      测试集目标目录
        split_ratio (float): 训练验证集比例
        rng (random.Random): 局部随机实例，保证复现性
    """
    # sorted() 保证文件列表顺序与操作系统无关，是复现性的关键
    image_files = sorted([
        f for f in os.listdir(source_dir)
        if os.path.isfile(os.path.join(source_dir, f))
        and os.path.splitext(f)[1].lower() in VALID_IMAGE_EXTENSIONS
    ])

    if not image_files:
        print(f"  ⚠️  警告：{source_dir} 中未找到任何图像文件！")
        return {'train_val': 0, 'test': 0, 'total': 0}

    # 按扩展名分组（分层抽样）
    file_groups: dict[str, list[str]] = {}
    for file in image_files:
        ext = os.path.splitext(file)[1].lower()
        file_groups.setdefault(ext, []).append(file)

    train_val_files = []
    test_files      = []
    detail_logs     = []

    for ext, files in sorted(file_groups.items()):
        rng.shuffle(files)
        split_idx = int(len(files) * split_ratio)
        tv = files[:split_idx]
        te = files[split_idx:]
        train_val_files.extend(tv)
        test_files.extend(te)
        detail_logs.append(
            f"  扩展名 {ext:6s}: 总数={len(files):4d}, "
            f"训练验证集={len(tv):4d}, 测试集={len(te):4d}"
        )

    print(f"  分层抽样结果: 训练验证集={len(train_val_files)}, "
          f"测试集={len(test_files)}, 合计={len(image_files)}")
    print("  分层详情:")
    for log in detail_logs:
        print(log)

    # ✅ 修复核心：直接保留原始文件名，不做任何替换
    def safe_copy(file: str, dst_dir: str):
        src_path = os.path.join(source_dir, file)
        dst_path = os.path.join(dst_dir, file)      # ✅ 直接使用原始文件名

        if os.path.exists(dst_path):
            raise FileExistsError(
                f"\n❌ 文件名冲突（源目录中存在重名文件）！\n"
                f"   目标路径:   {dst_path}\n"
                f"   当前源文件: {src_path}\n"
            )
        shutil.copy2(src_path, dst_path)

    for file in train_val_files:
        safe_copy(file, train_val_dir)
    for file in test_files:
        safe_copy(file, test_dir)

    print(f"  ✅ 文件复制完成")

    return {
        'train_val': len(train_val_files),
        'test':      len(test_files),
        'total':     len(image_files)
    }


## split_dataset函数的拆分详解

In [15]:
# 使用局部 Random 实例，完全隔离全局随机状态，保证复现性
random_seed = 42
rng = random.Random(random_seed)
print(rng)
# 验证输入路径
if not os.path.exists(benign_path):
    raise ValueError(f"良性图像路径不存在: {benign_path}")
if not os.path.exists(malignant_path):
    raise ValueError(f"恶性图像路径不存在: {malignant_path}")

split_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
split_logs = []

# 定义目录结构
output_dir = SPLIT_DATA_DIR
train_val_benign_dir    = os.path.join(output_dir, "train_val", "benign")
train_val_malignant_dir = os.path.join(output_dir, "train_val", "malignant")
test_benign_dir         = os.path.join(output_dir, "test",      "benign")
test_malignant_dir      = os.path.join(output_dir, "test",      "malignant")

# 划分前清空所有目标目录，杜绝历史残留
print("=" * 50)
print("Step 1: 清空目标目录...")
for directory in [train_val_benign_dir, train_val_malignant_dir,
                    test_benign_dir, test_malignant_dir]:
    clear_directory(directory)


Step 1: 清空目标目录...
  已创建目录: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\split_data\train_val\benign
  已创建目录: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\split_data\train_val\malignant
  已创建目录: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\split_data\test\benign
  已创建目录: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\split_data\test\malignant


## _process_images函数（用恶性细胞做逐行演示）

In [16]:
source_dir = malignant_path
train_val_dir = train_val_malignant_dir
test_dir = test_malignant_dir
split_ratio = 0.9
rng=rng
VALID_IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
# 获取并排序符合条件的图像文件列表
image_files = sorted([  # 使用sorted()对结果进行排序，确保顺序一致性
    f  # 文件名变量
    for f in os.listdir(source_dir)  # 遍历source_dir目录中的所有文件和子目录名称
    if os.path.isfile(os.path.join(source_dir, f))  # 第一个条件：检查是否为文件（不是目录）
    and os.path.splitext(f)[1].lower() in VALID_IMAGE_EXTENSIONS  # 第二个条件：检查文件扩展名是否在允许的图像扩展名列表中
])
print(image_files)

## 按扩展名分组的方式用于实现分层抽样：不同格式的图像可能有不同的特性（如分辨率、色彩空间等），为了确保训练集和测试集的分布一致，需要在每种格式中按相同比例抽样。
# 按文件扩展名对图像文件进行分组
file_groups: dict[str, list[str]] = {}  # 创建空字典，键为扩展名(str)，值为文件名列表(list[str])

for file in image_files:  # 遍历所有筛选出的图像文件
    ext = os.path.splitext(file)[1].lower()  # 提取文件扩展名并转换为小写（如 ".jpg"）
    
    # 使用setdefault方法确保字典中存在对应扩展名的键值对，然后将文件添加到对应列表中
    # 如果扩展名在字典中不存在，会创建一个新的空列表；如果已存在，则使用现有列表
    file_groups.setdefault(ext, []).append(file)

print(file_groups)

# 初始化用于存储分割结果的列表
train_val_files = []  # 存储所有将进入训练验证集的文件
test_files = []       # 存储所有将进入测试集的文件
detail_logs = []      # 存储每种扩展名的分割统计信息

print(f"\n开始按扩展名进行分层抽样，分割比例: {split_ratio:.2f}")
print(f"文件扩展名分组情况: {dict((ext, len(files)) for ext, files in file_groups.items())}")

# 对每种扩展名的文件进行分层抽样
for ext, files in sorted(file_groups.items()):
    print(f"\n处理扩展名: {ext}")
    print(f"  分割前文件列表 (显示前5个): {files[:5]}")
    print(f"  该扩展名文件总数: {len(files)}")
    
    # 随机打乱文件顺序，确保随机抽样
    rng.shuffle(files)
    print(f"  打乱后文件列表 (显示前5个): {files[:5]}")
    
    # 计算分割点索引
    split_idx = int(len(files) * split_ratio)
    print(f"  分割点索引: {split_idx} (总数 * 分割比例 = {len(files)} * {split_ratio:.2f})")
    
    # 分割为训练验证集和测试集
    tv = files[:split_idx]  # 训练验证集文件，不包含分割点数据
    te = files[split_idx:]  # 测试集文件，包含分割点数据
    print(f"  分配到训练验证集的文件数: {len(tv)}")
    print(f"  分配到测试集的文件数: {len(te)}")
    
    # 将当前扩展名的文件添加到对应的总列表中
    train_val_files.extend(tv)
    test_files.extend(te)
    print(f"  累计训练验证集文件数: {len(train_val_files)}")
    print(f"  累计测试集文件数: {len(test_files)}")
    
    # 记录分割统计信息
    log_entry = f"  扩展名 {ext:6s}: 总数={len(files):4d}, 训练验证集={len(tv):4d}, 测试集={len(te):4d}"
    detail_logs.append(log_entry)
    print(f"  添加日志: {log_entry}")

# 打印分层抽样的最终结果
print(f"\n分层抽样完成!")
print(f"总训练验证集文件数: {len(train_val_files)}")
print(f"总测试集文件数: {len(test_files)}")
print(f"总文件数: {len(train_val_files) + len(test_files)}")
print("\n分层详情:")
for log in detail_logs:
    print(log)

['GY_UR1.svs_GY_UR1_21_88.png', 'GY_UR1.svs_GY_UR1_52_43.png', 'GY_UR1.svs_GY_UR1_54_45.png', 'SC-UR1.svs_SC-UR28_31.png', 'SC-UR1.svs_SC-UR46_54.png', 'UR5_llhua.svs_llhua1ts159_163.png', 'UR5_llhua.svs_llhua1ts174_182.png', 'UR5_llhua.svs_llhua1ts186_201.png', 'UR5_llhua.svs_llhua1ts188_194.png', 'UR5_llhua.svs_llhua1ts190_196.png', 'UR5_llhua.svs_llhua1ts200_210.png', 'ccju.svs_ccjust18_17.png', 'ccju.svs_ccjust42_48.png', 'ccju.svs_ccjust50_58.png', 'ccju.svs_ccjust55_62.png', 'ftzhu.svs_ftzhust39_38.png', 'hxfang.svs_hxfang1ts13_12.png', 'hxfang.svs_hxfang1ts42_40.png', 'hxfang.svs_hxfangts28_66.png', 'lsfeng.svs_lsfeng1ts131_pos_155.png', 'lsfeng.svs_lsfeng1ts137_pos_161.png', 'lsfeng.svs_lsfeng1ts92_pos_90.png', 'lzyuan.svs_lzyuanst16_16.png', 'lzyuan.svs_lzyuanst33_35.png', 'pyyong.svs_pyyongst25_18.png', 'pyyong.svs_pyyongst55_50.png', 'pyyong2.svs_pyyongst8_8.png', 'sske1.svs_sskest57_56.png', 'wtdi.svs_wtdist29_29.png', 'xbwen.svs_xbwenst27_27.png', 'xbwen.svs_xbwenst30_30.p

In [17]:
# 处理良性图像（split_ratio=1.0，全部进训练验证集）
print("\n" + "=" * 50)
print("Step 2: 处理良性图像（全部放入训练验证集）...")
benign_log = _process_images(
    benign_path, train_val_benign_dir, test_benign_dir,
    split_ratio=1.0, rng=rng
)
split_logs.append({
    '划分时间':       split_time,
    '类别':          '良性',
    '输入路径':       benign_path,
    '输出路径':       output_dir,
    '分割比例':       '100% → 训练验证集',
    '训练验证集数量': benign_log['train_val'],
    '测试集数量':     benign_log['test'],
    '总数':          benign_log['total']
})

# 处理恶性图像（按 split_ratio 划分）
print("\n" + "=" * 50)
print(f"Step 3: 处理恶性图像（训练验证集比例={split_ratio*100:.1f}%）...")
malignant_log = _process_images(
    malignant_path, train_val_malignant_dir, test_malignant_dir,
    split_ratio=0.9, rng=rng
)
split_logs.append({
    '划分时间':       split_time,
    '类别':          '恶性',
    '输入路径':       malignant_path,
    '输出路径':       output_dir,
    '分割比例':       f'{split_ratio*100:.1f}% → 训练验证集',
    '训练验证集数量': malignant_log['train_val'],
    '测试集数量':     malignant_log['test'],
    '总数':          malignant_log['total']
})

# 保存日志
print("\n" + "=" * 50)
train_val_dir = os.path.join(output_dir, "train_val")
test_dir      = os.path.join(output_dir, "test")
print(f"数据集分割完成！")
print(f"  训练验证集保存在: {train_val_dir}")
print(f"  测试集保存在:     {test_dir}")

df = pd.DataFrame(split_logs)
excel_path = os.path.join(output_dir, "dataset_split_log.xlsx")
df.to_excel(excel_path, index=False, engine='openpyxl')
print(f"  日志已保存到:     {excel_path}")


Step 2: 处理良性图像（全部放入训练验证集）...
  分层抽样结果: 训练验证集=50, 测试集=0, 合计=50
  分层详情:
  扩展名 .png  : 总数=  50, 训练验证集=  50, 测试集=   0
  ✅ 文件复制完成

Step 3: 处理恶性图像（训练验证集比例=90.0%）...
  分层抽样结果: 训练验证集=44, 测试集=6, 合计=50
  分层详情:
  扩展名 .jpg  : 总数=   1, 训练验证集=   0, 测试集=   1
  扩展名 .png  : 总数=  49, 训练验证集=  44, 测试集=   5
  ✅ 文件复制完成

数据集分割完成！
  训练验证集保存在: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\split_data\train_val
  测试集保存在:     G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\split_data\test
  日志已保存到:     G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\split_data\dataset_split_log.xlsx


# 五折交叉验证分割：data\cross_validation.py

In [18]:
"""
5折交叉验证数据准备模块

🔴 Bug 修复（影响正确性）
#	位置	原代码	修改后
1	文件收集	image_files 无排序	image_files = sorted(image_files)
2	输出目录	os.makedirs(..., exist_ok=True) 不清空旧数据	clear_directory() 先清空再创建
3	文件复制	shutil.copy2() 直接覆盖同名文件	safe_copy() 检测冲突后再复制

🟡 功能增强（影响可用性）
#	位置	原代码	修改后
4	函数入口	无路径验证，glob 静默返回空列表	验证 data_path 存在且包含图像
5	折循环内	简单 print 文字	收集结构化数据到 fold_logs
6	函数末尾	无汇总表格	打印多级列名汇总表格
7	日志保存	单 Sheet Excel，列结构简单	两个 Sheet：折划分汇总 + 明细
8	返回值	无返回值（隐式 None）	返回含 df_summary 的字典
"""

import os
import pandas as pd
from sklearn.model_selection import KFold
from glob import glob
import shutil
from datetime import datetime

from shilab_classifier.data.dataset_splitter import clear_directory


# 支持的图像扩展名白名单
VALID_IMAGE_EXTENSIONS = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']


def prepare_cross_validation(data_path: str, output_base: str, n_splits: int = 5, random_state: int = 42):
    """
    准备N折交叉验证数据集

    Args:
        data_path (str):    原始数据路径（结构：data_path/类别名/图像文件）
        output_base (str):  输出基础路径
        n_splits (int):     折数，默认5
        random_state (int): 随机种子，默认42，保证完美复现

    Returns:
        dict: 包含交叉验证统计信息的字典

    Raises:
        ValueError:      数据路径不存在或未找到图像文件
        FileExistsError: 复制时发现同名文件冲突
    """
    print(f"开始准备 {n_splits} 折交叉验证数据...")
    print(f"数据路径: {data_path}")
    print(f"输出路径: {output_base}")

    # 验证输入路径
    if not os.path.exists(data_path):
        raise ValueError(f"数据路径不存在: {data_path}")

    # sorted() 固定文件列表顺序，保证复现性
    image_files = []
    for ext in VALID_IMAGE_EXTENSIONS:
        image_files.extend(glob(os.path.join(data_path, '**', ext), recursive=True))
    image_files = sorted(image_files)

    if not image_files:
        raise ValueError(f"在 {data_path} 中未找到任何图像文件！")
    print(f"找到 {len(image_files)} 张图像")

    # 提取类别信息（父目录名即类别名）
    file_paths = []
    classes    = []
    for img_path in image_files:
        class_name = os.path.basename(os.path.dirname(img_path))
        file_paths.append(img_path)
        classes.append(class_name)

    df = pd.DataFrame({'file_path': file_paths, 'class': classes})
    print(f"\n类别分布:")
    print(df['class'].value_counts())

    # 清空输出目录
    print(f"\n清空输出目录...")
    clear_directory(output_base)

    kf             = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    unique_classes = sorted(df['class'].unique())
    fold_logs      = []
    split_time     = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"\n开始划分 {n_splits} 折...")
    print("=" * 50)

    for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
        fold_dir  = os.path.join(output_base, f'fold_{fold + 1}')
        train_dir = os.path.join(fold_dir, 'train')
        val_dir   = os.path.join(fold_dir, 'val')

        for cls in unique_classes:
            clear_directory(os.path.join(train_dir, cls))
            clear_directory(os.path.join(val_dir,   cls))

        train_data = df.iloc[train_idx]
        val_data   = df.iloc[val_idx]

        def safe_copy(src: str, dst: str):
            if os.path.exists(dst):
                raise FileExistsError(
                    f"\n❌ 文件名冲突！\n"
                    f"   目标路径: {dst}\n"
                    f"   源文件:   {src}"
                )
            shutil.copy2(src, dst)

        for _, row in train_data.iterrows():
            src = row['file_path']
            dst = os.path.join(train_dir, row['class'], os.path.basename(src))
            safe_copy(src, dst)

        for _, row in val_data.iterrows():
            src = row['file_path']
            dst = os.path.join(val_dir, row['class'], os.path.basename(src))
            safe_copy(src, dst)

        # 统计每个类别的数量
        train_dist = train_data['class'].value_counts().to_dict()
        val_dist   = val_data['class'].value_counts().to_dict()

        # ✅ 按照表格结构收集每折数据
        row_data = {'Fold': f'fold_{fold + 1}'}
        for cls in unique_classes:
            row_data[f'Training_{cls}']   = train_dist.get(cls, 0)
            row_data[f'Validation_{cls}'] = val_dist.get(cls, 0)
        row_data['Training_Total']   = len(train_data)
        row_data['Validation_Total'] = len(val_data)
        fold_logs.append(row_data)

        print(f"Fold {fold + 1}: 训练集={len(train_data)}, 验证集={len(val_data)}")

    # ✅ 生成结构化汇总表格并打印
    df_summary = pd.DataFrame(fold_logs).set_index('Fold')

    # 构建多级列名，与图片表格一致
    col_map = {}
    for cls in unique_classes:
        col_map[f'Training_{cls}']   = ('Training Set',   cls.capitalize())
        col_map[f'Validation_{cls}'] = ('Validation Set', cls.capitalize())
    col_map['Training_Total']   = ('Training Set',   'Total')
    col_map['Validation_Total'] = ('Validation Set', 'Total')

    df_display = df_summary.rename(columns=col_map)
    df_display.columns = pd.MultiIndex.from_tuples(df_display.columns)

    print("\n" + "=" * 50)
    print("各折数据划分汇总：")
    print("=" * 50)
    print(df_display.to_string())
    print("=" * 50)

    # ✅ 保存 Excel（多级列名 + 划分时间）
    log_path = os.path.join(output_base, "cross_validation_log.xlsx")
    with pd.ExcelWriter(log_path, engine='openpyxl') as writer:
        df_display.to_excel(writer, sheet_name='折划分汇总')

        # 额外保存一张带划分时间的明细表
        df_detail = pd.DataFrame(fold_logs)
        df_detail.insert(1, '划分时间', split_time)
        df_detail.to_excel(writer, sheet_name='明细', index=False)

    print(f"\n{n_splits} 折交叉验证数据集准备完成！")
    print(f"输出目录: {output_base}")
    print(f"日志已保存: {log_path}")

    return {
        'n_splits':   n_splits,
        'total':      len(df),
        'log_path':   log_path,
        'df_summary': df_display,
    }

## 主要是为了得到每个图片路径及其对应的class值的数据框

In [22]:
VALID_IMAGE_EXTENSIONS = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
data_path=TRAIN_VAL_DIR
output_base=CV_DATA_DIR
n_splits=NUM_FOLDS
random_state=seed
print(f"开始准备 {n_splits} 折交叉验证数据...")
print(f"数据路径: {data_path}")
print(f"输出路径: {output_base}")
# 验证输入路径
if not os.path.exists(data_path):
    print(f"❌ 错误: 数据路径 {data_path} 不存在")
    raise ValueError(f"数据路径不存在: {data_path}")
else:
    print(f"✅ 数据路径验证通过: {data_path} 存在")

# sorted() 固定文件列表顺序，保证复现性
image_files = []
print(f"初始化空的图像文件列表: {image_files}")

for ext in VALID_IMAGE_EXTENSIONS:
    print(f"搜索扩展名为 {ext} 的文件...")
    found_files = glob(os.path.join(data_path, '**', ext), recursive=True)
    print(f"  - 找到 {len(found_files)} 个 {ext} 文件")
    if len(found_files) > 0:
        print(f"  - 示例: {found_files[0]}")
    image_files.extend(found_files)
    print(f"  - 当前文件列表长度: {len(image_files)}")

print(f"排序前的前5个文件: {image_files[:5] if len(image_files) >= 5 else image_files}")
image_files = sorted(image_files)
print(f"排序后的前5个文件: {image_files[:5] if len(image_files) >= 5 else image_files}")

if not image_files:
    print(f"❌ 错误: 在 {data_path} 中未找到任何图像文件")
    raise ValueError(f"在 {data_path} 中未找到任何图像文件！")
print(f"找到 {len(image_files)} 张图像")

# 提取类别信息（父目录名即类别名）
file_paths = []
classes = []
print(f"初始化空的文件路径列表和类别列表")

for i, img_path in enumerate(image_files):
    if i < 3 or i == len(image_files) - 1:  # 只打印前3个和最后1个，避免输出过多
        print(f"处理图像 {i+1}/{len(image_files)}: {img_path}")
    
    # 获取图像所在目录的路径
    img_dir = os.path.dirname(img_path)
    print(f"  - 图像所在目录: {img_dir}") if i < 3 or i == len(image_files) - 1 else None
    
    # 获取目录名（即类别名）
    class_name = os.path.basename(img_dir)
    print(f"  - 提取的类别名: {class_name}") if i < 3 or i == len(image_files) - 1 else None
    
    file_paths.append(img_path)
    classes.append(class_name)
    
    if i < 3 or i == len(image_files) - 1:
        print(f"  - 当前file_paths长度: {len(file_paths)}")
        print(f"  - 当前classes长度: {len(classes)}")

# 打印部分收集到的数据
print(f"\n前3个文件路径: {file_paths[:3]}")
print(f"前3个类别: {classes[:3]}")

# 创建DataFrame
print("\n创建DataFrame...")
df = pd.DataFrame({'file_path': file_paths, 'class': classes})
print(f"DataFrame形状: {df.shape}")
print(f"DataFrame前5行:\n{df.head()}")

print(f"\n类别分布:")
class_counts = df['class'].value_counts()
print(class_counts)
print(f"类别总数: {len(class_counts)}")

# 清空输出目录
print(f"\n清空输出目录...")
print(f"将清空目录: {output_base}")
clear_directory(output_base)
print(f"✅ 输出目录已清空并重新创建: {output_base}")

开始准备 3 折交叉验证数据...
数据路径: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\split_data\train_val
输出路径: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\cross_validation_data
✅ 数据路径验证通过: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\split_data\train_val 存在
初始化空的图像文件列表: []
搜索扩展名为 *.jpg 的文件...
  - 找到 0 个 *.jpg 文件
  - 当前文件列表长度: 0
搜索扩展名为 *.jpeg 的文件...
  - 找到 0 个 *.jpeg 文件
  - 当前文件列表长度: 0
搜索扩展名为 *.png 的文件...
  - 找到 94 个 *.png 文件
  - 示例: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\split_data\train_val\benign\chchun_251125_tile_12_20_1_conf0.85.png
  - 当前文件列表长度: 94
搜索扩展名为 *.bmp 的文件...
  - 找到 0 个 *.bmp 文件
  - 当前文件列表长度: 94
搜索扩展名为 *.tif 的文件...
  - 找到 0 个 *.tif 文件
  - 当前文件列表长度: 94
搜索扩展名为 *.tiff 的文件...
  - 找到 0 个 *.tiff 文件
  - 当前文件列表长度: 94
排序前的前5个文件: ['G:\\04PhD\\02Project\\03Urine\\StudyTest\\result\\classification_model\\UR_SingleCell\\split_data\\train_val\\benign\\chchun_

In [23]:
# 创建KFold交叉验证分割器
# n_splits: 将数据集分成几份（几折）
# shuffle=True: 在分割前随机打乱数据顺序，避免数据中可能存在的顺序偏差
# random_state: 随机种子，确保每次运行代码得到相同的随机分割结果，保证实验可重复性
kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
print(f"创建了{n_splits}折交叉验证分割器，随机种子为{random_state}")

# 提取所有不同的类别并排序
# df['class'].unique(): 获取DataFrame中'class'列的所有不重复值
# sorted(): 对这些类别名称按字母顺序排序，确保处理顺序的一致性
unique_classes = sorted(df['class'].unique())
print(f"数据集中包含以下{len(unique_classes)}个类别: {unique_classes}")

# 创建一个空列表，用于存储每一折交叉验证的日志信息
fold_logs = []
print(f"初始化空的fold_logs列表，用于记录每一折的数据分布")

# 记录数据分割的时间，格式为"年-月-日 时:分:秒"
split_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"记录当前数据分割时间: {split_time}")

print(f"\n开始划分 {n_splits} 折...")
print("=" * 50)

# 演示KFold.split()的工作原理
print("\n===== KFold.split()工作原理演示 =====")
print(f"原始数据集df的形状: {df.shape}（行数 x 列数）")
print(f"前5行数据示例:\n{df.head()}")

# 创建一个计数器，用于限制打印的信息量
split_demo_counter = 0

# 这里我们先演示一下kf.split(df)会生成什么
print("\n模拟kf.split(df)的输出:")
for train_indices, val_indices in kf.split(df):
    split_demo_counter += 1
    if split_demo_counter <= 2:  # 只打印前两折的详细信息，避免输出过多
        print(f"\n第{split_demo_counter}折的索引分割:")
        print(f"  训练集索引(前10个): {train_indices[:10]}... (共{len(train_indices)}个)")
        print(f"  验证集索引(前10个): {val_indices[:10]}... (共{len(val_indices)}个)")
        
        # 演示如何使用这些索引获取实际数据
        print(f"  使用训练集索引获取的前3行数据:\n{df.iloc[train_indices[:3]]}")
        print(f"  使用验证集索引获取的前3行数据:\n{df.iloc[val_indices[:3]]}")
    else:
        print(f"\n第{split_demo_counter}折: 训练集{len(train_indices)}个样本, 验证集{len(val_indices)}个样本")

# 重置KFold迭代器，因为上面的演示已经消耗了迭代器
kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
print("\n===== 演示结束，开始实际的交叉验证数据准备 =====")

# 正式开始交叉验证数据准备
for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
    # enumerate会给每次迭代添加一个计数器，fold从0开始
    # 为了使fold从1开始，我们在显示和命名时使用fold+1
    current_fold = fold + 1
    print(f"\n处理第 {current_fold} 折:")
    print(f"  训练集索引数量: {len(train_idx)}")
    print(f"  验证集索引数量: {len(val_idx)}")
    
    # 创建当前折的目录结构
    fold_dir = os.path.join(output_base, f'fold_{current_fold}')
    train_dir = os.path.join(fold_dir, 'train')
    val_dir = os.path.join(fold_dir, 'val')
    print(f"  创建目录结构:")
    print(f"    - 折目录: {fold_dir}")
    print(f"    - 训练集目录: {train_dir}")
    print(f"    - 验证集目录: {val_dir}")
    
    # 为每个类别创建子目录
    print(f"  为每个类别创建子目录:")
    for cls in unique_classes:
        train_class_dir = os.path.join(train_dir, cls)
        val_class_dir = os.path.join(val_dir, cls)
        print(f"    - 类别 '{cls}': {train_class_dir} 和 {val_class_dir}")
        # 清空并创建目录
        clear_directory(train_class_dir)
        clear_directory(val_class_dir)
    
    # 使用索引从DataFrame中提取训练集和验证集数据
    # df.iloc是基于整数位置的索引，用于按位置选择数据
    train_data = df.iloc[train_idx]
    val_data = df.iloc[val_idx]
    print(f"  提取数据集:")
    print(f"训练数据集train_data的形状: {train_data.shape}")  # 打印行数和列数
    print(f"训练数据集train_data的前3行:\n{train_data.head(3)}")  # 打印前3行数据
    print(f"    - 训练集: {len(train_data)}行")
    print(f"    - 验证集: {len(val_data)}行")
    
    # 定义安全复制函数，避免文件名冲突
    def safe_copy(src: str, dst: str):
        if os.path.exists(dst):
            print(f"    ❌ 文件已存在: {dst}")
            raise FileExistsError(
                f"\n❌ 文件名冲突！\n"
                f"   目标路径: {dst}\n"
                f"   源文件:   {src}"
            )
        # 确保目标目录存在
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        # print(f"    ✅ 复制文件: {os.path.basename(src)} -> {os.path.dirname(dst)}")
    
    # 复制训练集图像到对应目录
    print(f"  复制训练集图像:")
    train_copy_count = 0
    for _, row in train_data.iterrows(): #意思是索引不要，只要行的内容（即'file_path'和'class'列的值）
        src = row['file_path']
        dst = os.path.join(train_dir, row['class'], os.path.basename(src))
        safe_copy(src, dst)
        train_copy_count += 1
        if train_copy_count <= 3:  # 只打印前3个文件的详细信息
            print(f"    - 复制: {os.path.basename(src)} -> {os.path.join('train', row['class'])}")
    print(f"    ... 共复制了{train_copy_count}个训练集图像")
    
    # 复制验证集图像到对应目录
    print(f"  复制验证集图像:")
    val_copy_count = 0
    for _, row in val_data.iterrows():
        src = row['file_path']
        dst = os.path.join(val_dir, row['class'], os.path.basename(src))
        safe_copy(src, dst)
        val_copy_count += 1
        if val_copy_count <= 3:  # 只打印前3个文件的详细信息
            print(f"    - 复制: {os.path.basename(src)} -> {os.path.join('val', row['class'])}")
    print(f"    ... 共复制了{val_copy_count}个验证集图像")
    
    # 统计每个类别的数量
    train_dist = train_data['class'].value_counts().to_dict()
    val_dist = val_data['class'].value_counts().to_dict()
    print(f"  类别分布统计:")
    for cls in unique_classes:
        train_count = train_dist.get(cls, 0)
        val_count = val_dist.get(cls, 0)
        print(f"    - 类别 '{cls}': 训练集 {train_count}张, 验证集 {val_count}张")
    
    # 按照表格结构收集每折数据
    row_data = {'Fold': f'fold_{current_fold}'}
    for cls in unique_classes:
        row_data[f'Training_{cls}'] = train_dist.get(cls, 0)
        row_data[f'Validation_{cls}'] = val_dist.get(cls, 0)
    row_data['Training_Total'] = len(train_data)
    row_data['Validation_Total'] = len(val_data)
    fold_logs.append(row_data)
    
    print(f"  第{current_fold}折完成: 训练集={len(train_data)}张, 验证集={len(val_data)}张")
    print("  " + "-" * 40)

# 打印fold_logs收集的数据
print("\n收集到的所有折数据:")
for i, log in enumerate(fold_logs):
    print(f"第{i+1}折数据: {log}")

创建了3折交叉验证分割器，随机种子为42
数据集中包含以下2个类别: ['benign', 'malignant']
初始化空的fold_logs列表，用于记录每一折的数据分布
记录当前数据分割时间: 2026-03-25 21:32:21

开始划分 3 折...

===== KFold.split()工作原理演示 =====
原始数据集df的形状: (94, 2)（行数 x 列数）
前5行数据示例:
                                           file_path   class
0  G:\04PhD\02Project\03Urine\StudyTest\result\cl...  benign
1  G:\04PhD\02Project\03Urine\StudyTest\result\cl...  benign
2  G:\04PhD\02Project\03Urine\StudyTest\result\cl...  benign
3  G:\04PhD\02Project\03Urine\StudyTest\result\cl...  benign
4  G:\04PhD\02Project\03Urine\StudyTest\result\cl...  benign

模拟kf.split(df)的输出:

第1折的索引分割:
  训练集索引(前10个): [ 1  2  3  5  6  7  8 13 14 16]... (共62个)
  验证集索引(前10个): [ 0  4  9 10 11 12 15 18 22 26]... (共32个)
  使用训练集索引获取的前3行数据:
                                           file_path   class
1  G:\04PhD\02Project\03Urine\StudyTest\result\cl...  benign
2  G:\04PhD\02Project\03Urine\StudyTest\result\cl...  benign
3  G:\04PhD\02Project\03Urine\StudyTest\result\cl...  benign
  使用验证集索引获取的前3行数据:
  

# 训练前：数据预处理和定义模型

## training\preprocessing.py

In [2]:
"""
图像预处理模块 - 详细中文注释版

本模块提供了图像预处理的核心功能，包括：
1. 图像填充与调整大小（保持宽高比）
2. 根据不同模型需求调整图像尺寸
3. 计算数据集的均值和标准差用于标准化
"""

import os
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import traceback


def pad_to_square(image, fill_color=(255, 255, 255)):
    """
    将图像填充为正方形，然后调整大小为 224x224
    
    工作原理：
    1. 获取原始图像尺寸
    2. 创建一个正方形白色背景图像（边长为原图最长边）
    3. 将原图居中粘贴到白色背景上（保持宽高比）
    4. 将结果图像缩放到224x224像素
    
    参数:
    image: PIL.Image对象，输入图像
    fill_color: 填充颜色，默认为白色
    
    返回:
    处理后的224x224正方形图像
    """
    try:
        # 获取原始图像的大小
        width, height = image.size
        
        # 计算填充后的目标大小（取宽高中的较大值）
        max_side = max(width, height)
        
        # 创建一个新的白色背景图像（正方形，边长为max_side）
        new_image = Image.new('RGB', (max_side, max_side), fill_color)
        
        # 将原始图像粘贴到新图像的中心位置
        # (max_side - width) // 2: 水平居中的左上角x坐标
        # (max_side - height) // 2: 垂直居中的左上角y坐标
        new_image.paste(image, ((max_side - width) // 2, (max_side - height) // 2))
        
        # 调整到224x224大小，使用LANCZOS算法（高质量重采样）
        new_image = new_image.resize((224, 224), Image.LANCZOS)
        
        return new_image
    except Exception as e:
        # 异常处理：记录错误并返回空白图像
        print(f"Error in pad_to_square: {e}")
        print(f"Image info: {image}")
        return Image.new('RGB', (224, 224), fill_color)


def pad_to_square_transform(img):
    """
    将pad_to_square函数包装为可用于transforms.Lambda的转换函数
    
    这个包装函数便于在PyTorch的transforms流程中使用
    """
    try:
        return pad_to_square(img)
    except Exception as e:
        print(f"Error in pad_to_square_transform: {e}")
        return Image.new('RGB', (224, 224), (255, 255, 255))


def pad_to_square_299(image, fill_color=(255, 255, 255)):
    """
    将图像填充为正方形，然后调整大小为 299x299
    
    这个函数专为Inception V3模型设计，因为它需要299x299的输入
    工作原理与pad_to_square相同，只是目标尺寸不同
    """
    try:
        # 获取原始图像的大小
        width, height = image.size
        
        # 计算填充后的目标大小
        max_side = max(width, height)
        
        # 创建一个新的白色背景图像
        new_image = Image.new('RGB', (max_side, max_side), fill_color)
        
        # 将原始图像粘贴到新图像的中心
        new_image.paste(image, ((max_side - width) // 2, (max_side - height) // 2))
        
        # 调整到 299x299 大小，适合Inception V3模型
        new_image = new_image.resize((299, 299), Image.LANCZOS)
        
        return new_image
    except Exception as e:
        print(f"Error in pad_to_square_299: {e}")
        print(f"Image info: {image}")
        return Image.new('RGB', (299, 299), fill_color)


def pad_to_square_299_transform(img):
    """Inception V3模型使用的转换函数包装器"""
    try:
        return pad_to_square_299(img)
    except Exception as e:
        print(f"Error in pad_to_square_299_transform: {e}")
        return Image.new('RGB', (299, 299), (255, 255, 255))


def calculate_mean_std(data_dir, model_name=None):
    """
    计算数据集的均值和标准差
    
    这个函数会遍历整个数据集，计算所有图像在RGB三个通道上的均值和标准差。
    这些统计量用于后续的图像标准化处理，使模型训练更稳定。
    
    工作流程:
    1. 根据模型类型选择合适的图像尺寸
    2. 加载数据集并创建数据加载器
    3. 遍历所有图像，累计计算均值和标准差
    4. 返回最终计算结果
    
    参数:
    data_dir: 数据目录，包含分类子文件夹
    model_name: 模型名称，用于确定图像大小
    
    返回:
    均值和标准差，各为一个包含3个元素的张量（对应RGB三通道）
    """
    print(f"开始计算均值和标准差，数据目录: {data_dir}")
    # 检查目录是否存在
    if not os.path.exists(data_dir):
        print(f"错误: 目录 {data_dir} 不存在")
        # 如果目录不存在，返回ImageNet数据集的标准均值和标准差作为后备
        return torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225])
    
    try:
        # 根据模型名称选择合适的转换
        if model_name and 'Inception' in model_name:
            # Inception V3 使用 299x299 输入
            transform = transforms.Compose([
                transforms.Lambda(pad_to_square_299_transform),  # 自定义转换
                transforms.ToTensor()  # 将PIL图像转换为张量，并将像素值从[0,255]缩放到[0,1]
            ])
            print(f"使用 299x299 图像大小计算均值和标准差 (用于 {model_name})")
        else:
            # 其他模型使用 224x224 输入
            transform = transforms.Compose([
                transforms.Lambda(pad_to_square_transform),
                transforms.ToTensor()
            ])
            print("使用 224x224 图像大小计算均值和标准差")
        
        # 加载数据集 - ImageFolder会自动从文件夹结构中识别类别
        # 每个子文件夹名称被视为一个类别
        dataset = datasets.ImageFolder(data_dir, transform=transform)
        
        # 创建数据加载器 - 用于批量处理图像
        # batch_size=16: 每次处理16张图像
        # shuffle=False: 不打乱顺序，因为这里只是计算统计量
        # num_workers=0: 不使用多进程加载，避免潜在问题
        dataloader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0)
        
        print(f"数据集加载成功，共 {len(dataset)} 张图像")
        
        # 初始化均值和标准差累加器
        mean = torch.zeros(3)  # RGB三通道
        std = torch.zeros(3)
        total_images = 0  # 图像计数器
        
        # 遍历数据集中的所有批次
        for i, (images, _) in enumerate(dataloader):
            # i是批次索引，从0开始
            # images是当前批次的图像数据
            # _是标签数据（我们不需要，所以用_忽略）            
            if i % 10 == 0: # 每10个批次打印一次进度
                print(f"处理批次 {i}/{len(dataloader)}")
            
            batch_size = images.size(0)  # 当前批次的实际大小
            
            # 重塑图像张量以便计算每个通道的统计量
            # view() 是PyTorch中用来重新排列张量形状的函数，不改变数据内容，只改变数据的排列方式。
            # 从[batch_size, 3, height, width]变为[batch_size, 3, height*width]，把原来的高度和宽度两个维度合并成一个维度（-1表示自动计算）
            images = images.view(batch_size, images.size(1), -1)
            
            # 计算均值
            # 1. images.mean(2)：计算每张图片每个通道的均值，结果形状是[batch_size, 3]
            # 2. .sum(0)：在批次维度上求和，结果形状是[3]
            mean += images.mean(2).sum(0)  # mean(2)计算每个通道的均值，sum(0)在批次维度上求和
            
            # 计算每个图像每个通道的标准差，然后在批次维度上求和
            std += images.std(2).sum(0)
            
            # 累计处理过的图片数量
            total_images += batch_size
        
        # 计算最终的均值和标准差
        mean /= total_images
        std /= total_images
        
        print(f"均值和标准差计算完成: mean={mean}, std={std}")
        return mean, std
    
    except Exception as e:
        # 异常处理：记录错误并返回默认值
        print(f"计算均值和标准差时出错: {e}")
        traceback.print_exc()
        # 返回ImageNet的默认均值和标准差作为后备
        return torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225])

## models\model_definitions.py

In [3]:
import torch.nn as nn
from torchvision import models
import traceback

# ==================== AlexNet 模型定制 ====================
class CustomAlexNet(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 AlexNet 模型
        
        AlexNet 是2012年ImageNet竞赛冠军模型，结构相对简单但有效
        这里加载预训练的 AlexNet，并修改最后的分类层以适应二分类任务
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        # 调用父类 nn.Module 的初始化方法，设置必要的内部状态。
        super(CustomAlexNet, self).__init__()
        # 加载预训练的AlexNet模型（在ImageNet上训练的权重）
        self.alexnet = models.alexnet(pretrained=True)
        # 获取原始分类器最后一层的输入特征数量
        in_features = self.alexnet.classifier[6].in_features
        # 替换最后一层为适合二分类的全连接层
        # AlexNet的分类器是Sequential，第6个元素是最后的线性层
        self.alexnet.classifier[6] = nn.Linear(in_features, num_classes)
        # 添加softmax层（注意：在训练时使用CrossEntropyLoss时不需要这个），softmax函数输出值0到1，输出值的总和为1，所以可以看作概率
        self.softmax = nn.Softmax(dim=1) #虽然定义了这个层，但在 forward 方法中并没有使用它
    
    def forward(self, x):
        """
        前向传播函数
        
        参数:
            x: 输入图像张量
            
        返回:
            模型输出
        """
        return self.alexnet(x)  # 直接返回AlexNet的输出


# ==================== ResNeXt 模型定制 ====================
class CustomResNeXt(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 ResNeXt 模型
        
        ResNeXt 是 ResNet 的改进版，引入了"基数"概念，提高了模型性能
        这里使用 ResNeXt-50 32x4d 变体（50层，32个分组，每组4个通道）
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomResNeXt, self).__init__()
        # 加载预训练的ResNeXt模型
        self.resnext = models.resnext50_32x4d(pretrained=True)
        # 获取全连接层的输入特征数
        in_features = self.resnext.fc.in_features
        # 替换为二分类的全连接层
        self.resnext.fc = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.resnext(x)


# ==================== ConvNeXt Large 模型定制 ====================
class CustomConvNextLarge(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 ConvNeXt Large 模型
        
        ConvNeXt 是一种现代CNN架构，结合了Transformer的设计理念
        Large变体具有更多参数，能力更强
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomConvNextLarge, self).__init__()
        # 加载预训练的ConvNeXt Large模型
        self.convnext = models.convnext_large(pretrained=True)
        # ConvNeXt的分类器是Sequential，第2个元素是最后的线性层
        in_features = self.convnext.classifier[2].in_features
        # 替换为二分类的线性层
        self.convnext.classifier[2] = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.convnext(x)


# ==================== GoogLeNet (Inception v1) 模型定制 ====================
class CustomGoogLeNet(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 GoogLeNet 模型
        
        GoogLeNet 引入了Inception模块，是第一个探索网络宽度的模型
        特点是使用了辅助分类器来帮助训练深层网络
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomGoogLeNet, self).__init__()
        # 加载预训练的GoogLeNet模型，启用辅助分类器
        self.googlenet = models.googlenet(pretrained=True, aux_logits=True)
        
        # 修改主分类器以适应二分类任务
        in_features = self.googlenet.fc.in_features
        self.googlenet.fc = nn.Linear(in_features, num_classes)

        # 修改辅助分类器1（如果存在）
        if hasattr(self.googlenet, 'aux1'):
            print("self.googlenet.aux1:", self.googlenet.aux1)
            in_features_aux1 = self.googlenet.aux1.fc2.in_features
            self.googlenet.aux1.fc2 = nn.Linear(in_features_aux1, num_classes)
        
        # 修改辅助分类器2（如果存在）
        if hasattr(self.googlenet, 'aux2'):
            in_features_aux2 = self.googlenet.aux2.fc2.in_features
            self.googlenet.aux2.fc2 = nn.Linear(in_features_aux2, num_classes)
    
    def forward(self, x):
        """
        前向传播函数
        
        注意：GoogLeNet在训练时会返回三个输出（主输出和两个辅助输出）
        """
        return self.googlenet(x)  # 返回所有输出，包括辅助分类器


# ==================== ResNet50 模型定制 ====================
class CustomResNet50(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 ResNet50 模型
        
        ResNet 引入了残差连接，解决了深层网络的梯度消失问题
        ResNet50 是一个50层的变体，平衡了性能和复杂度
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomResNet50, self).__init__()
        # 加载预训练的ResNet50模型
        self.resnet = models.resnet50(pretrained=True)
        # 获取全连接层的输入特征数
        in_features = self.resnet.fc.in_features
        # 替换为二分类的全连接层
        self.resnet.fc = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.resnet(x)


# ==================== ResNet101 模型定制 ====================
class CustomResNet101(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 ResNet101 模型
        
        ResNet101 是ResNet的更深版本，有101层，具有更强的特征提取能力
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomResNet101, self).__init__()
        # 加载预训练的ResNet101模型
        self.resnet = models.resnet101(pretrained=True)
        # 获取全连接层的输入特征数
        in_features = self.resnet.fc.in_features
        # 替换为二分类的全连接层
        self.resnet.fc = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.resnet(x)


# ==================== DenseNet121 模型定制 ====================
class CustomDenseNet121(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 DenseNet121 模型
        
        DenseNet 通过密集连接提高了特征重用，每一层都与之前所有层相连
        DenseNet121 是一个相对轻量级的变体
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomDenseNet121, self).__init__()
        # 加载预训练的DenseNet121模型
        self.densenet = models.densenet121(pretrained=True)
        # 获取分类器的输入特征数
        in_features = self.densenet.classifier.in_features
        # 替换为二分类的分类器
        self.densenet.classifier = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.densenet(x)


# ==================== DenseNet161 模型定制 ====================
class CustomDenseNet161(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 DenseNet161 模型
        
        DenseNet161 是DenseNet的更大变体，具有更多参数和更强的能力
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomDenseNet161, self).__init__()
        # 加载预训练的DenseNet161模型
        self.densenet = models.densenet161(pretrained=True)
        # 获取分类器的输入特征数
        in_features = self.densenet.classifier.in_features
        # 替换为二分类的分类器
        self.densenet.classifier = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.densenet(x)


# ==================== VGG16 模型定制 ====================
class CustomVGG16(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 VGG16 模型
        
        VGG 使用了简单的3x3卷积堆叠设计，结构简单但有效
        VGG16 有16层，是常用的变体之一
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomVGG16, self).__init__()
        # 加载预训练的VGG16模型
        self.vgg = models.vgg16(pretrained=True)
        # VGG的分类器是Sequential，第6个元素是最后的线性层
        in_features = self.vgg.classifier[6].in_features
        # 替换为二分类的线性层
        self.vgg.classifier[6] = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.vgg(x)


# ==================== Inception V3 模型定制 ====================
class CustomInceptionV3(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 Inception V3 模型
        
        Inception V3 是GoogLeNet的改进版，使用了多种尺寸的卷积核
        特点是有辅助分类器，需要特殊处理
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomInceptionV3, self).__init__()
        # 加载预训练的Inception V3模型，启用辅助分类器
        self.inception = models.inception_v3(pretrained=True, aux_logits=True)
        
        # 修改主分类器以适应二分类任务
        in_features = self.inception.fc.in_features
        self.inception.fc = nn.Linear(in_features, num_classes)

        # 修改辅助分类器（如果存在）
        if hasattr(self.inception, 'AuxLogits'):
            print("self.inception.AuxLogits:", self.inception.AuxLogits)
            in_features_AuxLogits = self.inception.AuxLogits.fc.in_features
            self.inception.AuxLogits.fc = nn.Linear(in_features_AuxLogits, num_classes)
    
    def forward(self, x):
        """
        前向传播函数
        
        注意：Inception V3在训练时会返回两个输出（主输出和辅助输出）
        """
        return self.inception(x)  # 返回所有输出，包括辅助分类器


# ==================== EfficientNet B0 模型定制 ====================
class CustomEfficientNetB0(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 EfficientNet B0 模型
        
        EfficientNet 通过复合缩放方法平衡网络深度、宽度和分辨率
        B0 是最小的基础版本，适合资源受限场景
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomEfficientNetB0, self).__init__()
        # 加载预训练的EfficientNet B0模型
        self.efficientnet = models.efficientnet_b0(pretrained=True)
        # EfficientNet的分类器是Sequential，第1个元素是线性层
        in_features = self.efficientnet.classifier[1].in_features
        # 替换为二分类的线性层
        self.efficientnet.classifier[1] = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.efficientnet(x)


# ==================== EfficientNet V2 Large 模型定制 ====================
class CustomEfficientNet_v2_l(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 EfficientNet V2 Large 模型
        
        EfficientNet V2 是EfficientNet的改进版，训练更快，性能更好
        Large变体具有更多参数，能力更强
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomEfficientNet_v2_l, self).__init__()
        # 加载预训练的EfficientNet V2 Large模型
        self.efficientnet = models.efficientnet_v2_l(pretrained=True)
        # 获取分类器线性层的输入特征数
        in_features = self.efficientnet.classifier[1].in_features
        # 替换为二分类的线性层
        self.efficientnet.classifier[1] = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.efficientnet(x)


# ==================== MobileNet V2 模型定制 ====================
class CustomMobileNetV2(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 MobileNet V2 模型
        
        MobileNet 系列专为移动设备设计，轻量高效
        V2 版本引入了倒残差结构和线性瓶颈，提高了性能
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomMobileNetV2, self).__init__()
        # 加载预训练的MobileNet V2模型
        self.mobilenet = models.mobilenet_v2(pretrained=True)
        # 获取分类器线性层的输入特征数
        in_features = self.mobilenet.classifier[1].in_features
        # 替换为二分类的线性层
        self.mobilenet.classifier[1] = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.mobilenet(x)


# ==================== MobileNet V3 Large 模型定制 ====================
class CustomMobileNetV3(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 MobileNet V3 Large 模型
        
        MobileNet V3 是MobileNet的最新版本，结合了自动搜索和人工设计
        比V2更高效，引入了新的激活函数和注意力机制
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomMobileNetV3, self).__init__()
        # 加载预训练的MobileNet V3 Large模型
        self.mobilenet = models.mobilenet_v3_large(pretrained=True)
        
        # 获取原始分类器
        original_classifier = self.mobilenet.classifier
        
        # MobileNetV3的分类器结构较特殊，需要特别处理
        # 只替换最后一层（输出层），保留前面的层
        if isinstance(original_classifier, nn.Sequential):
            # 获取分类器中的所有层
            layers = list(original_classifier.children())
            # 假设最后一层是线性层
            if isinstance(layers[-1], nn.Linear):
                # 获取最后一层的输入特征数
                in_features = layers[-1].in_features
                # 创建新的分类器，保留前面所有层，只替换最后一层
                new_layers = layers[:-1] + [nn.Linear(in_features, num_classes)]
                self.mobilenet.classifier = nn.Sequential(*new_layers)
            else:
                # 如果最后一层不是线性层，使用默认方法完全替换分类器
                print("警告: 无法识别MobileNetV3分类器的最后一层，使用完全替换")
                last_channel = 1280  # MobileNetV3 Large的默认通道数
                self.mobilenet.classifier = nn.Sequential(
                    nn.Linear(last_channel, 1024),
                    nn.Hardswish(inplace=True),
                    nn.Dropout(p=0.2, inplace=True),
                    nn.Linear(1024, num_classes)
                )
        else:
            # 如果分类器不是Sequential类型，完全替换分类器
            print("警告: MobileNetV3分类器不是Sequential类型，使用完全替换")
            last_channel = 1280  # MobileNetV3 Large的默认通道数
            self.mobilenet.classifier = nn.Sequential(
                nn.Linear(last_channel, 1024),
                nn.Hardswish(inplace=True),
                nn.Dropout(p=0.2, inplace=True),
                nn.Linear(1024, num_classes)
            )
    
    def forward(self, x):
        """前向传播函数"""
        return self.mobilenet(x)


# ==================== RegNet 模型定制 ====================
class CustomRegNet(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 RegNet 模型
        
        RegNet 是通过系统搜索得到的高效网络架构
        这里使用的是 RegNet Y 400MF 变体（较小版本）
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomRegNet, self).__init__()
        # 加载预训练的RegNet Y 400MF模型
        self.regnet = models.regnet_y_400mf(pretrained=True)
        # 获取全连接层的输入特征数
        in_features = self.regnet.fc.in_features
        # 替换为二分类的全连接层
        self.regnet.fc = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.regnet(x)


# ==================== ShuffleNet V2 模型定制 ====================
class CustomShuffleNetV2(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 ShuffleNet V2 模型
        
        ShuffleNet 专为移动设备设计，使用通道混洗操作提高效率
        V2 版本基于硬件特性优化，更适合实际部署
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomShuffleNetV2, self).__init__()
        # 加载预训练的ShuffleNet V2 x1.0模型
        self.shufflenet = models.shufflenet_v2_x1_0(pretrained=True)
        # 获取全连接层的输入特征数
        in_features = self.shufflenet.fc.in_features
        # 替换为二分类的全连接层
        self.shufflenet.fc = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.shufflenet(x)


# ==================== SqueezeNet 模型定制 ====================
class CustomSqueezeNet(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 SqueezeNet 模型
        
        SqueezeNet 设计目标是在保持准确率的同时大幅减少参数量
        使用了"压缩-扩展"模块和全局平均池化
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomSqueezeNet, self).__init__()
        # 加载预训练的SqueezeNet 1.0模型
        self.squeezenet = models.squeezenet1_0(pretrained=True)
        # SqueezeNet的分类器使用1x1卷积而非全连接层
        self.squeezenet.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1, 1), stride=(1, 1))
        self.squeezenet.num_classes = num_classes
    
    def forward(self, x):
        """前向传播函数"""
        return self.squeezenet(x)


# ==================== Vision Transformer (ViT) Large 模型定制 ====================
class CustomViT_L16(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 Vision Transformer Large 模型
        
        ViT 将Transformer架构应用于图像分类，将图像分割为补丁序列
        L16表示Large版本，使用16x16像素的补丁大小
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomViT_L16, self).__init__()
        # 加载预训练的Vision Transformer Large模型
        self.vit = models.vit_l_16(pretrained=True)
        # 获取分类头的输入特征数
        in_features = self.vit.heads.head.in_features
        # 替换为二分类的线性层
        self.vit.heads.head = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.vit(x)


# ==================== Vision Transformer (ViT) Base 模型定制 ====================
class CustomViT_B16(nn.Module):
    def __init__(self, num_classes=2):
        """
        初始化自定义 Vision Transformer Base 模型
        
        ViT Base是比Large小的版本，参数更少，训练更快
        B16表示Base版本，使用16x16像素的补丁大小
        
        参数:
            num_classes: 分类数量，默认为2（二分类）
        """
        super(CustomViT_B16, self).__init__()
        # 加载预训练的Vision Transformer Base模型
        self.vit = models.vit_b_16(pretrained=True)
        # 获取分类头的输入特征数
        in_features = self.vit.heads.head.in_features
        # 替换为二分类的线性层
        self.vit.heads.head = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        """前向传播函数"""
        return self.vit(x)


# ==================== 模型创建函数 ====================
def create_model(model_name, num_classes=2):
    """
    根据模型名称创建对应的模型实例
    
    这个函数提供了一个统一的接口来创建各种模型，
    通过字符串匹配来识别用户想要的模型类型
    
    参数:
        model_name: 模型名称（字符串）
        num_classes: 类别数量，默认为2（二分类）
    
    返回:
        模型实例
    """
    # 将模型名称转为小写，便于匹配
    model_name_lower = model_name.lower()
    
    # 通过字符串匹配选择对应的模型
    if 'alexnet' in model_name_lower:
        return CustomAlexNet(num_classes)
    elif 'resnext' in model_name_lower:
        return CustomResNeXt(num_classes)
    elif 'convnext' in model_name_lower or 'convnextlarge' in model_name_lower:
        return CustomConvNextLarge(num_classes)
    elif 'googlenet' in model_name_lower:
        return CustomGoogLeNet(num_classes)
    elif 'resnet50' in model_name_lower:
        return CustomResNet50(num_classes)
    elif 'resnet101' in model_name_lower:
        return CustomResNet101(num_classes)
    elif 'densenet121' in model_name_lower:
        return CustomDenseNet121(num_classes)
    elif 'densenet161' in model_name_lower:
        return CustomDenseNet161(num_classes)
    elif 'vgg16' in model_name_lower or 'vgg' in model_name_lower:
        return CustomVGG16(num_classes)
    elif 'inceptionv3' in model_name_lower or ('inception' in model_name_lower and 'v3' in model_name_lower):
        return CustomInceptionV3(num_classes)
    elif 'efficientnetv2' in model_name_lower or 'efficientnet_v2' in model_name_lower:
        return CustomEfficientNet_v2_l(num_classes)
    elif 'efficientnetb0' in model_name_lower or 'efficientnet_b0' in model_name_lower:
        return CustomEfficientNetB0(num_classes)
    elif 'mobilenetv3' in model_name_lower or 'mobilenet_v3' in model_name_lower:
        return CustomMobileNetV3(num_classes)
    elif 'mobilenetv2' in model_name_lower or 'mobilenet_v2' in model_name_lower:
        return CustomMobileNetV2(num_classes)
    elif 'regnet' in model_name_lower:
        return CustomRegNet(num_classes)
    elif 'shufflenet' in model_name_lower:
        return CustomShuffleNetV2(num_classes)
    elif 'squeezenet' in model_name_lower:
        return CustomSqueezeNet(num_classes)
    elif 'vit_l' in model_name_lower or 'vitl' in model_name_lower:
        return CustomViT_L16(num_classes)
    elif 'vit_b' in model_name_lower or 'vitb' in model_name_lower:
        return CustomViT_B16(num_classes)
    else:
        # 如果没有匹配的模型，使用默认的ResNeXt
        print(f"警告: 未知的模型名称 '{model_name}'，使用默认的 ResNeXt")
        return CustomResNeXt(num_classes)


# ==================== 模型信息函数 ====================
def get_model_info(model_name):
    """
    获取模型的基本信息
    
    创建模型并计算其参数数量，用于比较不同模型的复杂度
    
    参数:
        model_name: 模型名称
    
    返回:
        包含模型信息的字典
    """
    # 创建指定的模型
    model = create_model(model_name)
    # 计算模型总参数数量
    total_params = sum(p.numel() for p in model.parameters())
    # 计算可训练参数数量（不包括冻结层）
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # 返回包含模型信息的字典
    return {
        'model_name': model_name,
        'total_params': total_params,
        'trainable_params': trainable_params,
        # InceptionV3需要299x299输入，其他模型使用224x224
        'input_size': 299 if 'Inception' in model_name else 224
    }

# 支持的模型列表（用于外部引用）
SUPPORTED_MODELS = [
    'AlexNet', 'ResNeXt', 'ConvNextLarge', 'GoogLeNet',
    'ResNet50', 'ResNet101', 'DenseNet121', 'DenseNet161',
    'VGG16', 'InceptionV3', 'EfficientNetB0', 'EfficientNetV2Large',
    'MobileNetV2', 'MobileNetV3', 'RegNet', 'ShuffleNetV2',
    'SqueezeNet', 'ViT_B16', 'ViT_L16'
]

# ==================== 测试函数 ====================
if __name__ == "__main__":
    """
    当直接运行此文件时，测试所有模型的创建
    这有助于验证所有模型是否能正确加载和初始化
    """
    # 测试所有支持的模型
    model_list = [
        'AlexNet', 'ResNeXt', 'ConvNextLarge', 'GoogLeNet',
        'ResNet50', 'ResNet101', 'DenseNet121', 'DenseNet161',
        'VGG16', 'InceptionV3', 'EfficientNetB0', 'EfficientNetV2Large',
        'MobileNetV2', 'MobileNetV3', 'RegNet', 'ShuffleNetV2',
        'SqueezeNet', 'ViT_B16', 'ViT_L16'
    ]
    
    print("测试所有模型创建...")
    for model_name in model_list:
        try:
            # 尝试创建模型
            model = create_model(model_name)
            # 获取模型信息
            info = get_model_info(model_name)
            # 打印成功信息，包括参数数量和输入尺寸
            print(f"✅ {model_name}: {info['total_params']:,} 参数, 输入尺寸: {info['input_size']}x{info['input_size']}")
        except Exception as e:
            # 如果创建失败，打印错误信息
            print(f"❌ {model_name}: {e}")

测试所有模型创建...


c:\ProgramData\miniconda3\envs\shilab\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\ProgramData\miniconda3\envs\shilab\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to C:\Users\lillian/.cache\torch\hub\checkpoints\alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:29<00:00, 8.30MB/s] 


✅ AlexNet: 57,012,034 参数, 输入尺寸: 224x224


...

✅ ViT_B16: 85,800,194 参数, 输入尺寸: 224x224


c:\ProgramData\miniconda3\envs\shilab\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_L_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_L_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vit_l_16-852ce7e3.pth" to C:\Users\lillian/.cache\torch\hub\checkpoints\vit_l_16-852ce7e3.pth


100%|██████████| 1.13G/1.13G [02:34<00:00, 7.86MB/s]


: 

# 训练模型

## training\trainer.py

### 余弦学习率调度（cosine_lr_schedule）

In [ ]:
def cosine_lr_schedule(optimizer, epoch, num_epochs, initial_lr):
    """
    余弦学习率调度
    严格按照原代码实现
    """
    # 计算当前epoch的学习率：从initial_lr平滑衰减到0
    # 公式解析: 0.5 * (1 + cos(π * epoch / num_epochs)) 在[0, num_epochs]区间从1衰减到0
    lr = initial_lr * 0.5 * (1. + math.cos(math.pi * epoch / num_epochs))
    
    # 更新优化器中所有参数组的学习率
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    # 返回当前学习率，用于记录和打印
    return lr

### 模型训练过程（train_model）

In [ ]:
def train_model(model, 
                train_loader, 
                val_loader, 
                criterion, 
                optimizer, 
                num_epochs,
                initial_lr,
                patience,
                device,
                log_file=None
                ):
    """
    训练模型
    
    参数:
        model: 待训练的模型
        train_loader: 训练数据加载器
        val_loader: 验证数据加载器
        criterion: 损失函数(train_single_model_fold中定义)
        optimizer: 优化器(train_single_model_fold中定义)
        num_epochs: 训练轮数
        initial_lr: 初始学习率
        patience: 早停耐心值
        device: 训练设备
        log_file: 日志文件
        
    返回:
        tuple: (model, history, best_val_acc, best_train_acc, best_detail_matrix)
    """
    # 初始化最佳模型跟踪变量
    best_val_acc = 0.0
    best_train_acc = 0.0
    best_model_wts = None
    best_detail_matrix = None  # 用于保存最佳模型的 detail_matrix
    
    # 初始化训练历史记录字典
    history = {
        'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
        'train_malignant_acc': [], 'train_benign_acc': [],
        'val_malignant_acc': [], 'val_benign_acc': [],
        'learning_rate': []  # 添加学习率跟踪
    }
    
    # 添加早停机制 - 监控验证集loss
    patience = patience  # 早停耐心值
    counter = 0  # 早停计数器
    best_val_loss = float('inf')  # 初始化为无穷大
    
    # 定义日志打印函数
    def log_message(message):
        print(message)
        if log_file:
            with open(log_file, 'a', encoding='utf-8') as f:
                f.write(message + '\n')
    
    # === 添加调试信息：检查训练集和验证集的结构 ===
    # 获取一个批次的数据样本
    log_message("\n=== 数据加载器结构分析 ===")
    for inputs, labels in train_loader:
        log_message(f"训练批次数据结构:")
        log_message(f"- 输入张量形状: {inputs.shape}")  # 通常是[batch_size, channels, height, width]
        log_message(f"- 输入张量类型: {inputs.dtype}")
        log_message(f"- 标签张量形状: {labels.shape}")  # 通常是[batch_size]
        log_message(f"- 标签张量类型: {labels.dtype}")
        log_message(f"- 标签值范围: 最小值={labels.min().item()}, 最大值={labels.max().item()}")
        
        # 显示批次中的类别分布
        unique_labels, counts = torch.unique(labels, return_counts=True)
        log_message(f"- 批次中的类别分布: {dict(zip(unique_labels.tolist(), counts.tolist()))}")
        
        # 只打印第一个批次就退出
        break
    
    # 开始训练循环
    for epoch in range(num_epochs):
        start_time = time.time()
        log_message(f"Epoch {epoch+1}/{num_epochs} 开始时间: {time.strftime('%H:%M:%S', time.localtime(start_time))}")
        
        # 更新学习率
        current_lr = cosine_lr_schedule(optimizer, epoch, num_epochs, initial_lr)
        log_message(f"当前学习率: {current_lr:.8f}")
        history['learning_rate'].append(current_lr)
        
        # 训练阶段
        model.train()  # 设置模型为训练模式（启用dropout、batch normalization等）
        log_message("开始训练阶段...")
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        train_preds = []
        train_labels = []
        
        # 尝试迭代训练数据
        try:
            log_message(f"训练数据批次数: {len(train_loader)}")
            
            # === 添加调试信息：详细了解训练数据加载器的结构 ===
            if epoch == 0:  # 只在第一个epoch打印详细信息
                log_message("\n训练数据加载器详细信息:")
                log_message(f"- 批次大小: {train_loader.batch_size}")
                log_message(f"- 数据集大小: {len(train_loader.dataset)}")
                log_message(f"- 类别映射: {train_loader.dataset.class_to_idx}")  # 类别名到索引的映射
                
                # 查看数据集的第一个样本
                first_img, first_label = train_loader.dataset[0]
                log_message(f"- 第一个样本图像形状: {first_img.shape}")
                log_message(f"- 第一个样本标签: {first_label} (对应类别: {list(train_loader.dataset.class_to_idx.keys())[list(train_loader.dataset.class_to_idx.values()).index(first_label)]})")
            
            # 遍历训练数据批次
            for batch_idx, (inputs, labels) in enumerate(train_loader):
                # === 添加调试信息：查看批次数据 ===
                if epoch == 0 and batch_idx == 0:  # 只在第一个epoch的第一个批次打印
                    log_message(f"\n第一个训练批次详情:")
                    log_message(f"- 输入形状: {inputs.shape}")
                    log_message(f"- 标签形状: {labels.shape}")
                    log_message(f"- 标签内容: {labels.tolist()}")
                    
                    # 计算批次中每个类别的数量
                    unique_labels, counts = torch.unique(labels, return_counts=True)
                    class_distribution = dict(zip(unique_labels.tolist(), counts.tolist()))
                    log_message(f"- 批次中的类别分布: {class_distribution}")
                
                # 将数据移动到指定设备（CPU或GPU）
                inputs, labels = inputs.to(device), labels.to(device)
                
                # 梯度清零（防止梯度累积）
                optimizer.zero_grad()
                
                # 前向传播
                outputs = model(inputs)
                
                # 处理不同模型的输出格式
                if hasattr(outputs, 'logits'):
                    # GoogLeNet输出
                    logits = outputs.logits
                elif isinstance(outputs, tuple) and len(outputs) > 1:
                    # InceptionV3输出 (通常是一个元组，主输出在第一个位置)
                    logits = outputs[0]
                else:
                    # 常规模型输出
                    logits = outputs
                
                # === 添加调试信息：查看模型输出 ===
                if epoch == 0 and batch_idx == 0:
                    log_message(f"\n模型输出详情:")
                    log_message(f"- 输出形状: {logits.shape}")
                    # 打印第一个样本的预测概率分布
                    if logits.shape[1] <= 10:  # 类别数不多时才打印
                        log_message(f"- 第一个样本的预测分数: {logits[0].tolist()}")
                        log_message(f"- 第一个样本的真实标签: {labels[0].item()}")
                
                # 计算损失
                loss = criterion(logits, labels)
                
                # 反向传播和优化
                loss.backward()
                optimizer.step()
                
                # 统计训练指标
                train_loss += loss.item() * inputs.size(0)  # 累加批次损失
                _, predicted = torch.max(logits, 1)  # 获取预测类别
                train_total += labels.size(0)  # 累加样本数
                train_correct += (predicted == labels).sum().item()  # 累加正确预测数
                train_preds.extend(predicted.cpu().numpy())  # 收集预测结果
                train_labels.extend(labels.cpu().numpy())  # 收集真实标签
                
            log_message("训练阶段完成!")
            train_loss = train_loss / train_total  # 计算平均损失
            train_acc = train_correct / train_total  # 计算准确率
            
        except Exception as e:
            error_msg = f"训练阶段出错: {e}"
            log_message(error_msg)
            traceback.print_exc()
            
            # 将错误信息保存到单独的错误日志文件
            if log_file:
                error_file = log_file.replace('.log', '_error.log')
                with open(error_file, 'a', encoding='utf-8') as f:
                    f.write(f"训练阶段出错: {e}\n")
                    f.write(traceback.format_exc())
            
            return model, history, 0, 0, {}
        
        # 验证阶段
        model.eval()  # 设置模型为评估模式（禁用dropout等）
        log_message("开始验证阶段...")
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        val_preds = []
        val_labels = []

        try:
            # 在评估模式下，不需要计算梯度
            with torch.no_grad():
                for batch_idx, (inputs, labels) in enumerate(val_loader):
                    # === 添加调试信息：查看验证批次 ===
                    if epoch == 0 and batch_idx == 0:
                        log_message(f"\n第一个验证批次详情:")
                        log_message(f"- 输入形状: {inputs.shape}")
                        log_message(f"- 标签形状: {labels.shape}")
                    
                    # 处理单样本批次问题（某些模型如BN层需要至少2个样本）
                    if inputs.shape[0] == 1:
                        # 复制样本以避免批次归一化问题
                        inputs = torch.cat([inputs, inputs], dim=0)
                        labels = torch.cat([labels, labels], dim=0)
                    
                    inputs, labels = inputs.to(device), labels.to(device)
                    
                    # 前向传播
                    outputs = model(inputs)
                    
                    # 处理不同模型的输出格式
                    if hasattr(outputs, 'logits'):
                        # GoogLeNet输出
                        logits = outputs.logits
                    elif isinstance(outputs, tuple) and len(outputs) > 1:
                        # InceptionV3输出 (通常是一个元组，主输出在第一个位置)
                        logits = outputs[0]
                    else:
                        # 常规模型输出
                        logits = outputs
                    
                    # 计算损失
                    loss = criterion(logits, labels)
                    
                    # 统计验证指标
                    val_loss += loss.item() * inputs.size(0)
                    _, predicted = torch.max(logits, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()
                    val_preds.extend(predicted.cpu().numpy())
                    val_labels.extend(labels.cpu().numpy())
            
            log_message("验证阶段完成!")
            val_loss = val_loss / val_total
            val_acc = val_correct / val_total
            
        except Exception as e:
            error_msg = f"验证阶段出错: {e}"
            log_message(error_msg)
            traceback.print_exc()
            
            # 将错误信息保存到单独的错误日志文件
            if log_file:
                error_file = log_file.replace('.log', '_error.log')
                with open(error_file, 'a', encoding='utf-8') as f:
                    f.write(f"验证阶段出错: {e}\n")
                    f.write(traceback.format_exc())
                    
            return model, history, 0, 0, {}
        
        # 计算混淆矩阵并更新 detail_matrix
        try:
            # 计算训练集和验证集的混淆矩阵
            train_cm = confusion_matrix(train_labels, train_preds)
            val_cm = confusion_matrix(val_labels, val_preds)
            
            # === 添加调试信息：查看混淆矩阵 ===
            if epoch == 0:
                log_message(f"\n第一个epoch的混淆矩阵:")
                log_message(f"训练集混淆矩阵:\n{train_cm}")
                log_message(f"验证集混淆矩阵:\n{val_cm}")
            
            # 从混淆矩阵中提取详细指标
            detail_matrix = {
                'train_malignant_correct': train_cm[1, 1] if train_cm.shape == (2, 2) else 0,
                'train_malignant_total': train_cm[1, :].sum() if train_cm.shape == (2, 2) else 0,
                'train_benign_correct': train_cm[0, 0] if train_cm.shape == (2, 2) else 0,
                'train_benign_total': train_cm[0, :].sum() if train_cm.shape == (2, 2) else 0,
                'train_malignant_acc': (train_cm[1, 1] / train_cm[1, :].sum()) if train_cm.shape == (2, 2) and train_cm[1, :].sum() > 0 else 0.0,
                'train_benign_acc': (train_cm[0, 0] / train_cm[0, :].sum()) if train_cm.shape == (2, 2) and train_cm[0, :].sum() > 0 else 0.0,

                'val_malignant_correct': val_cm[1, 1] if val_cm.shape == (2, 2) else 0,
                'val_malignant_total': val_cm[1, :].sum() if val_cm.shape == (2, 2) else 0,
                'val_benign_correct': val_cm[0, 0] if val_cm.shape == (2, 2) else 0,
                'val_benign_total': val_cm[0, :].sum() if val_cm.shape == (2, 2) else 0,
                'val_malignant_acc': (val_cm[1, 1] / val_cm[1, :].sum()) if val_cm.shape == (2, 2) and val_cm[1, :].sum() > 0 else 0.0,
                'val_benign_acc': (val_cm[0, 0] / val_cm[0, :].sum()) if val_cm.shape == (2, 2) and val_cm[0, :].sum() > 0 else 0.0
            }
            
            # === 添加调试信息：查看详细指标 ===
            if epoch == 0:
                log_message(f"\n第一个epoch的详细指标:")
                log_message(f"训练集良性准确率: {detail_matrix['train_benign_acc']:.4f} ({detail_matrix['train_benign_correct']}/{detail_matrix['train_benign_total']})")
                log_message(f"训练集恶性准确率: {detail_matrix['train_malignant_acc']:.4f} ({detail_matrix['train_malignant_correct']}/{detail_matrix['train_malignant_total']})")
                log_message(f"验证集良性准确率: {detail_matrix['val_benign_acc']:.4f} ({detail_matrix['val_benign_correct']}/{detail_matrix['val_benign_total']})")
                log_message(f"验证集恶性准确率: {detail_matrix['val_malignant_acc']:.4f} ({detail_matrix['val_malignant_correct']}/{detail_matrix['val_malignant_total']})")
            
        except Exception as e:
            error_msg = f"计算混淆矩阵出错: {e}"
            log_message(error_msg)
            traceback.print_exc()
            
            # 将错误信息保存到单独的错误日志文件
            if log_file:
                error_file = log_file.replace('.log', '_error.log')
                with open(error_file, 'a', encoding='utf-8') as f:
                    f.write(f"计算混淆矩阵出错: {e}\n")
                    f.write(traceback.format_exc())
                    
            detail_matrix = {
                'train_malignant_acc': 0.0, 'train_benign_acc': 0.0,
                'val_malignant_acc': 0.0, 'val_benign_acc': 0.0
            }

        # 保存最佳模型 - 根据验证准确率
        if val_acc > best_val_acc:
            log_message(f"发现更好的模型! 验证准确率从 {best_val_acc:.4f} 提升到 {val_acc:.4f}")
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_model_wts = model.state_dict().copy()
            best_detail_matrix = detail_matrix.copy()
        
        # 早停机制 - 监控验证集loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0  # 重置早停计数器
        else:
            counter += 1  # 增加早停计数器
            log_message(f"验证损失未改善，早停计数: {counter}/{patience}")
            
        # 记录历史
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['train_malignant_acc'].append(detail_matrix['train_malignant_acc'])
        history['train_benign_acc'].append(detail_matrix['train_benign_acc'])
        history['val_malignant_acc'].append(detail_matrix['val_malignant_acc'])
        history['val_benign_acc'].append(detail_matrix['val_benign_acc'])
        
        # 打印当前epoch的训练结果
        epoch_time = time.time() - start_time
        log_message(f'Epoch {epoch+1}/{num_epochs} | '
              f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | '
              f'LR: {current_lr:.8f} | Time: {epoch_time:.2f}s')
    
        # 检查是否早停
        if counter >= patience:
            log_message(f'Early stopping triggered after {epoch+1} epochs (validation loss not improved for {patience} epochs)')
            break
    
    # 训练结束，加载最佳模型权重
    if best_model_wts is not None:
        log_message(f"加载最佳模型权重 (验证准确率: {best_val_acc:.4f})")
        model.load_state_dict(best_model_wts)
    
    # === 添加调试信息：训练结束总结 ===
    log_message(f"\n=== 训练结束总结 ===")
    log_message(f"最佳验证准确率: {best_val_acc:.4f}")
    log_message(f"对应的训练准确率: {best_train_acc:.4f}")
    if best_detail_matrix:
        log_message(f"最佳模型的详细指标:")
        log_message(f"- 训练集良性准确率: {best_detail_matrix['train_benign_acc']:.4f} ({best_detail_matrix['train_benign_correct']}/{best_detail_matrix['train_benign_total']})")
        log_message(f"- 训练集恶性准确率: {best_detail_matrix['train_malignant_acc']:.4f} ({best_detail_matrix['train_malignant_correct']}/{best_detail_matrix['train_malignant_total']})")
        log_message(f"- 验证集良性准确率: {best_detail_matrix['val_benign_acc']:.4f} ({best_detail_matrix['val_benign_correct']}/{best_detail_matrix['val_benign_total']})")
        log_message(f"- 验证集恶性准确率: {best_detail_matrix['val_malignant_acc']:.4f} ({best_detail_matrix['val_malignant_correct']}/{best_detail_matrix['val_malignant_total']})")
    
    return model, history, best_val_acc, best_train_acc, best_detail_matrix

### train_single_model_fold训练单个模型

In [ ]:
def train_single_model_fold(   
    model_name,  # 模型名称，如'AlexNet'、'ResNet50'等
    fold,  # 当前训练的折数，用于交叉验证
    train_dir,  # 训练数据目录路径
    val_dir,  # 验证数据目录路径
    save_dir,  # 模型保存目录
    num_epochs=50,  # 训练轮数，默认50轮
    batch_size=16,  # 批次大小，默认16
    initial_lr=0.001,  # 初始学习率，默认0.001
    device='cuda',  # 训练设备，默认使用GPU
    num_classes=2,  # 分类数量，默认为2（二分类）
    num_workers=0,  # 数据加载线程数，默认0（不使用多线程）
    patience=10,  # 早停耐心值，如果验证集损失连续10轮未改善则停止训练
    seed=42):  # 随机种子，确保实验可重复性
    """
    训练单个模型的单个fold
    
    参数:
        model_name (str): 模型名称
        fold (int): 折数
        train_dir (str): 训练数据目录
        val_dir (str): 验证数据目录
        save_dir (str): 模型保存目录
        num_epochs (int): 训练轮数
        batch_size (int): 批次大小
        initial_lr (float): 初始学习率
        device (str): 训练设备
        num_classes (int): 分类数量
        num_workers (int): 数据加载线程数
        patience (int): 早停耐心值
        seed (int): 随机种子，默认42
        
    返回:
        dict: 训练历史记录
    """
    # 第一步：固定所有全局随机状态，确保实验可重复
    set_seed(seed)
    
    # 设置日志文件，用于记录训练过程
    log_file = os.path.join(save_dir, f'{model_name.lower()}_fold_{fold}.log')
    
    # 定义日志记录函数，同时打印到控制台和写入日志文件
    def log_message(message):
        print(message)  # 打印到控制台
        with open(log_file, 'a', encoding='utf-8') as f:  # 写入日志文件
            f.write(message + '\n')
    
    # 记录训练开始信息
    log_message(f"\n{'='*60}")
    log_message(f"训练 {model_name} - Fold {fold}")
    log_message(f"随机种子: {seed}")  # 记录种子到日志，方便以后查
    log_message(f"{'='*60}")
    
    # 1. 计算训练集的均值和标准差，用于数据标准化
    log_message("计算训练集的均值和标准差...")
    mean, std = calculate_mean_std(train_dir, model_name)  # 调用外部函数计算
    log_message(f"均值: {mean}")
    log_message(f"标准差: {std}")
    
    # 2. 创建数据转换（包括数据增强）
    # 根据模型选择不同的填充转换函数（InceptionV3需要299x299的输入，其他模型通常是224x224）
    if 'Inception' in model_name:
        pad_transform = pad_to_square_299_transform  # 填充到299x299
    else:
        pad_transform = pad_to_square_transform  # 填充到224x224
    
    # 训练数据增强 - 提高模型泛化能力
    train_transform = transforms.Compose([
        transforms.Lambda(pad_transform),  # 首先填充图像到正方形
        # 以下是数据增强操作
        transforms.RandomRotation(15),  # 随机旋转±15度
        transforms.RandomHorizontalFlip(p=0.5),  # 50%概率水平翻转
        transforms.RandomVerticalFlip(p=0.5),  # 50%概率垂直翻转
        transforms.ColorJitter(brightness=0.1, contrast=0.1),  # 亮度和对比度微调
        transforms.ToTensor(),  # 将图像转换为张量
        transforms.Normalize(mean=mean, std=std)  # 使用计算得到的均值和标准差进行标准化
    ])
    
    # 验证数据不需要增强，只需要基本处理
    val_transform = transforms.Compose([
        transforms.Lambda(pad_transform),  # 填充到正方形
        transforms.ToTensor(),  # 转换为张量
        transforms.Normalize(mean=mean, std=std)  # 标准化
    ])
    
    # 3. 加载数据集
    log_message("加载数据集...")
    # ImageFolder会自动根据文件夹结构加载数据和标签
    train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
    val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)
    
    # 创建固定种子的生成器，确保数据加载的随机性可重复
    generator = torch.Generator()
    generator.manual_seed(seed) 
    
    # 固定多进程worker的随机状态
    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    
    # 创建训练数据加载器
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True,  # 打乱训练数据
        num_workers=num_workers,  # 数据加载线程数
        drop_last=True,  # 丢弃最后一个不完整的批次
        generator=generator,  # 使用固定种子的生成器
        worker_init_fn=seed_worker  # 初始化worker的随机状态
    )
    
    # 创建验证数据加载器
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False,  # 验证数据不需要打乱
        num_workers=num_workers,
        worker_init_fn=seed_worker
    )
    
    # 记录数据集信息
    log_message(f"训练集样本数: {len(train_dataset)}")
    log_message(f"验证集样本数: {len(val_dataset)}")
    log_message(f"类别: {train_dataset.classes}")  # 输出类别名称
    
    # 4. 创建模型
    log_message(f"创建模型: {model_name}")
    # 使用外部函数根据模型名称创建对应的模型实例
    model = create_model(model_name, num_classes)
    model = model.to(device)  # 将模型移动到指定设备（CPU或GPU）
    
    # 5. 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()  # 交叉熵损失函数，适用于分类任务
    # 使用SGD优化器，带动量和权重衰减
    optimizer = optim.SGD(model.parameters(), lr=initial_lr, momentum=0.9, weight_decay=0.0005)
    
    # 6. 训练模型
    log_message(f"\n开始训练 {num_epochs} 个epoch...")
    start_time = time.time()  # 记录训练开始时间
    
    # 调用train_model函数进行实际训练
    model, history, best_val_acc, best_train_acc, best_detail_matrix = train_model(
        model, train_loader, val_loader, criterion, optimizer, 
        num_epochs, initial_lr, patience, device, log_file
    )
    
    # 计算训练时间
    end_time = time.time()
    training_time = (end_time - start_time) / 60  # 转换为分钟
    
    # 7. 保存最佳模型
    model_save_path = os.path.join(save_dir, f'{model_name.lower()}_fold_{fold}.pth')
    torch.save(model.state_dict(), model_save_path)  # 只保存模型参数，不保存整个模型
    log_message(f"✅ 模型已保存到: {model_save_path}")
    
    # 8. 绘制训练曲线
    plot_training_curves(history, model_name, fold, save_dir)
    log_message(f"✅ 训练曲线已保存到: {os.path.join(save_dir, 'plots')}")
    
    # 9. 打印详细结果
    log_message(f"\n{'='*60}")
    log_message(f"{model_name} - Fold {fold} 训练完成!")
    log_message(f"最佳验证准确率: {best_val_acc:.4f}")
    log_message(f"对应的训练准确率: {best_train_acc:.4f}")
    
    # 如果有详细指标，则打印
    if best_detail_matrix:
        log_message(f"详细指标:")
        log_message(f"  训练集:")
        # 打印良性样本的准确率
        log_message(f"    良性 - 正确: {best_detail_matrix.get('train_benign_correct', 0)}/{best_detail_matrix.get('train_benign_total', 0)}, "
              f"准确率: {best_detail_matrix.get('train_benign_acc', 0):.4f}")
        # 打印恶性样本的准确率
        log_message(f"    恶性 - 正确: {best_detail_matrix.get('train_malignant_correct', 0)}/{best_detail_matrix.get('train_malignant_total', 0)}, "
              f"准确率: {best_detail_matrix.get('train_malignant_acc', 0):.4f}")
        log_message(f"  验证集:")
        # 打印良性样本的准确率
        log_message(f"    良性 - 正确: {best_detail_matrix.get('val_benign_correct', 0)}/{best_detail_matrix.get('val_benign_total', 0)}, "
              f"准确率: {best_detail_matrix.get('val_benign_acc', 0):.4f}")
        # 打印恶性样本的准确率
        log_message(f"    恶性 - 正确: {best_detail_matrix.get('val_malignant_correct', 0)}/{best_detail_matrix.get('val_malignant_total', 0)}, "
              f"准确率: {best_detail_matrix.get('val_malignant_acc', 0):.4f}")
    
    log_message(f"训练时间: {training_time:.2f} 分钟")
    log_message(f"{'='*60}")
    
    # 返回训练历史记录
    return history

### train_all_models训练所有模型

In [ ]:
def train_all_models(model_list, cv_data_dir, output_dir, num_folds, 
                     num_epochs, batch_size, initial_lr, device, num_classes, 
                     num_workers, patience, save_log=True,
                     seed=42):
    """
    训练所有模型
    
    参数:
        model_list (list): 模型名称列表
        cv_data_dir (str): 交叉验证数据目录
        output_dir (str): 输出目录
        num_folds (int): 折数
        num_epochs (int): 训练轮数
        batch_size (int): 批次大小
        initial_lr (float): 初始学习率
        device (str): 训练设备
        num_classes (int): 分类数量
        num_workers (int): 数据加载线程数
        patience (int): 早停耐心值
        save_log (bool): 是否保存日志
        seed (int): 随机种子，默认42
        
    返回:
        dict: 训练结果统计
    """
    print(f"\n{'='*60}")
    print(f"开始训练所有模型")
    print(f"模型数量: {len(model_list)}")
    print(f"折数: {num_folds}")
    print(f"总训练次数: {len(model_list) * num_folds}")
    print(f"随机种子: {seed}")
    print(f"{'='*60}")
    
    # 创建总日志文件
    main_log_file = os.path.join(output_dir, "training_main.log")
    with open(main_log_file, 'w', encoding='utf-8') as f:
        f.write(f"训练开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"模型数量: {len(model_list)}\n")
        f.write(f"折数: {num_folds}\n")
        f.write(f"总训练次数: {len(model_list) * num_folds}\n")
        f.write(f"随机种子: {seed}\n")
        f.write(f"{'='*60}\n")
    
    total_start_time = time.time()
    training_logs = []
    
    for model_name in model_list:
        model_save_dir = os.path.join(output_dir, model_name)
        os.makedirs(model_save_dir, exist_ok=True)
        
        for fold in range(1, num_folds + 1):
            train_dir = os.path.join(cv_data_dir, f'fold_{fold}', 'train')
            val_dir = os.path.join(cv_data_dir, f'fold_{fold}', 'val')
            
            if not os.path.exists(train_dir) or not os.path.exists(val_dir):
                error_msg = f"❌ 错误: 数据路径不存在 - {train_dir} 或 {val_dir}"
                print(error_msg)
                with open(main_log_file, 'a', encoding='utf-8') as f:
                    f.write(error_msg + '\n')
                continue
            
            try:
                fold_start_time = time.time()
                history = train_single_model_fold(
                    model_name=model_name,
                    fold=fold,
                    train_dir=train_dir,
                    val_dir=val_dir,
                    save_dir=model_save_dir,
                    num_epochs=num_epochs,
                    batch_size=batch_size,
                    initial_lr=initial_lr,
                    device=device,
                    num_classes=num_classes,
                    num_workers=num_workers,
                    patience=patience,
                    seed=seed 
                )
                fold_time = (time.time() - fold_start_time) / 60
                
                # 记录日志
                log_entry = {
                    '训练时间': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    '模型': model_name,
                    '折数': fold,
                    '随机种子': seed, 
                    '最佳验证准确率': max(history['val_acc']) if history['val_acc'] else 0.0,  # 添加检查
                    '对应训练准确率': history['train_acc'][history['val_acc'].index(max(history['val_acc']))] if history['val_acc'] else 0.0,
                    '训练时长(分钟)': f"{fold_time:.2f}"
                }
                training_logs.append(log_entry)
                
                # 添加到主日志
                with open(main_log_file, 'a', encoding='utf-8') as f:
                    f.write(f"完成 {model_name} - Fold {fold} | "
                           f"最佳验证准确率: {log_entry['最佳验证准确率']:.4f} | "
                           f"训练时长: {log_entry['训练时长(分钟)']} 分钟\n")
                
            except Exception as e:
                error_msg = f"❌ 训练 {model_name} Fold {fold} 时出错: {e}"
                print(error_msg)
                traceback.print_exc()
                
                # 将错误信息保存到错误日志
                with open(main_log_file, 'a', encoding='utf-8') as f:
                    f.write(error_msg + '\n')
                    f.write(traceback.format_exc() + '\n')
                
                continue
    
    total_time = (time.time() - total_start_time) / 60
    
    completion_msg = f"\n{'='*60}\n✅ 所有模型训练完成!\n总训练时间: {total_time:.2f} 分钟\n{'='*60}"
    print(completion_msg)
    with open(main_log_file, 'a', encoding='utf-8') as f:
        f.write(completion_msg + '\n')
    
    # 保存日志
    if save_log and training_logs:
        log_path = os.path.join(output_dir, "training_log.xlsx")
        df = pd.DataFrame(training_logs)
        df.to_excel(log_path, index=False, engine='openpyxl')
        print(f"\n📝 训练日志已保存到: {log_path}")
    
    return {
        'total_time': total_time,
        'training_logs': training_logs,
        'output_dir': output_dir
    }

# 评估模型

## evaluator.py(补充好测试集)

In [ ]:
"""
模型评估模块
对训练好的模型进行全面评估，包括：
- 预测概率保存
- 混淆矩阵绘制
- ROC曲线绘制
- 多模型汇总对比
"""

# ============================================================
# 导入所需的库
# ============================================================
import os                          # 操作系统接口，用于文件路径拼接、目录创建等
import time                        # 计时器，用于记录评估总耗时
import traceback                   # 错误追踪，出错时打印详细的错误堆栈信息
import numpy as np                 # 数值计算库，用于数组操作
import pandas as pd                # 表格数据处理库，用于保存CSV结果
import torch                       # PyTorch深度学习框架
import torch.nn as nn              # 神经网络模块（此处导入但主要在其他模块用）
from torch.utils.data import DataLoader          # 数据加载器，按批次加载图片
from torchvision import transforms, datasets     # 图像变换工具 + ImageFolder数据集
import matplotlib.pyplot as plt    # 绘图库，用于画混淆矩阵、ROC曲线等
import seaborn as sns              # 高级绘图库，用于绘制热图
from sklearn.metrics import roc_curve, auc       # ROC曲线计算工具
from datetime import datetime      # 日期时间工具（导入备用）

# 从自定义模块导入
from shilab_classifier.models.model_definitions import create_model          # 创建模型结构的函数
from shilab_classifier.training.preprocessing import (
    pad_to_square_transform,          # 普通图片填充为正方形的变换（224×224用）
    pad_to_square_299_transform,      # Inception专用填充变换（299×299用）
    calculate_mean_std                # 计算训练集的均值和标准差，用于归一化
)
from shilab_classifier.evaluation.metrics import calculate_metrics, print_metrics
# calculate_metrics：计算准确率、AUC、MCC等所有性能指标
# print_metrics：格式化打印性能指标


# ============================================================
# 函数1：对数据集进行预测并保存详细结果
# ============================================================
def predict_and_save_probabilities(model, dataloader, dataset_name, save_dir, device='cuda'):
    """
    对数据集进行预测并保存详细结果

    参数:
        model:        训练好的模型（已加载权重）
        dataloader:   数据加载器（按批次提供图片和标签）
        dataset_name: 数据集名称，如 'train' / 'val' / 'test'
        save_dir:     结果保存目录
        device:       运算设备，'cuda'=GPU，'cpu'=CPU

    返回:
        dict: 包含accuracy、AUC、MCC等所有性能指标的字典
    """

    # ---------- 切换到评估模式 ----------
    model.eval()
    # 告诉模型"现在是评估阶段，不是训练阶段"
    # 效果：关闭Dropout（不随机丢弃神经元），BatchNorm使用全局统计量
    # 如果不切换，预测结果会因随机性而不稳定

    # ---------- 初始化结果容器 ----------
    all_logits = []   # 存储每个批次的原始输出值（未经softmax的分数）
    all_probs = []    # 存储每个批次的概率值（经过softmax后，加和为1）
    all_preds = []    # 存储每个批次的预测类别（0=良性，1=恶性）
    all_labels = []   # 存储每个批次的真实标签
    all_paths = []    # 存储每张图片的文件路径

    print(f"为 {dataset_name} 预测概率并保存结果...")

    # ---------- 关闭梯度计算 ----------
    with torch.no_grad():
        # 评估时不需要计算梯度（梯度只在训练时用于更新参数）
        # 关闭梯度计算可以：节省内存 + 加快推理速度（约快2倍）

        for batch_idx, (inputs, labels) in enumerate(dataloader):
            # enumerate(dataloader)：逐批次取数据
            # batch_idx：当前是第几个批次（从0开始）
            # inputs：这个批次的图片张量，形状 [batch_size, 3, 224, 224]
            # labels：这个批次的真实标签，形状 [batch_size]，值为0或1

            if batch_idx % 10 == 0:
                # 每处理10个批次打印一次进度，避免刷屏
                print(f"预测 {dataset_name} 批次 {batch_idx}/{len(dataloader)}")

            inputs = inputs.to(device)
            # 将图片数据移动到指定设备（GPU或CPU）
            # 注意：labels不需要移到GPU，因为后面只用于numpy计算

            # ---------- 前向传播（模型推理）----------
            outputs = model(inputs)
            # 将图片送入模型，得到输出
            # outputs形状：[batch_size, 2]
            # 例如：[[1.2, -0.3],   ← 第1张图：良性分数1.2，恶性分数-0.3
            #         [-0.5, 2.1]]  ← 第2张图：良性分数-0.5，恶性分数2.1
            # 这些原始分数叫做 logits（还不是概率）

            # ---------- 处理多输出情况 ----------
            if isinstance(outputs, tuple):
                outputs = outputs[0]
            # 某些模型（如InceptionV3）在训练时会输出多个值（主输出+辅助输出）
            # isinstance检查outputs是否是元组
            # 如果是元组，只取第一个元素（主输出）
            # 普通模型不会进入这个分支

            # ---------- 计算概率 ----------
            probs = torch.softmax(outputs, dim=1)
            # softmax把logits转换为概率（所有类别概率加和=1）
            # dim=1 表示在"类别"这个维度上做softmax
            # 例如：logits = [1.2, -0.3]
            #       softmax后 = [0.82, 0.18]  ← 82%良性，18%恶性

            _, predicted = torch.max(outputs, 1)
            # torch.max(outputs, 1)：在第1维（类别维度）找最大值
            # 返回两个值：最大值本身（用_忽略）和最大值的索引
            # 索引就是预测的类别：0=良性，1=恶性
            # 例如：outputs = [[1.2, -0.3], [-0.5, 2.1]]
            #       predicted = [0, 1]  ← 第1张预测良性，第2张预测恶性

            # ---------- 保存当前批次的结果 ----------
            all_logits.append(outputs.cpu().numpy())
            # .cpu()：把数据从GPU移回CPU（numpy不支持GPU张量）
            # .numpy()：把PyTorch张量转换为numpy数组
            # .append()：追加到列表中

            all_probs.append(probs.cpu().numpy())
            # 保存概率值（softmax后的结果）

            all_preds.append(predicted.cpu().numpy())
            # 保存预测类别（0或1）

            all_labels.append(labels.numpy())
            # labels本来就在CPU上，直接转numpy

            # ---------- 保存图片路径 ----------
            if hasattr(dataloader.dataset, 'samples'):
                # hasattr：检查dataset对象是否有'samples'属性
                # ImageFolder数据集有samples属性，存储所有图片的(路径, 标签)对
                # 如果是自定义数据集可能没有这个属性

                batch_paths = [dataloader.dataset.samples[i][0] for i in range(
                    batch_idx * dataloader.batch_size,
                    min((batch_idx + 1) * dataloader.batch_size, len(dataloader.dataset))
                )]
                # 计算当前批次对应的图片索引范围：
                # 起始索引 = batch_idx × batch_size
                # 结束索引 = min(下一批次起始, 总图片数)  ← min防止最后一批越界
                # [0]：取samples中的路径部分（samples[i]是(路径, 标签)的元组）
                all_paths.extend(batch_paths)
                # extend：把这个批次的路径列表追加到总路径列表

    # ============================================================
    # 合并所有批次的结果（从列表变成完整的数组）
    # ============================================================
    all_logits = np.concatenate(all_logits, axis=0)
    # np.concatenate：把多个数组沿axis=0（行方向）拼接
    # 之前：[[batch1结果], [batch2结果], ...]  每个是[16,2]的数组
    # 之后：一个完整的 [总图片数, 2] 的数组

    all_probs = np.concatenate(all_probs, axis=0)
    # 同上，拼接所有批次的概率，形状：[总图片数, 2]

    all_preds = np.concatenate(all_preds, axis=0)
    # 拼接所有批次的预测类别，形状：[总图片数]

    all_labels = np.concatenate(all_labels, axis=0)
    # 拼接所有批次的真实标签，形状：[总图片数]

    # ============================================================
    # 创建DataFrame，整理成表格形式保存
    # ============================================================
    results_df = pd.DataFrame({
        'image_path': all_paths if all_paths else [''] * len(all_labels),
        # 图片完整路径；如果没有路径信息则填空字符串

        'filename': [os.path.basename(p) if p else '' for p in all_paths] if all_paths else [''] * len(all_labels),
        # os.path.basename：从完整路径中提取文件名
        # 例如：'/data/cell/img001.jpg' → 'img001.jpg'

        'logit_benign': all_logits[:, 0],
        # 良性类别的原始分数（第0列）

        'logit_malignant': all_logits[:, 1],
        # 恶性类别的原始分数（第1列）

        'prob_benign': all_probs[:, 0],
        # 良性的预测概率（softmax后）

        'prob_malignant': all_probs[:, 1],
        # 恶性的预测概率（softmax后）
        # 注意：prob_benign + prob_malignant = 1.0

        'predicted_class': all_preds,
        # 预测类别：0=良性，1=恶性

        'true_label': all_labels
        # 真实标签：0=良性，1=恶性
    })

    # ---------- 保存概率结果CSV ----------
    prob_save_path = os.path.join(save_dir, f'{dataset_name}_probabilities.csv')
    # os.path.join：拼接目录和文件名，自动处理路径分隔符
    # 例如：save_dir='results/fold_1', dataset_name='test'
    #       → 'results/fold_1/test_probabilities.csv'

    results_df.to_csv(prob_save_path, index=False)
    # 保存为CSV文件，index=False表示不保存行号列

    print(f"概率结果已保存到 {prob_save_path}")

    # ============================================================
    # 计算性能指标
    # ============================================================
    metrics = calculate_metrics(all_labels, all_preds, all_probs[:, 1])
    # 调用自定义函数计算所有性能指标
    # 参数1：真实标签数组
    # 参数2：预测类别数组
    # 参数3：恶性类别的预测概率（用于计算AUC，AUC只需要正类的概率）
    # 返回：包含accuracy、sensitivity、specificity、AUC、MCC等的字典

    # ---------- 保存性能指标CSV ----------
    metrics_df = pd.DataFrame([{
        'dataset': dataset_name,              # 数据集名称（train/val/test）
        'accuracy': metrics['accuracy'],       # 总体准确率
        'balanced_accuracy': metrics['balanced_accuracy'],  # 平衡准确率（考虑类别不平衡）
        'sensitivity': metrics['sensitivity'], # 敏感性 = 恶性中预测正确的比例（召回率）
        'specificity': metrics['specificity'], # 特异性 = 良性中预测正确的比例
        'precision_benign': metrics['precision_benign'],    # 良性的精确率
        'recall_benign': metrics['recall_benign'],          # 良性的召回率
        'f1_benign': metrics['f1_benign'],                  # 良性的F1分数
        'precision_malignant': metrics['precision_malignant'],  # 恶性的精确率
        'recall_malignant': metrics['recall_malignant'],        # 恶性的召回率
        'f1_malignant': metrics['f1_malignant'],                # 恶性的F1分数
        'mcc': metrics['mcc'],                # MCC（马修斯相关系数，综合评估指标）
        'auc': metrics['auc'],                # AUC（ROC曲线下面积）
        'benign_correct': metrics['benign_correct'],    # 良性预测正确的数量
        'benign_total': metrics['benign_total'],        # 良性总数量
        'benign_acc': metrics['benign_acc'],            # 良性准确率
        'malignant_correct': metrics['malignant_correct'],  # 恶性预测正确的数量
        'malignant_total': metrics['malignant_total'],      # 恶性总数量
        'malignant_acc': metrics['malignant_acc']           # 恶性准确率
    }])

    metrics_save_path = os.path.join(save_dir, f'{dataset_name}_metrics.csv')
    metrics_df.to_csv(metrics_save_path, index=False)
    print(f"性能指标已保存到 {metrics_save_path}")

    # ============================================================
    # 绘制并保存混淆矩阵
    # ============================================================
    plt.figure(figsize=(8, 6))
    # 创建一个8×6英寸的画布

    sns.heatmap(metrics['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=['Benign', 'Malignant'],
                yticklabels=['Benign', 'Malignant'])
    # sns.heatmap：用颜色深浅表示数值大小的热图
    # metrics['confusion_matrix']：混淆矩阵，2×2的矩阵
    #   [[真良性预测为良性,  真良性预测为恶性],
    #    [真恶性预测为良性,  真恶性预测为恶性]]
    # annot=True：在每个格子里显示数字
    # fmt='d'：数字格式为整数（d=digit）
    # cmap='Blues'：颜色主题为蓝色渐变
    # xticklabels：X轴标签（预测标签）
    # yticklabels：Y轴标签（真实标签）

    plt.title(f'Confusion Matrix - {dataset_name}')  # 图标题
    plt.ylabel('True Label')                          # Y轴标签
    plt.xlabel('Predicted Label')                     # X轴标签

    cm_save_path = os.path.join(save_dir, f'{dataset_name}_confusion_matrix.png')
    plt.savefig(cm_save_path, dpi=300, bbox_inches='tight')
    # dpi=300：分辨率300，适合论文发表
    # bbox_inches='tight'：自动裁剪多余空白

    plt.close()
    # 关闭当前图，释放内存（避免多个图叠加显示）

    print(f"混淆矩阵已保存到 {cm_save_path}")

    return metrics
    # 返回性能指标字典，供外部函数使用


# ============================================================
# 函数2：绘制ROC曲线（同时展示训练集、验证集、测试集）
# ============================================================
def plot_roc_curve(train_probs, train_labels, val_probs, val_labels,
                   test_probs, test_labels, save_path):
    """
    绘制三条ROC曲线（训练集/验证集/测试集）并保存

    ROC曲线说明：
        X轴：假阳性率（FPR）= 良性被误判为恶性的比例
        Y轴：真阳性率（TPR）= 恶性被正确识别的比例
        AUC：曲线下面积，越接近1越好，0.5=随机猜测
    """
    plt.figure(figsize=(10, 8))
    # 创建10×8英寸的画布

    # 把三个数据集打包成列表，方便循环处理
    datasets = [
        ('train', train_probs, train_labels, 'blue'),   # 训练集用蓝色
        ('val', val_probs, val_labels, 'green'),         # 验证集用绿色
        ('test', test_probs, test_labels, 'red')         # 测试集用红色
    ]

    for name, probs, labels, color in datasets:
        # 逐个处理每个数据集
        if len(np.unique(labels)) > 1:
            # np.unique(labels)：获取标签中的唯一值
            # len > 1：确保标签中既有0又有1（如果全是同一类，无法计算ROC）
            fpr, tpr, _ = roc_curve(labels, probs)
            # roc_curve：计算ROC曲线的坐标点
            # labels：真实标签（0/1）
            # probs：恶性的预测概率（值越高=越可能是恶性）
            # 返回：fpr（假阳性率数组）、tpr（真阳性率数组）、_（阈值，不用）

            roc_auc = auc(fpr, tpr)
            # auc：计算曲线下面积（梯形积分法）

            plt.plot(fpr, tpr, color=color, lw=2,
                    label=f'{name} (AUC = {roc_auc:.4f})')
            # 绘制ROC曲线
            # lw=2：线宽为2
            # label：图例标签，显示数据集名称和AUC值（保留4位小数）

    # ---------- 绘制对角线（随机猜测基准线）----------
    plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random')
    # 从(0,0)到(1,1)的虚线
    # 表示随机猜测的性能（AUC=0.5）
    # 模型的ROC曲线应该在这条线的上方

    plt.xlim([0.0, 1.0])          # X轴范围0到1
    plt.ylim([0.0, 1.05])         # Y轴范围0到1.05（留一点空间显示图例）
    plt.xlabel('False Positive Rate', fontsize=12)   # X轴标签
    plt.ylabel('True Positive Rate', fontsize=12)    # Y轴标签
    plt.title('ROC Curves', fontsize=14)             # 图标题
    plt.legend(loc="lower right", fontsize=10)        # 图例放在右下角
    plt.grid(alpha=0.3)                               # 显示网格线，透明度0.3

    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"ROC曲线已保存到 {save_path}")


# ============================================================
# 函数3：评估单个模型的所有fold
# ============================================================
def evaluate_single_model(model_name, cv_data_dir, test_dir, models_dir,
                          output_dir, num_folds=5, batch_size=16,
                          num_workers=0, num_classes=2, device='cuda'):
    """
    对一个模型的所有交叉验证折进行完整评估

    参数:
        model_name:   模型名称，如 'MobileNetV2'
        cv_data_dir:  交叉验证数据根目录（包含fold_1到fold_5）
        test_dir:     独立测试集目录（所有fold共用同一个测试集）
        models_dir:   保存训练好的模型权重的目录
        output_dir:   评估结果输出目录
        num_folds:    交叉验证折数，默认5折
        batch_size:   每批次处理的图片数量
        num_workers:  数据加载的并行线程数（0=主线程加载）
        num_classes:  分类数量，默认2（良性/恶性）
        device:       运算设备
    """
    print(f"\n{'='*60}")
    print(f"评估模型: {model_name}")
    print(f"{'='*60}")

    # ---------- 创建结果保存目录 ----------
    model_results_dir = os.path.join(output_dir, model_name)
    os.makedirs(model_results_dir, exist_ok=True)
    # exist_ok=True：如果目录已存在不报错，直接使用

    roc_dir = os.path.join(model_results_dir, 'roc_curves')
    os.makedirs(roc_dir, exist_ok=True)
    # 专门存放ROC曲线图片的子目录

    all_results = []
    # 用于收集所有fold的评估结果

    # ============================================================
    # 逐fold评估
    # ============================================================
    for fold in range(1, num_folds + 1):
        # range(1, 6) → 1, 2, 3, 4, 5（5折交叉验证）
        print(f"\n{'='*60}")
        print(f"Fold {fold}")
        print(f"{'='*60}")

        # ---------- 创建当前fold的结果目录 ----------
        fold_results_dir = os.path.join(model_results_dir, f'fold_{fold}')
        os.makedirs(fold_results_dir, exist_ok=True)

        # ---------- 设置数据路径 ----------
        train_dir = os.path.join(cv_data_dir, f'fold_{fold}', 'train')
        val_dir = os.path.join(cv_data_dir, f'fold_{fold}', 'val')
        # 每个fold有自己的train和val数据
        # 例如：cv_data_dir/fold_1/train/  和  cv_data_dir/fold_1/val/

        # ---------- 检查路径是否存在 ----------
        if not os.path.exists(train_dir) or not os.path.exists(val_dir):
            print(f"错误: 数据目录不存在，跳过此折")
            continue
            # continue：跳过当前fold，继续下一个fold

        # ---------- 计算均值和标准差（用于归一化）----------
        print("计算均值和标准差...")
        mean, std = calculate_mean_std(train_dir, model_name)
        # 用训练集图片计算均值和标准差
        # 重要：评估时也用训练集的均值标准差来归一化，保持一致性
        print(f"均值: {mean}, 标准差: {std}")

        # ---------- 创建图像预处理流程 ----------
        if 'Inception' in model_name:
            pad_transform = pad_to_square_299_transform
            # InceptionV3需要299×299的输入
        else:
            pad_transform = pad_to_square_transform
            # 其他模型用224×224的输入

        transform = transforms.Compose([
            transforms.Lambda(pad_transform),
            # 第1步：填充图片为正方形（保持宽高比，用黑色填充）
            transforms.ToTensor(),
            # 第2步：把PIL图片转换为PyTorch张量，像素值从[0,255]变为[0,1]
            transforms.Normalize(mean=mean, std=std)
            # 第3步：标准化，使每个通道均值为0、标准差为1
            # 公式：(像素值 - mean) / std
        ])
        # transforms.Compose：把多个变换串联起来，按顺序执行

        # ---------- 加载数据集 ----------
        print("加载数据集...")
        train_dataset = datasets.ImageFolder(train_dir, transform=transform)
        # ImageFolder：自动从文件夹结构读取图片和标签
        # 文件夹结构：train_dir/benign/xxx.jpg  → 标签0
        #             train_dir/malignant/xxx.jpg → 标签1

        val_dataset = datasets.ImageFolder(val_dir, transform=transform)
        test_dataset = datasets.ImageFolder(test_dir, transform=transform)
        # 注意：val和test也用同样的transform（包括用训练集的mean/std归一化）

        train_loader = DataLoader(train_dataset, batch_size=batch_size,
                                 shuffle=False, num_workers=num_workers)
        # DataLoader：把数据集包装成可迭代的批次加载器
        # shuffle=False：评估时不打乱顺序（保证结果可复现、路径对应正确）
        # 训练时才需要shuffle=True

        val_loader = DataLoader(val_dataset, batch_size=batch_size,
                               shuffle=False, num_workers=num_workers)
        test_loader = DataLoader(test_dataset, batch_size=batch_size,
                                shuffle=False, num_workers=num_workers)

        # ---------- 查找模型权重文件 ----------
        model_dir = os.path.join(models_dir, model_name)

        # 尝试多种可能的文件命名格式（兼容不同训练代码的命名习惯）
        possible_names = [
            f'{model_name.lower()}_fold_{fold}.pth',  # 如：mobilenetv2_fold_1.pth
            f'{model_name}_fold_{fold}.pth',           # 如：MobileNetV2_fold_1.pth
            f'{model_name.lower()}_fold{fold}.pth',   # 如：mobilenetv2_fold1.pth
            f'{model_name}_fold{fold}.pth',            # 如：MobileNetV2_fold1.pth
            f'fold_{fold}.pth',                        # 如：fold_1.pth
            f'fold{fold}.pth',                         # 如：fold1.pth
            f'model_fold_{fold}.pth'                   # 如：model_fold_1.pth
        ]

        model_path = None
        for name in possible_names:
            temp_path = os.path.join(model_dir, name)
            if os.path.exists(temp_path):
                model_path = temp_path
                break
                # 找到第一个存在的文件就停止搜索

        if model_path is None:
            print(f"错误: 无法找到 Fold {fold} 的模型文件，跳过此折")
            continue

        # ---------- 加载模型 ----------
        print(f"加载模型权重: {model_path}")
        model = create_model(model_name, num_classes=num_classes)
        # 先创建模型结构（空的，参数随机）

        try:
            model.load_state_dict(torch.load(model_path, map_location=device))
            # torch.load：从文件加载保存的参数字典
            # map_location=device：如果训练时用GPU保存，加载时可以映射到CPU
            # load_state_dict：把加载的参数填入模型结构中

            model = model.to(device)
            # 把模型移到指定设备（GPU/CPU）

            model.eval()
            # 切换到评估模式

        except Exception as e:
            print(f"错误: 加载模型权重失败: {e}")
            traceback.print_exc()
            # 打印详细错误堆栈，方便调试
            continue

        # ---------- 对三个数据集分别评估 ----------
        train_metrics = predict_and_save_probabilities(
            model, train_loader, 'train', fold_results_dir, device
        )
        # 评估训练集（查看是否过拟合）

        val_metrics = predict_and_save_probabilities(
            model, val_loader, 'val', fold_results_dir, device
        )
        # 评估验证集（模型选择的依据）

        test_metrics = predict_and_save_probabilities(
            model, test_loader, 'test', fold_results_dir, device
        )
        # 评估测试集（最终性能报告）

        # ---------- 打印结果 ----------
        print(f"\n{model_name} Fold {fold} 结果:")
        print_metrics(train_metrics, "训练集")
        print_metrics(val_metrics, "验证集")
        print_metrics(test_metrics, "测试集")

        # ---------- 绘制ROC曲线 ----------
        # 从刚保存的CSV文件读取概率数据
        train_probs_df = pd.read_csv(os.path.join(fold_results_dir, 'train_probabilities.csv'))
        val_probs_df = pd.read_csv(os.path.join(fold_results_dir, 'val_probabilities.csv'))
        test_probs_df = pd.read_csv(os.path.join(fold_results_dir, 'test_probabilities.csv'))

        roc_save_path = os.path.join(roc_dir, f'roc_curve_fold_{fold}.png')
        plot_roc_curve(
            train_probs_df['prob_malignant'].values, train_probs_df['true_label'].values,
            val_probs_df['prob_malignant'].values, val_probs_df['true_label'].values,
            test_probs_df['prob_malignant'].values, test_probs_df['true_label'].values,
            roc_save_path
        )
        # .values：把pandas Series转换为numpy数组

        # ---------- 收集当前fold的所有指标 ----------
        fold_result = {
            'model': model_name,
            'fold': fold,
            # 训练集指标
            'train_accuracy': train_metrics['accuracy'],
            'train_balanced_accuracy': train_metrics['balanced_accuracy'],
            'train_sensitivity': train_metrics['sensitivity'],
            'train_specificity': train_metrics['specificity'],
            'train_precision_benign': train_metrics['precision_benign'],
            'train_recall_benign': train_metrics['recall_benign'],
            'train_f1_benign': train_metrics['f1_benign'],
            'train_precision_malignant': train_metrics['precision_malignant'],
            'train_recall_malignant': train_metrics['recall_malignant'],
            'train_f1_malignant': train_metrics['f1_malignant'],
            'train_mcc': train_metrics['mcc'],
            'train_auc': train_metrics['auc'],
            'train_benign_acc': train_metrics['benign_acc'],
            'train_malignant_acc': train_metrics['malignant_acc'],
            # 验证集指标（同上，前缀改为val_）
            'val_accuracy': val_metrics['accuracy'],
            'val_balanced_accuracy': val_metrics['balanced_accuracy'],
            'val_sensitivity': val_metrics['sensitivity'],
            'val_specificity': val_metrics['specificity'],
            'val_precision_benign': val_metrics['precision_benign'],
            'val_recall_benign': val_metrics['recall_benign'],
            'val_f1_benign': val_metrics['f1_benign'],
            'val_precision_malignant': val_metrics['precision_malignant'],
            'val_recall_malignant': val_metrics['recall_malignant'],
            'val_f1_malignant': val_metrics['f1_malignant'],
            'val_mcc': val_metrics['mcc'],
            'val_auc': val_metrics['auc'],
            'val_benign_acc': val_metrics['benign_acc'],
            'val_malignant_acc': val_metrics['malignant_acc'],
            # 测试集指标（同上，前缀改为test_）
            'test_accuracy': test_metrics['accuracy'],
            'test_balanced_accuracy': test_metrics['balanced_accuracy'],
            'test_sensitivity': test_metrics['sensitivity'],
            'test_specificity': test_metrics['specificity'],
            'test_precision_benign': test_metrics['precision_benign'],
            'test_recall_benign': test_metrics['recall_benign'],
            'test_f1_benign': test_metrics['f1_benign'],
            'test_precision_malignant': test_metrics['precision_malignant'],
            'test_recall_malignant': test_metrics['recall_malignant'],
            'test_f1_malignant': test_metrics['f1_malignant'],
            'test_mcc': test_metrics['mcc'],
            'test_auc': test_metrics['auc'],
            'test_benign_acc': test_metrics['benign_acc'],
            'test_malignant_acc': test_metrics['malignant_acc']
        }

        all_results.append(fold_result)
        # 把这个fold的结果加入总列表

    # ---------- 检查是否有成功的fold ----------
    if not all_results:
        print(f"警告: {model_name} 没有成功评估任何fold")
        return None

    # ============================================================
    # 汇总所有fold的结果
    # ============================================================
    results_df = pd.DataFrame(all_results)
    # 把列表转换为DataFrame，每行是一个fold的结果

    # ---------- 计算各指标的平均值 ----------
    numeric_cols = results_df.select_dtypes(include=[np.number]).columns
    # select_dtypes：选择数值类型的列（排除model、fold等字符串列）

    avg_result = results_df[numeric_cols].mean().to_dict()
    # .mean()：对每一列计算平均值（即5折的平均性能）
    # .to_dict()：转换为字典格式

    avg_result['model'] = model_name
    avg_result['fold'] = 'Average'
    # 添加模型名和标识（fold='Average'表示这是平均值行）

    # ---------- 把平均值行追加到DataFrame ----------
    avg_df = pd.DataFrame([avg_result])
    results_df = pd.concat([results_df, avg_df], ignore_index=True)
    # pd.concat：纵向拼接两个DataFrame
    # ignore_index=True：重新生成行索引（0,1,2,...）

    # ---------- 保存详细结果（每折+平均值）----------
    detailed_results_path = os.path.join(model_results_dir, f'{model_name}_detailed_results.csv')
    results_df.to_csv(detailed_results_path, index=False)
    print(f"\n{model_name} 详细结果已保存到 {detailed_results_path}")

    # ---------- 保存交叉验证汇总（只有平均值行）----------
    cv_results_path = os.path.join(model_results_dir, f'{model_name}_cross_validation_results.csv')
    avg_df.to_csv(cv_results_path, index=False)
    print(f"{model_name} 交叉验证结果已保存到 {cv_results_path}")

    return results_df


# ============================================================
# 函数4：评估所有模型（主入口函数）
# ============================================================
def evaluate_all_models(model_list, cv_data_dir, test_dir, models_dir,
                       output_dir, num_folds=5, batch_size=16, num_workers=0,
                       num_classes=2, device='cuda', save_summary=True):
    """
    批量评估所有模型，并生成汇总报告

    参数:
        model_list:   模型名称列表；若为None则自动扫描models_dir
        save_summary: 是否生成汇总报告（热图、排名表格等）
    """
    print(f"\n{'='*60}")
    print(f"开始评估所有模型")
    print(f"{'='*60}")

    os.makedirs(output_dir, exist_ok=True)
    # 确保输出目录存在

    # ---------- 自动检测模型列表 ----------
    if model_list is None:
        if not os.path.exists(models_dir):
            raise ValueError(f"模型目录不存在: {models_dir}")
            # raise：主动抛出异常，终止程序并显示错误信息

        model_list = [d for d in os.listdir(models_dir)
                     if os.path.isdir(os.path.join(models_dir, d))]
        # os.listdir：列出目录下的所有文件和文件夹名称
        # os.path.isdir：判断是否是文件夹（排除.pth文件等）
        # 列表推导式：只保留文件夹名称作为模型名

        if not model_list:
            raise ValueError(f"在 {models_dir} 中没有找到任何模型文件夹")

    print(f"\n找到 {len(model_list)} 个模型:")
    for model_name in model_list:
        print(f"  - {model_name}")

    # ---------- 逐个评估所有模型 ----------
    all_model_results = []
    start_time = time.time()
    # 记录开始时间，用于计算总耗时

    for model_name in model_list:
        try:
            results_df = evaluate_single_model(
                model_name=model_name,
                cv_data_dir=cv_data_dir,
                test_dir=test_dir,
                models_dir=models_dir,
                output_dir=output_dir,
                num_folds=num_folds,
                batch_size=batch_size,
                num_workers=num_workers,
                num_classes=num_classes,
                device=device
            )
            # 调用单模型评估函数

            if results_df is not None:
                avg_result = results_df[results_df['fold'] == 'Average'].iloc[0].to_dict()
                # 筛选fold=='Average'的行（平均值行）
                # .iloc[0]：取第一行（只有一行）
                # .to_dict()：转为字典
                all_model_results.append(avg_result)

        except Exception as e:
            print(f"评估 {model_name} 时出错: {e}")
            traceback.print_exc()
            continue
            # 某个模型出错不影响其他模型的评估

    total_time = (time.time() - start_time) / 60
    # 计算总耗时（秒转分钟）

    if not all_model_results:
        print("错误: 没有成功评估任何模型")
        return None

    # ---------- 保存汇总报告 ----------
    if save_summary:
        _save_evaluation_summary(all_model_results, output_dir)

    print(f"\n总运行时间: {total_time:.2f} 分钟")

    return {
        'total_time': total_time,
        'num_models': len(all_model_results),
        'results': all_model_results,
        'output_dir': output_dir
    }


# ============================================================
# 内部函数5：保存评估汇总结果
# ============================================================
def _save_evaluation_summary(all_model_results, output_dir):
    """
    保存所有模型的汇总结果，生成排名表格和热图
    函数名以_开头，表示这是内部函数，不建议外部直接调用
    """
    try:
        from tabulate import tabulate
        has_tabulate = True
    except ImportError:
        has_tabulate = False
        print("提示: 安装 tabulate 可以获得更好的表格显示效果")
    # tabulate：第三方库，用于在终端打印美观的表格
    # 用try/except处理未安装的情况，不影响主流程

    summary_df = pd.DataFrame(all_model_results)
    # 把所有模型的平均结果列表转为DataFrame

    # ---------- 只保留测试集指标 ----------
    test_cols = ['model'] + [col for col in summary_df.columns if col.startswith('test_')]
    # 筛选列名以'test_'开头的列（测试集指标）
    # 加上'model'列用于标识

    summary_df_test = summary_df[test_cols]

    # ---------- 按MCC降序排列 ----------
    summary_df_test = summary_df_test.sort_values('test_mcc', ascending=False)
    # sort_values：按指定列排序
    # ascending=False：降序（MCC越高越好，排在前面）

    # ---------- 保存汇总CSV ----------
    summary_path = os.path.join(output_dir, 'all_models_summary.csv')
    summary_df_test.to_csv(summary_path, index=False)
    print(f"\n所有模型的汇总结果已保存到 {summary_path}")

    if has_tabulate:
        print("\n所有模型的测试集性能指标汇总:")
        print(tabulate(summary_df_test, headers='keys', tablefmt='psql',
                      showindex=False, floatfmt='.4f'))
        # headers='keys'：用列名作为表头
        # tablefmt='psql'：PostgreSQL风格的表格边框
        # floatfmt='.4f'：浮点数保留4位小数

    # ---------- 绘制热图和Top模型分析 ----------
    _plot_performance_heatmap(summary_df_test, output_dir)
    _analyze_top_models(summary_df, summary_df_test, output_dir, top_n=4)


# ============================================================
# 内部函数6：绘制所有模型的性能热图
# ============================================================
def _plot_performance_heatmap(summary_df_test, output_dir):
    """绘制所有模型在测试集上的性能热图，方便横向对比"""
    plt.figure(figsize=(16, 10))

    # 选择要展示的指标
    metrics_to_plot = [
        'test_precision_benign', 'test_recall_benign', 'test_f1_benign',
        'test_precision_malignant', 'test_recall_malignant', 'test_f1_malignant',
        'test_balanced_accuracy', 'test_sensitivity', 'test_specificity',
        'test_mcc', 'test_auc'
    ]
    # 涵盖良性/恶性各自的精确率、召回率、F1，以及综合指标

    heatmap_data = summary_df_test[['model'] + metrics_to_plot].set_index('model')
    # set_index('model')：把model列设为行索引（热图的Y轴标签）

    sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd',
                cbar_kws={'label': 'Score'}, linewidths=0.5)
    # cmap='YlOrRd'：黄→橙→红的颜色渐变（值越大颜色越深）
    # cbar_kws：颜色条的设置
    # linewidths=0.5：格子之间的分隔线宽度

    plt.title('Model Performance Heatmap (Test Set)', fontsize=16, pad=20)
    plt.xlabel('Metrics', fontsize=12)
    plt.ylabel('Models', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    # rotation=45：X轴标签旋转45度，避免重叠
    # ha='right'：文字右对齐（配合旋转效果更好）
    plt.yticks(rotation=0)
    plt.tight_layout()
    # tight_layout：自动调整布局，防止标签被裁剪

    heatmap_path = os.path.join(output_dir, 'all_models_heatmap.png')
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"热图已保存到 {heatmap_path}")


# ============================================================
# 内部函数7：分析Top N模型的最佳折
# ============================================================
def _analyze_top_models(summary_df, summary_df_test, output_dir, top_n=4):
    """
    找出MCC最高的前N个模型，并在每个模型的所有fold中
    找出test_mcc最高的那一折，作为该模型的代表性最佳性能展示
    """
    try:
        from tabulate import tabulate
        has_tabulate = True
    except ImportError:
        has_tabulate = False

    # ---------- 找出MCC前N的模型 ----------
    top_models = summary_df_test.nlargest(top_n, 'test_mcc')
    # nlargest(n, col)：找出指定列值最大的n行
    top_model_names = top_models['model'].tolist()
    # .tolist()：把pandas Series转为Python列表

    print(f"\nMCC值前{top_n}的模型: {top_model_names}")

    best_folds_data = []
    # 存储每个Top模型的最佳折数据

    for model_name in top_model_names:
        # ---------- 读取该模型的详细结果文件 ----------
        model_eval_folder = os.path.join(output_dir, model_name)
        detailed_results_file = f"{model_name}_detailed_results.csv"
        detailed_results_path = os.path.join(model_eval_folder, detailed_results_file)

        if not os.path.exists(detailed_results_path):
            print(f"警告: 找不到详细结果文件 {detailed_results_path}，跳过模型 {model_name}")
            continue

        detailed_df = pd.read_csv(detailed_results_path)

        # ---------- 排除平均值行，只看实际的折 ----------
        fold_df = detailed_df[detailed_df['fold'] != 'Average']
        # 布尔索引：筛选fold列不等于'Average'的行

        if fold_df.empty:
            print(f"警告: {model_name} 的详细结果不包含任何折数据")
            continue

        # ---------- 找test_mcc最高的折 ----------
        max_mcc_idx = fold_df['test_mcc'].idxmax()
        # idxmax()：返回最大值所在行的索引（DataFrame的行号）

        best_fold = fold_df.loc[max_mcc_idx, 'fold']
        # .loc[行索引, 列名]：按标签定位具体的值
        best_mcc = fold_df.loc[max_mcc_idx, 'test_mcc']

        best_fold_data = fold_df.loc[max_mcc_idx].to_dict()
        # 取出这一行的所有数据，转为字典
        best_folds_data.append(best_fold_data)

        print(f"模型 {model_name}: 最佳折数 {best_fold}, MCC值 {best_mcc:.4f}")

    if not best_folds_data:
        print("警告: 无法找到任何模型的最佳折数据")
        return

    best_folds_df = pd.DataFrame(best_folds_data)

    # ---------- 选择关键指标展示 ----------
    key_metrics = ['model', 'fold',
                   'train_sensitivity', 'train_specificity', 'train_auc',
                   'val_sensitivity', 'val_specificity', 'val_auc',
                   'test_sensitivity', 'test_specificity', 'test_auc']
    # 只展示最核心的三个指标（sensitivity/specificity/auc）
    # 分别看训练集/验证集/测试集，方便判断过拟合情况

    available_metrics = [m for m in key_metrics if m in best_folds_df.columns]
    # 过滤掉不存在的列（防止KeyError）
    top_summary = best_folds_df[available_metrics]

    # ---------- 保存最佳折汇总 ----------
    top_path = os.path.join(output_dir, f'top{top_n}_models_summary.csv')
    top_summary.to_csv(top_path, index=False)
    print(f"\nMCC值前{top_n}的模型最佳折汇总结果已保存到 {top_path}")

    if has_tabulate:
        print(f"\nMCC值前{top_n}的模型最佳折性能指标汇总:")
        print(tabulate(top_summary, headers='keys', tablefmt='psql',
                      showindex=False, floatfmt='.4f'))

    # ---------- 绘制Top N模型的热图 ----------
    plt.figure(figsize=(12, 6))
    metrics_to_plot = [
        'test_precision_benign', 'test_recall_benign', 'test_f1_benign',
        'test_precision_malignant', 'test_recall_malignant', 'test_f1_malignant',
        'test_balanced_accuracy', 'test_sensitivity', 'test_specificity',
        'test_mcc', 'test_auc'
    ]

    available_plot_metrics = [m for m in metrics_to_plot if m in best_folds_df.columns]

    heatmap_df = best_folds_df.set_index('model')
    # 把model列设为行索引，作为热图Y轴标签
    top_heatmap_data = heatmap_df[available_plot_metrics]

    sns.heatmap(top_heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd',
                cbar_kws={'label': 'Score'}, linewidths=0.5)
    plt.title(f'Top {top_n} Models Best Fold Performance Heatmap (Test Set)', fontsize=14, pad=15)
    plt.xlabel('Metrics', fontsize=11)
    plt.ylabel('Models', fontsize=11)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()

    top_heatmap_path = os.path.join(output_dir, f'top{top_n}_models_heatmap.png')
    plt.savefig(top_heatmap_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"前{top_n}名模型最佳折热图已保存到 {top_heatmap_path}")


## metrics.py

In [ ]:
"""
评估指标计算模块
功能：根据模型的预测结果和真实标签，计算各种分类性能指标
"""

# ── 导入数值计算库 ──────────────────────────────────────────────────────────────
import numpy as np                          # 用于数组操作和数值计算

# ── 从sklearn导入评估指标函数 ────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,                         # 准确率（正确预测数 / 总预测数）
    balanced_accuracy_score,                # 平衡准确率（各类别准确率的平均值）
    precision_recall_fscore_support,        # 精确率、召回率、F1分数（此处导入备用，实际未直接调用）
    confusion_matrix,                       # 混淆矩阵（展示TP/TN/FP/FN的2×2矩阵）
    roc_auc_score,                          # AUC分数（此处导入备用，实际用roc_curve+auc手动计算）
    matthews_corrcoef,                      # MCC（马修斯相关系数，综合评价指标）
    roc_curve,                              # ROC曲线的FPR和TPR数据点
    auc                                     # 计算曲线下面积（配合roc_curve使用）
)


# ════════════════════════════════════════════════════════════════════════════════
# 函数1：calculate_metrics —— 核心指标计算函数
# ════════════════════════════════════════════════════════════════════════════════

def calculate_metrics(y_true, y_pred, y_prob=None):
    """
    计算各种性能指标

    参数:
        y_true : array-like, 真实标签，如 [0, 1, 0, 1, 1, ...]
                 0 = 良性(benign)，1 = 恶性(malignant)
        y_pred : array-like, 模型预测的类别标签，如 [0, 1, 1, 1, 0, ...]
        y_prob : array-like, 模型预测为"恶性"的概率，如 [0.2, 0.8, 0.7, ...]
                 用于计算AUC，可以为None（不计算AUC）

    返回:
        dict: 包含所有性能指标的字典
    """

    # ── 第1步：计算混淆矩阵 ──────────────────────────────────────────────────────
    # confusion_matrix(y_true, y_pred) 返回一个2×2矩阵：
    #
    #              预测=良性(0)   预测=恶性(1)
    # 真实=良性(0) [  TN(真阴性)    FP(假阳性) ]
    # 真实=恶性(1) [  FN(假阴性)    TP(真阳性) ]
    #
    # TN: 真的是良性，预测也是良性 → 预测正确
    # FP: 真的是良性，预测成了恶性 → 误报（把好的说成坏的）
    # FN: 真的是恶性，预测成了良性 → 漏报（把坏的说成好的）← 医学上最危险！
    # TP: 真的是恶性，预测也是恶性 → 预测正确
    cm = confusion_matrix(y_true, y_pred)

    # ── 第2步：检查混淆矩阵是否为标准的2×2形状 ──────────────────────────────────
    # 正常情况下cm.shape == (2, 2)
    # 但如果某个类别在这批数据中完全没有出现，矩阵可能变成(1,2)或(2,1)
    if cm.shape != (2, 2):
        print(f"警告: 混淆矩阵形状不是(2,2), 而是{cm.shape}")

        if cm.shape[0] == 2 and cm.shape[1] == 1:
            # 情况：有2个真实类别，但模型只预测出了1种类别
            # 例如：模型把所有图片都预测为良性，没有预测出恶性
            # 在右边补一列全0，凑成2×2
            cm = np.column_stack((cm, np.zeros(2)))

        elif cm.shape[0] == 1 and cm.shape[1] == 2:
            # 情况：数据中只有1种真实类别，但模型预测出了2种
            # 在下面补一行全0，凑成2×2
            cm = np.row_stack((cm, np.zeros(2)))

        else:
            # 其他异常情况（如数据完全异常），直接返回全零的指标字典
            # 避免后续计算报错
            return {
                'accuracy': 0,
                'balanced_accuracy': 0,
                'sensitivity': 0,
                'specificity': 0,
                'precision_benign': 0,
                'recall_benign': 0,
                'f1_benign': 0,
                'precision_malignant': 0,
                'recall_malignant': 0,
                'f1_malignant': 0,
                'mcc': 0,
                'auc': 0 if y_prob is not None else None,  # 有概率输入才返回0，否则返回None
                'benign_correct': 0,
                'benign_total': 0,
                'benign_acc': 0,
                'malignant_correct': 0,
                'malignant_total': 0,
                'malignant_acc': 0,
                'confusion_matrix': np.zeros((2, 2))       # 返回空的2×2矩阵
            }

    # ── 第3步：从混淆矩阵中提取四个基本数值 ────────────────────────────────────────
    # cm[行, 列]，行=真实标签，列=预测标签
    # cm[0,0] = TN（真阴性）：真实良性 且 预测良性
    # cm[0,1] = FP（假阳性）：真实良性 但 预测恶性
    # cm[1,0] = FN（假阴性）：真实恶性 但 预测良性  ← 漏诊，医学上最危险
    # cm[1,1] = TP（真阳性）：真实恶性 且 预测恶性
    tn, fp = cm[0, 0], cm[0, 1]   # 第0行：真实为良性的样本
    fn, tp = cm[1, 0], cm[1, 1]   # 第1行：真实为恶性的样本

    # ── 第4步：计算总体准确率 ────────────────────────────────────────────────────
    # 准确率 = 预测正确的数量 / 总数量
    # 公式：(TP + TN) / (TP + TN + FP + FN)
    # 注意：if条件是为了防止分母为0（数据集为空时）
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0

    # ── 第5步：计算各类别的单独准确率 ──────────────────────────────────────────────
    # 良性类别的准确率：
    #   正确预测为良性的数量(TN) / 真实良性的总数量(TN + FP)
    benign_correct = tn                                         # 良性预测正确的数量
    benign_total   = tn + fp                                    # 真实良性的总数量
    benign_acc     = tn / benign_total if benign_total > 0 else 0  # 良性准确率

    # 恶性类别的准确率：
    #   正确预测为恶性的数量(TP) / 真实恶性的总数量(TP + FN)
    malignant_correct = tp                                              # 恶性预测正确的数量
    malignant_total   = tp + fn                                         # 真实恶性的总数量
    malignant_acc     = tp / malignant_total if malignant_total > 0 else 0  # 恶性准确率

    # ── 第6步：计算灵敏度（Sensitivity）和特异性（Specificity） ─────────────────────
    # 灵敏度（敏感性）= TP / (TP + FN)
    # 含义：在所有真实恶性样本中，有多少被正确识别出来了？
    # 也叫：真阳性率（TPR）、召回率（Recall）
    # 医学意义：灵敏度低 → 漏诊率高 → 危险！
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

    # 特异性 = TN / (TN + FP)
    # 含义：在所有真实良性样本中，有多少被正确识别出来了？
    # 也叫：真阴性率（TNR）
    # 医学意义：特异性低 → 误诊率高 → 患者不必要地接受治疗
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    # ── 第7步：计算平衡准确率（Balanced Accuracy） ──────────────────────────────────
    # 平衡准确率 = (灵敏度 + 特异性) / 2
    # 为什么需要它？
    #   普通准确率在类别不平衡时会失真：
    #   比如：90张良性 + 10张恶性，模型全预测良性 → 准确率90%，但毫无用处
    #   平衡准确率 = (0/10 + 90/90) / 2 = 50% → 揭露了模型的真实问题
    balanced_accuracy = (sensitivity + specificity) / 2

    # ── 第8步：计算良性类别的精确率、召回率、F1分数 ─────────────────────────────────
    # 注意：这里把"良性(0)"当作"正类"来计算
    #
    # 精确率(Precision) = 预测为良性中，真的是良性的比例
    # 公式：TN / (TN + FN)
    # 分母：所有被预测为良性的样本（TN=真良性预测良性 + FN=真恶性预测良性）
    precision_benign = tn / (tn + fn) if (tn + fn) > 0 else 0

    # 召回率(Recall) = 真实良性中，被正确预测为良性的比例
    # 公式：TN / (TN + FP) = specificity（与特异性相同）
    recall_benign = tn / (tn + fp) if (tn + fp) > 0 else 0

    # F1分数 = 精确率和召回率的调和平均数
    # 公式：2 * P * R / (P + R)
    # 调和平均数的特点：如果P和R有一个很低，F1就会很低
    # 只有P和R都高，F1才高 → 比算术平均更严格
    f1_benign = (2 * precision_benign * recall_benign /
                 (precision_benign + recall_benign)
                 if (precision_benign + recall_benign) > 0 else 0)

    # ── 第9步：计算恶性类别的精确率、召回率、F1分数 ─────────────────────────────────
    # 注意：这里把"恶性(1)"当作"正类"来计算
    #
    # 精确率 = 预测为恶性中，真的是恶性的比例
    # 公式：TP / (TP + FP)
    # 分母：所有被预测为恶性的样本（TP=真恶性预测恶性 + FP=真良性预测恶性）
    precision_malignant = tp / (tp + fp) if (tp + fp) > 0 else 0

    # 召回率 = 真实恶性中，被正确预测为恶性的比例
    # 公式：TP / (TP + FN) = sensitivity（与灵敏度相同）
    recall_malignant = tp / (tp + fn) if (tp + fn) > 0 else 0

    # F1分数（恶性）
    f1_malignant = (2 * precision_malignant * recall_malignant /
                    (precision_malignant + recall_malignant)
                    if (precision_malignant + recall_malignant) > 0 else 0)

    # ── 第10步：计算MCC（马修斯相关系数） ───────────────────────────────────────────
    # MCC 是目前公认最全面的二分类单一指标
    # 公式：(TP*TN - FP*FN) / sqrt((TP+FP)(TP+FN)(TN+FP)(TN+FN))
    # 取值范围：-1 到 +1
    #   +1 = 完美预测
    #    0 = 等同于随机猜测
    #   -1 = 完全反向预测
    # 优点：同时考虑了TP/TN/FP/FN四个值，类别不平衡时也公平
    mcc = matthews_corrcoef(y_true, y_pred)

    # ── 第11步：计算AUC（ROC曲线下面积） ────────────────────────────────────────────
    auc_value = None                        # 默认为None（如果没有提供概率）

    if y_prob is not None:                  # 只有提供了预测概率才能计算AUC
        try:
            # roc_curve：根据不同的分类阈值，计算FPR和TPR
            # y_prob：模型预测为恶性(1)的概率值
            # fpr：假阳性率列表（x轴）
            # tpr：真阳性率列表（y轴）
            # _：对应的阈值列表（不需要，用_接收）
            fpr, tpr, _ = roc_curve(y_true, y_prob)

            # auc()：计算ROC曲线下的面积
            # 取值范围：0.5（随机猜测）到 1.0（完美分类）
            auc_value = auc(fpr, tpr)

        except Exception as e:
            # 如果计算过程中出错（如标签只有一个类别），打印错误并设为0
            print(f"计算AUC时出错: {e}")
            auc_value = 0

    # ── 第12步：将所有指标打包成字典返回 ────────────────────────────────────────────
    return {
        'accuracy':             accuracy,           # 总体准确率
        'balanced_accuracy':    balanced_accuracy,  # 平衡准确率（类别不平衡时更可靠）
        'sensitivity':          sensitivity,        # 灵敏度（恶性召回率，越高漏诊越少）
        'specificity':          specificity,        # 特异性（良性召回率，越高误诊越少）
        'precision_benign':     precision_benign,   # 良性精确率
        'recall_benign':        recall_benign,      # 良性召回率（= specificity）
        'f1_benign':            f1_benign,          # 良性F1分数
        'precision_malignant':  precision_malignant,# 恶性精确率
        'recall_malignant':     recall_malignant,   # 恶性召回率（= sensitivity）
        'f1_malignant':         f1_malignant,       # 恶性F1分数
        'mcc':                  mcc,                # MCC（综合最公平的指标）
        'auc':                  auc_value,          # AUC（ROC曲线下面积）
        'benign_correct':       int(benign_correct),# 良性预测正确的数量
        'benign_total':         int(benign_total),  # 良性样本总数
        'benign_acc':           benign_acc,         # 良性类别准确率
        'malignant_correct':    int(malignant_correct), # 恶性预测正确的数量
        'malignant_total':      int(malignant_total),   # 恶性样本总数
        'malignant_acc':        malignant_acc,          # 恶性类别准确率
        'confusion_matrix':     cm                      # 完整的2×2混淆矩阵
    }


# ════════════════════════════════════════════════════════════════════════════════
# 函数2：print_metrics —— 格式化打印所有指标
# ════════════════════════════════════════════════════════════════════════════════

def print_metrics(metrics, dataset_name="Dataset"):
    """
    将 calculate_metrics() 返回的指标字典格式化打印到控制台

    参数:
        metrics     : calculate_metrics() 返回的字典
        dataset_name: 数据集名称，用于打印标题（如"训练集"、"测试集"）
    """
    print(f"\n{dataset_name}:")                                          # 打印数据集名称作为标题

    # ── 基础准确率指标 ────────────────────────────────────────────────────────────
    print(f"准确率:      {metrics['accuracy']:.4f}")          # 保留4位小数打印准确率
    print(f"平衡准确率:  {metrics['balanced_accuracy']:.4f}") # 平衡准确率（类别不平衡时更有意义）

    # ── 灵敏度 & 特异性（医学诊断最关注的两个指标） ──────────────────────────────────
    print(f"灵敏度:      {metrics['sensitivity']:.4f}")        # 恶性识别率，越高越好
    print(f"特异性:      {metrics['specificity']:.4f}")        # 良性识别率，越高越好

    # ── 良性类别的精确率、召回率、F1 ─────────────────────────────────────────────────
    print(f"良性精确度:  {metrics['precision_benign']:.4f}")   # 预测为良性中真的是良性的比例
    print(f"良性召回率:  {metrics['recall_benign']:.4f}")      # 真实良性中被正确预测的比例
    print(f"良性F1分数:  {metrics['f1_benign']:.4f}")          # 良性的精确率和召回率的调和平均

    # ── 恶性类别的精确率、召回率、F1 ─────────────────────────────────────────────────
    print(f"恶性精确度:  {metrics['precision_malignant']:.4f}")# 预测为恶性中真的是恶性的比例
    print(f"恶性召回率:  {metrics['recall_malignant']:.4f}")   # 真实恶性中被正确预测的比例
    print(f"恶性F1分数:  {metrics['f1_malignant']:.4f}")       # 恶性的精确率和召回率的调和平均

    # ── 综合评价指标 ──────────────────────────────────────────────────────────────
    print(f"MCC:         {metrics['mcc']:.4f}")                # 马修斯相关系数，最综合的单一指标

    # AUC只有在提供了预测概率时才有值，否则为None，需要判断
    if metrics['auc'] is not None:
        print(f"AUC:         {metrics['auc']:.4f}")            # ROC曲线下面积，越接近1越好

    # ── 各类别详细统计 ────────────────────────────────────────────────────────────
    # 格式：准确率 (正确数/总数)，方便直观看到样本量
    print(f"良性准确率:  {metrics['benign_acc']:.4f} "
          f"({metrics['benign_correct']}/{metrics['benign_total']})")

    print(f"恶性准确率:  {metrics['malignant_acc']:.4f} "
          f"({metrics['malignant_correct']}/{metrics['malignant_total']})")

    # ── 混淆矩阵（最直观的结果展示） ──────────────────────────────────────────────
    # 输出格式：
    # [[TN  FP]
    #  [FN  TP]]
    print(f"混淆矩阵:\n{metrics['confusion_matrix']}")


# 测试数据

In [2]:
# MODELS_TO_TRAIN = [
# "DenseNet161", "ConvNextLarge", "InceptionV3", "MobileNetV3"
# ]
MODELS_TO_TRAIN = [
"DenseNet161"
]
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"开始时间: {current_time}")
print(f"开始训练 {len(MODELS_TO_TRAIN)} 个模型，每个模型训练 {NUM_FOLDS} 折")
print(f"总训练次数: {len(MODELS_TO_TRAIN)} × {NUM_FOLDS} = {len(MODELS_TO_TRAIN) * NUM_FOLDS}")
train_result = train_all_models(
    model_list=MODELS_TO_TRAIN,
    cv_data_dir=CV_DATA_DIR,
    output_dir=MODELS_DIR,            
    num_folds=NUM_FOLDS,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    initial_lr=INITIAL_LR,
    device=DEVICE,
    num_classes=NUM_CLASSES,
    num_workers=NUM_WORKERS,
    patience=PATIENCE
)
print("\n✅ 所有模型训练完成！") 

开始时间: 2026-03-30 20:51:01
开始训练 1 个模型，每个模型训练 3 折
总训练次数: 1 × 3 = 3

开始训练所有模型
模型数量: 1
折数: 3
总训练次数: 3
随机种子: 42

训练 DenseNet161 - Fold 1
随机种子: 42
计算训练集的均值和标准差...
开始计算均值和标准差，数据目录: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\cross_validation_data\fold_1\train
使用 224x224 图像大小计算均值和标准差
数据集加载成功，共 62 张图像
处理批次 0/4
均值和标准差计算完成: mean=tensor([0.5873, 0.5659, 0.7208]), std=tensor([0.2554, 0.2382, 0.1304])
均值: tensor([0.5873, 0.5659, 0.7208])
标准差: tensor([0.2554, 0.2382, 0.1304])
加载数据集...
训练集样本数: 62
验证集样本数: 32
类别: ['benign', 'malignant']
创建模型: DenseNet161


c:\ProgramData\miniconda3\envs\shilab\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\ProgramData\miniconda3\envs\shilab\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet161_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet161_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



开始训练 3 个epoch...
Epoch 1/3 开始时间: 20:51:02
当前学习率: 0.00100000
开始训练阶段...
训练数据批次数: 10
训练阶段完成!
开始验证阶段...
验证阶段完成!
Epoch 1/3 | Train Loss: 0.7437 | Train Acc: 0.4667 | Val Loss: 0.6284 | Val Acc: 0.5625 | LR: 0.00100000 | Time: 20.58s
Epoch 2/3 开始时间: 20:51:23
当前学习率: 0.00075000
开始训练阶段...
训练数据批次数: 10
训练阶段完成!
开始验证阶段...
验证阶段完成!
Epoch 2/3 | Train Loss: 0.6101 | Train Acc: 0.6500 | Val Loss: 0.5302 | Val Acc: 0.6562 | LR: 0.00075000 | Time: 20.04s
Epoch 3/3 开始时间: 20:51:43
当前学习率: 0.00025000
开始训练阶段...
训练数据批次数: 10
训练阶段完成!
开始验证阶段...
验证阶段完成!
Epoch 3/3 | Train Loss: 0.5738 | Train Acc: 0.7167 | Val Loss: 0.4716 | Val Acc: 0.7500 | LR: 0.00025000 | Time: 19.34s
✅ 模型已保存到: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\train_val_models\DenseNet161\densenet161_fold_1.pth
✅ 训练曲线已保存到: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\train_val_models\DenseNet161\plots

DenseNet161 - Fold 1 训练完成!
最佳验证准确率: 0.7500
对应的训练准确率: 0.7167
详细指标:
  训练集:
    良性 -

c:\ProgramData\miniconda3\envs\shilab\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\ProgramData\miniconda3\envs\shilab\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet161_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet161_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



开始训练 3 个epoch...
Epoch 1/3 开始时间: 20:52:05
当前学习率: 0.00100000
开始训练阶段...
训练数据批次数: 10
训练阶段完成!
开始验证阶段...
验证阶段完成!
Epoch 1/3 | Train Loss: 0.7070 | Train Acc: 0.5833 | Val Loss: 0.6862 | Val Acc: 0.6250 | LR: 0.00100000 | Time: 20.23s
Epoch 2/3 开始时间: 20:52:25
当前学习率: 0.00075000
开始训练阶段...
训练数据批次数: 10
训练阶段完成!
开始验证阶段...
验证阶段完成!
Epoch 2/3 | Train Loss: 0.5300 | Train Acc: 0.7667 | Val Loss: 0.6630 | Val Acc: 0.5938 | LR: 0.00075000 | Time: 20.05s
Epoch 3/3 开始时间: 20:52:45
当前学习率: 0.00025000
开始训练阶段...
训练数据批次数: 10
训练阶段完成!
开始验证阶段...
验证阶段完成!
Epoch 3/3 | Train Loss: 0.4648 | Train Acc: 0.7667 | Val Loss: 0.7205 | Val Acc: 0.5625 | LR: 0.00025000 | Time: 20.17s
✅ 模型已保存到: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\train_val_models\DenseNet161\densenet161_fold_2.pth
✅ 训练曲线已保存到: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\train_val_models\DenseNet161\plots

DenseNet161 - Fold 2 训练完成!
最佳验证准确率: 0.6250
对应的训练准确率: 0.5833
详细指标:
  训练集:
    良性 -

c:\ProgramData\miniconda3\envs\shilab\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\ProgramData\miniconda3\envs\shilab\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet161_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet161_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



开始训练 3 个epoch...
Epoch 1/3 开始时间: 20:53:08
当前学习率: 0.00100000
开始训练阶段...
训练数据批次数: 10
训练阶段完成!
开始验证阶段...
验证阶段完成!
Epoch 1/3 | Train Loss: 0.7876 | Train Acc: 0.5167 | Val Loss: 0.8447 | Val Acc: 0.3750 | LR: 0.00100000 | Time: 20.23s
Epoch 2/3 开始时间: 20:53:29
当前学习率: 0.00075000
开始训练阶段...
训练数据批次数: 10
训练阶段完成!
开始验证阶段...
验证阶段完成!
Epoch 2/3 | Train Loss: 0.6567 | Train Acc: 0.6333 | Val Loss: 1.1686 | Val Acc: 0.3438 | LR: 0.00075000 | Time: 19.53s
Epoch 3/3 开始时间: 20:53:48
当前学习率: 0.00025000
开始训练阶段...
训练数据批次数: 10
训练阶段完成!
开始验证阶段...
验证阶段完成!
Epoch 3/3 | Train Loss: 0.4775 | Train Acc: 0.7667 | Val Loss: 0.6621 | Val Acc: 0.5312 | LR: 0.00025000 | Time: 20.04s
✅ 模型已保存到: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\train_val_models\DenseNet161\densenet161_fold_3.pth
✅ 训练曲线已保存到: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\train_val_models\DenseNet161\plots

DenseNet161 - Fold 3 训练完成!
最佳验证准确率: 0.5312
对应的训练准确率: 0.7667
详细指标:
  训练集:
    良性 -

# 推理数据

## 带标签

In [5]:
"""
使用训练好的二分类模型对新收集的数据进行识别处理
"""

import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 导入自定义模块
from models import create_model  # 假设您有一个create_model函数用于创建模型
from shilab_classifier.training.preprocessing import pad_to_square_transform, pad_to_square_299_transform, calculate_mean_std
from shilab_classifier.evaluation.metrics import calculate_metrics, print_metrics

# ============================================================
# 配置参数
# ============================================================
# 这些参数可以根据需要修改
MODEL_NAME = "DenseNet161"  # 使用的模型名称，应该是您训练好的模型之一
MODEL_PATH = r"G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\train_val_models\DenseNet161\densenet161_fold_1.pth"  # 模型权重文件路径（选择性能最好的一折）
NEW_DATA_DIR = r"G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\newdata"  # 新收集的数据目录
OUTPUT_DIR = r"G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\new_data_predictions"  # 预测结果保存目录
BATCH_SIZE = 16
NUM_WORKERS = 0
NUM_CLASSES = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# 主函数
# ============================================================
def predict_new_data():
    """对新收集的数据进行预测并分析结果"""
    print(f"\n{'='*60}")
    print(f"使用 {MODEL_NAME} 模型预测新数据")
    print(f"{'='*60}")
    
    # 创建输出目录
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 加载模型
    print(f"加载模型权重: {MODEL_PATH}")
    model = create_model(MODEL_NAME, num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()  # 设置为评估模式
    
    # 使用训练时的均值和标准差  
    if MODEL_NAME == "DenseNet161":
        # InceptionV3的标准ImageNet归一化参数
        mean = [0.5873, 0.5659, 0.7208]
        std = [0.2554, 0.2382, 0.1304]
    else:
        # 其他模型的标准ImageNet归一化参数
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]

    print(f"使用训练时的均值: {mean}, 标准差: {std}")
    
    # 创建数据转换
    if 'Inception' in MODEL_NAME:
        pad_transform = pad_to_square_299_transform
    else:
        pad_transform = pad_to_square_transform
    
    transform = transforms.Compose([
        transforms.Lambda(pad_transform),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])
    
    # 加载新数据
    print(f"加载新数据: {NEW_DATA_DIR}")
    try:
        dataset = datasets.ImageFolder(NEW_DATA_DIR, transform=transform)
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, 
                               shuffle=False, num_workers=NUM_WORKERS)
        
        # 获取类别名称映射
        class_to_idx = dataset.class_to_idx  # 例如 {'benign': 0, 'malignant': 1}
        idx_to_class = {v: k for k, v in class_to_idx.items()}  # 反向映射
        
        print(f"找到 {len(dataset)} 张图片，分为 {len(class_to_idx)} 个类别")
        for class_name, idx in class_to_idx.items():
            print(f"  - 类别 {idx}: {class_name}")
            
    except Exception as e:
        print(f"加载数据时出错: {e}")
        print("请确保数据目录结构正确，应包含每个类别的子文件夹")
        return
    
    # 进行预测
    print("\n开始预测...")
    all_paths = []
    all_logits = []
    all_probs = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(dataloader):
            if batch_idx % 10 == 0:
                print(f"预测批次 {batch_idx}/{len(dataloader)}")
            
           # 获取当前批次的图像路径
            batch_size = inputs.size(0)  # 当前批次的实际大小
            start_idx = batch_idx * BATCH_SIZE
            batch_paths = [dataset.samples[i][0] for i in range(start_idx, start_idx + batch_size)]
            
            # 前向传播
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            
            # 处理可能的多输出情况
            if isinstance(outputs, tuple):
                outputs = outputs[0]
            
            # 计算概率和预测类别
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            # 保存结果
            all_paths.extend(batch_paths)
            all_logits.append(outputs.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_preds.append(predicted.cpu().numpy())
            all_labels.append(labels.numpy())
    
    # 合并所有批次的结果
    all_logits = np.concatenate(all_logits, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    # 创建详细结果DataFrame
    results_df = pd.DataFrame({
        'image_path': all_paths,
        'filename': [os.path.basename(p) for p in all_paths],
        'true_label_idx': all_labels,
        'true_label': [idx_to_class[idx] for idx in all_labels],
        'predicted_idx': all_preds,
        'predicted_class': [idx_to_class[idx] for idx in all_preds],
        'logit_benign': all_logits[:, 0],
        'logit_malignant': all_logits[:, 1],
        'prob_benign': all_probs[:, 0],
        'prob_malignant': all_probs[:, 1],
        'correct': all_labels == all_preds
    })
    
    # 保存详细预测结果
    results_path = os.path.join(OUTPUT_DIR, 'prediction_results.csv')
    results_df.to_csv(results_path, index=False)
    print(f"\n详细预测结果已保存到 {results_path}")
    
    # 计算性能指标
    metrics = calculate_metrics(all_labels, all_preds, all_probs[:, 1])
    
    # 打印性能指标
    print("\n新数据预测性能指标:")
    print_metrics(metrics, "新数据集")
    
    # 保存性能指标
    metrics_df = pd.DataFrame([{
        'dataset': 'new_data',
        'accuracy': metrics['accuracy'],
        'balanced_accuracy': metrics['balanced_accuracy'],
        'sensitivity': metrics['sensitivity'],
        'specificity': metrics['specificity'],
        'precision_benign': metrics['precision_benign'],
        'recall_benign': metrics['recall_benign'],
        'f1_benign': metrics['f1_benign'],
        'precision_malignant': metrics['precision_malignant'],
        'recall_malignant': metrics['recall_malignant'],
        'f1_malignant': metrics['f1_malignant'],
        'mcc': metrics['mcc'],
        'auc': metrics['auc'],
        'benign_correct': metrics['benign_correct'],
        'benign_total': metrics['benign_total'],
        'benign_acc': metrics['benign_acc'],
        'malignant_correct': metrics['malignant_correct'],
        'malignant_total': metrics['malignant_total'],
        'malignant_acc': metrics['malignant_acc']
    }])
    
    metrics_path = os.path.join(OUTPUT_DIR, 'performance_metrics.csv')
    metrics_df.to_csv(metrics_path, index=False)
    print(f"性能指标已保存到 {metrics_path}")
    
    # 绘制混淆矩阵
    plt.figure(figsize=(8, 6))
    sns.heatmap(metrics['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=list(class_to_idx.keys()),
                yticklabels=list(class_to_idx.keys()))
    plt.title('Confusion Matrix - New Data')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"混淆矩阵已保存到 {cm_path}")
    
    # # 分析错误预测的样本
    # analyze_misclassifications(results_df, OUTPUT_DIR)
    
    # 绘制ROC曲线
    plot_roc_curve(all_labels, all_probs[:, 1], os.path.join(OUTPUT_DIR, 'roc_curve.png'))
    
    print("\n✅ 预测完成！")
    return results_df

# ============================================================
# 辅助函数
# ============================================================
def analyze_misclassifications(results_df, output_dir):
    """分析错误分类的样本"""
    # 找出错误分类的样本
    errors_df = results_df[~results_df['correct']]
    
    if len(errors_df) == 0:
        print("\n所有样本都被正确分类！")
        return
    
    # 保存错误分类的样本信息
    errors_path = os.path.join(output_dir, 'misclassified_samples.csv')
    errors_df.to_csv(errors_path, index=False)
    print(f"\n找到 {len(errors_df)} 个错误分类的样本，详情已保存到 {errors_path}")
    
    # 按类别统计错误
    error_types = errors_df.groupby(['true_label', 'predicted_class']).size().reset_index()
    error_types.columns = ['真实类别', '预测类别', '数量']
    
    print("\n错误类型统计:")
    print(error_types)
    
    # 保存错误类型统计
    error_types_path = os.path.join(output_dir, 'error_types.csv')
    error_types.to_csv(error_types_path, index=False)
    
    # 找出最高置信度的错误样本
    high_conf_errors = errors_df.copy()
    
    # 计算错误预测的置信度
    for idx, row in high_conf_errors.iterrows():
        if row['predicted_idx'] == 0:  # 预测为良性
            high_conf_errors.loc[idx, 'error_confidence'] = row['prob_benign']
        else:  # 预测为恶性
            high_conf_errors.loc[idx, 'error_confidence'] = row['prob_malignant']
    
    # 按错误置信度排序
    high_conf_errors = high_conf_errors.sort_values('error_confidence', ascending=False)
    
    # 保存高置信度错误
    high_conf_path = os.path.join(output_dir, 'high_confidence_errors.csv')
    high_conf_errors.to_csv(high_conf_path, index=False)
    print(f"高置信度错误样本已保存到 {high_conf_path}")

def plot_roc_curve(y_true, y_prob, save_path):
    """绘制ROC曲线"""
    from sklearn.metrics import roc_curve, auc
    
    # 计算ROC曲线点
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    
    # 绘制ROC曲线
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    
    # 保存ROC曲线
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"ROC曲线已保存到 {save_path}")

# ============================================================
# 主程序入口
# ============================================================
if __name__ == "__main__":
    predict_new_data()
    # 在预测完成后调用函数处理恶性预测的图片
    malignant_output_dir = os.path.join(OUTPUT_DIR, 'malignant_images')
    # 方案二：创建带有置信度的副本（推荐）
    copy_malignant_images_with_confidence(results_df, malignant_output_dir, confidence_threshold=0.5)


使用 DenseNet161 模型预测新数据
加载模型权重: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\train_val_models\DenseNet161\densenet161_fold_1.pth
使用训练时的均值: [0.5873, 0.5659, 0.7208], 标准差: [0.2554, 0.2382, 0.1304]
加载新数据: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\newdata
加载数据时出错: Couldn't find any class folder in G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\newdata.
请确保数据目录结构正确，应包含每个类别的子文件夹


NameError: name 'results_df' is not defined

## 创建带置信度的副本

In [4]:
def copy_malignant_images_with_confidence(results_df, output_folder, confidence_threshold=0.0):
    """
    将预测为恶性的图片复制到指定文件夹，并在文件名中添加置信度值
    
    参数:
        results_df: 包含预测结果的DataFrame
        output_folder: 输出文件夹路径
        confidence_threshold: 置信度阈值，只处理高于此阈值的预测(默认0.0表示处理所有恶性预测)
    
    返回:
        复制的文件数量
    """
    import shutil
    
    # 确保输出文件夹存在
    os.makedirs(output_folder, exist_ok=True)
    
    # 筛选预测为恶性的图片
    malignant_predictions = results_df[
        (results_df['predicted_class'] == 'malignant') & 
        (results_df['prob_malignant'] >= confidence_threshold)
    ]
    
    print(f"\n找到 {len(malignant_predictions)} 张预测为恶性的图片 (置信度 >= {confidence_threshold})")
    
    if len(malignant_predictions) == 0:
        print("没有找到符合条件的恶性预测图片")
        return 0
    
    # 复制文件
    copied_count = 0
    copied_files = []
    
    for idx, row in malignant_predictions.iterrows():
        original_path = row['image_path']
        
        # 获取文件名
        filename = os.path.basename(original_path)
        
        # 分离文件名和扩展名
        name, ext = os.path.splitext(filename)
        
        # 创建新文件名: 原名_malignant_置信度.扩展名
        confidence_str = f"{row['prob_malignant']:.4f}".replace("0.", "")  # 去掉前导0
        new_filename = f"{name}_malignant_{confidence_str}{ext}"
        new_path = os.path.join(output_folder, new_filename)
        
        try:
            # 复制文件
            shutil.copy2(original_path, new_path)
            copied_count += 1
            copied_files.append({
                'original_path': original_path,
                'new_path': new_path,
                'confidence': row['prob_malignant']
            })
            
        except Exception as e:
            print(f"复制文件 {original_path} 时出错: {e}")
    
    print(f"成功复制 {copied_count} 个文件到 {output_folder}")
    
    # 保存复制文件的信息
    if copied_files:
        copied_df = pd.DataFrame(copied_files)
        copied_df_path = os.path.join(output_folder, 'malignant_images_info.csv')
        copied_df.to_csv(copied_df_path, index=False)
        print(f"复制文件的信息已保存到 {copied_df_path}")
    
    return copied_count

## 测试推理代码

### 给图片加上前缀

In [1]:
import os
import shutil

def add_prefix_to_images(directory_path, prefix="cjin.svs_"):
    """
    为指定目录下的所有图片文件添加前缀
    
    参数:
        directory_path: 图片文件所在的目录路径
        prefix: 要添加的前缀，默认为'cjin.svs_'
    
    返回:
        重命名的文件数量
    """
    # 支持的图片文件扩展名
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.gif')
    
    # 确保目录存在
    if not os.path.exists(directory_path):
        print(f"错误: 目录 '{directory_path}' 不存在")
        return 0
    
    # 获取目录中的所有文件
    files = os.listdir(directory_path)
    
    # 筛选出图片文件
    image_files = [f for f in files if f.lower().endswith(image_extensions)]
    
    if not image_files:
        print(f"目录 '{directory_path}' 中没有找到图片文件")
        return 0
    
    print(f"在目录 '{directory_path}' 中找到 {len(image_files)} 个图片文件")
    
    # 重命名文件
    renamed_count = 0
    for filename in image_files:
        # 如果文件名已经有前缀，跳过
        if filename.startswith(prefix):
            print(f"跳过已有前缀的文件: {filename}")
            continue
        
        old_path = os.path.join(directory_path, filename)
        new_filename = prefix + filename
        new_path = os.path.join(directory_path, new_filename)
        
        try:
            # 重命名文件
            os.rename(old_path, new_path)
            renamed_count += 1
            print(f"已重命名: {filename} -> {new_filename}")
        except Exception as e:
            print(f"重命名文件 '{filename}' 时出错: {e}")
    
    print(f"\n成功重命名 {renamed_count} 个文件")
    return renamed_count

# 指定目录路径
directory_path = r"G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\newdata"

# 调用函数添加前缀
add_prefix_to_images(directory_path)

在目录 'G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\newdata' 中找到 6 个图片文件
已重命名: tile_1_26_0_conf0.65.png -> cjin.svs_tile_1_26_0_conf0.65.png
已重命名: tile_1_26_1_conf0.59.png -> cjin.svs_tile_1_26_1_conf0.59.png
已重命名: tile_1_27_0_conf0.80.png -> cjin.svs_tile_1_27_0_conf0.80.png
已重命名: tile_2_19_0_conf0.73.png -> cjin.svs_tile_2_19_0_conf0.73.png
已重命名: tile_2_20_0_conf0.83.png -> cjin.svs_tile_2_20_0_conf0.83.png
已重命名: tile_2_20_1_conf0.44.png -> cjin.svs_tile_2_20_1_conf0.44.png

成功重命名 6 个文件


6

## 推理代码（jupyter版本）

In [6]:
"""
使用训练好的DenseNet161模型对新数据进行推理并为恶性预测添加置信度标记
- 预测判断：torch.max（等价于固定阈值0.5），二分类标准做法
- 文件名置信度：保留4位小数原始数值
- CONFIDENCE_THRESHOLD：只用于筛选哪些恶性图片值得复制
"""

import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import shutil

from shilab_classifier.models.model_definitions import create_model
from shilab_classifier.training.preprocessing import pad_to_square_transform

# ============================================================
# 配置参数
# ============================================================
MODEL_NAME   = "DenseNet161"
MODEL_PATH   = r"G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\train_val_models\DenseNet161\densenet161_fold_1.pth"
NEW_DATA_DIR = r"G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\newdata"
OUTPUT_DIR   = r"G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\prediction_results"
MALIGNANT_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'malignant_images')

BATCH_SIZE  = 16
NUM_WORKERS = 0
NUM_CLASSES = 2
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# ⚠️ 必须和训练时完全一致！
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ⚠️ 必须和训练时完全一致！
IDX_TO_CLASS = {0: "benign", 1: "malignant"}

# 恶性图片复制的置信度筛选阈值
# 只影响"哪些恶性图片被复制到输出文件夹"，不影响预测判断本身
CONFIDENCE_THRESHOLD = 0.5


# ============================================================
# 数据加载类
# ============================================================
class UnlabeledImageDataset(Dataset):
    """处理单文件夹结构的无标签图像数据集"""

    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform   = transform
        exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
        self.image_files = sorted([
            f for f in os.listdir(folder_path)
            if f.lower().endswith(exts)
        ])
        self.samples = [(os.path.join(folder_path, f), 0) for f in self.image_files]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.samples[idx][0]
        image    = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, 0


# ============================================================
# 模型权重加载（兼容多种保存格式）
# ============================================================
def load_model_weights(model, model_path, device):
    checkpoint = torch.load(model_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print("已加载 checkpoint['model_state_dict']")
    elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
        print("已加载 checkpoint['state_dict']")
    else:
        model.load_state_dict(checkpoint)
        print("已加载直接保存的 state_dict")
    return model


# ============================================================
# 主函数
# ============================================================
def predict_new_data():
    """对新收集的数据进行预测并为恶性预测添加置信度标记"""
    print(f"\n{'='*60}")
    print(f"使用 {MODEL_NAME} 模型预测新数据")
    print(f"{'='*60}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(MALIGNANT_OUTPUT_DIR, exist_ok=True)

    # ----------------------------------------------------------
    # 1. 加载模型
    # ----------------------------------------------------------
    print(f"加载模型权重: {MODEL_PATH}")
    model = create_model(MODEL_NAME, num_classes=NUM_CLASSES)
    model = load_model_weights(model, MODEL_PATH, DEVICE)
    model = model.to(DEVICE)
    model.eval()

    # ----------------------------------------------------------
    # 2. 数据预处理（必须和训练时完全一致）
    # ----------------------------------------------------------
    print(f"使用标准化参数 - 均值: {MEAN}, 标准差: {STD}")
    transform = transforms.Compose([
        transforms.Lambda(pad_to_square_transform),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ])

    # ----------------------------------------------------------
    # 3. 加载数据
    # ----------------------------------------------------------
    print(f"加载新数据: {NEW_DATA_DIR}")

    # 过滤隐藏文件夹（如 .ipynb_checkpoints）再判断目录结构
    has_subfolders = any(
        os.path.isdir(os.path.join(NEW_DATA_DIR, d))
        for d in os.listdir(NEW_DATA_DIR)
        if not d.startswith('.')
    )

    if has_subfolders:
        print("检测到子文件夹结构，使用ImageFolder加载数据...")
        print("⚠️  注意：请确认下方打印的类别映射和训练时一致！")
        try:
            dataset    = datasets.ImageFolder(NEW_DATA_DIR, transform=transform)
            dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                                    shuffle=False, num_workers=NUM_WORKERS)
            print(f"找到 {len(dataset)} 张图片，分为 {len(dataset.class_to_idx)} 个类别")
            for class_name, idx in dataset.class_to_idx.items():
                print(f"  - 文件夹类别 {idx}: {class_name}")
        except Exception as e:
            print(f"加载数据时出错: {e}")
            return
    else:
        print("检测到单一文件夹结构，使用自定义数据集加载...")
        dataset    = UnlabeledImageDataset(NEW_DATA_DIR, transform=transform)
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=NUM_WORKERS)
        print(f"找到 {len(dataset)} 张图片")

    if len(dataset) == 0:
        print("错误：没有找到任何图片，请检查路径和文件格式。")
        return

    # ----------------------------------------------------------
    # 4. 推理
    # ----------------------------------------------------------
    print("\n开始预测...")
    all_paths  = []
    all_logits = []
    all_probs  = []
    all_preds  = []

    with torch.no_grad():
        for batch_idx, (inputs, _) in enumerate(dataloader):
            if batch_idx % 10 == 0:
                print(f"  预测批次 {batch_idx + 1}/{len(dataloader)}")

            # 获取当前批次图像路径
            batch_size_actual = inputs.size(0)
            start_idx   = batch_idx * BATCH_SIZE
            batch_paths = [
                dataset.samples[i][0]
                for i in range(start_idx, min(start_idx + batch_size_actual, len(dataset)))
            ]

            inputs  = inputs.to(DEVICE)
            outputs = model(inputs)

            # 兼容 Inception 等多输出模型
            if isinstance(outputs, tuple):
                outputs = outputs[0]

            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            all_paths.extend(batch_paths)
            all_logits.append(outputs.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_preds.append(predicted.cpu().numpy())

    # ----------------------------------------------------------
    # 5. 整理结果
    # ----------------------------------------------------------
    all_logits = np.concatenate(all_logits, axis=0)
    all_probs  = np.concatenate(all_probs,  axis=0)
    all_preds  = np.concatenate(all_preds,  axis=0)

    results_df = pd.DataFrame({
        'image_path'     : all_paths,
        'filename'       : [os.path.basename(p) for p in all_paths],
        'predicted_idx'  : all_preds,
        'predicted_class': [IDX_TO_CLASS[int(idx)] for idx in all_preds],
        'logit_benign'   : all_logits[:, 0],
        'logit_malignant': all_logits[:, 1],
        'prob_benign'    : all_probs[:, 0],
        'prob_malignant' : all_probs[:, 1],
    })

    results_path = os.path.join(OUTPUT_DIR, 'prediction_results.csv')
    results_df.to_csv(results_path, index=False, encoding='utf-8-sig')
    print(f"\n预测结果已保存到 {results_path}")

    # 统计
    class_counts = results_df['predicted_class'].value_counts()
    print("\n预测类别统计:")
    for class_name, count in class_counts.items():
        print(f"  - {class_name}: {count} 张 ({count / len(results_df) * 100:.1f}%)")

    # ----------------------------------------------------------
    # 6. 处理恶性图片
    # ----------------------------------------------------------
    print("\n开始处理恶性预测图片...")
    process_malignant_predictions(results_df)

    # ----------------------------------------------------------
    # 7. 绘制概率分布图
    # ----------------------------------------------------------
    plt.figure(figsize=(10, 6))
    plt.hist(results_df['prob_malignant'], bins=20, alpha=0.7,
             color='steelblue', edgecolor='white')
    plt.axvline(x=CONFIDENCE_THRESHOLD, color='red', linestyle='--',
                label=f'筛选阈值 = {CONFIDENCE_THRESHOLD}')
    plt.title('恶性预测概率分布')
    plt.xlabel('恶性概率')
    plt.ylabel('图片数量')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    prob_hist_path = os.path.join(OUTPUT_DIR, 'malignant_probability_distribution.png')
    plt.savefig(prob_hist_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"预测概率分布图已保存到 {prob_hist_path}")

    print("\n✅ 预测完成！")
    return results_df


# ============================================================
# 处理恶性预测图片
# ============================================================
def process_malignant_predictions(results_df):
    """
    复制恶性预测图片到输出目录，文件名附带4位小数置信度
    筛选条件：predicted_class == 'malignant' 且 prob_malignant >= CONFIDENCE_THRESHOLD
    """
    malignant_predictions = results_df[
        (results_df['predicted_class'] == 'malignant') &
        (results_df['prob_malignant'] >= CONFIDENCE_THRESHOLD)
    ]

    print(f"找到 {len(malignant_predictions)} 张恶性预测图片（置信度 ≥ {CONFIDENCE_THRESHOLD}）")

    if len(malignant_predictions) == 0:
        print("没有找到符合条件的恶性预测图片")
        return

    processed_count    = 0
    renamed_files_info = []

    for _, row in malignant_predictions.iterrows():
        original_path = row['image_path']
        confidence    = row['prob_malignant']

        name, ext = os.path.splitext(os.path.basename(original_path))

        # ✅ 保留4位小数原始数值，格式统一且可读
        # 示例：0.9521 → "cell_001_malignant_0.9521.jpg"
        #       1.0000 → "cell_002_malignant_1.0000.jpg"
        confidence_str = f"{confidence:.4f}"
        new_filename   = f"{name}_malignant_{confidence_str}{ext}"
        new_path       = os.path.join(MALIGNANT_OUTPUT_DIR, new_filename)

        try:
            shutil.copy2(original_path, new_path)
            processed_count += 1
            renamed_files_info.append({
                'original_path': original_path,
                'new_path'     : new_path,
                'confidence'   : confidence,
            })
        except Exception as e:
            print(f"  ⚠️  处理文件 {original_path} 时出错: {e}")

    if renamed_files_info:
        renamed_df   = pd.DataFrame(renamed_files_info)
        renamed_path = os.path.join(OUTPUT_DIR, 'malignant_images_with_confidence.csv')
        renamed_df.to_csv(renamed_path, index=False, encoding='utf-8-sig')
        print(f"已处理 {processed_count} 张恶性图片，详情保存到 {renamed_path}")
        print(f"带置信度的恶性图片已保存到 {MALIGNANT_OUTPUT_DIR}")


# ============================================================
# 主程序入口
# ============================================================
if __name__ == "__main__":
    predict_new_data()


使用 DenseNet161 模型预测新数据
加载模型权重: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\train_val_models\DenseNet161\densenet161_fold_1.pth
已加载直接保存的 state_dict
使用标准化参数 - 均值: [0.485, 0.456, 0.406], 标准差: [0.229, 0.224, 0.225]
加载新数据: G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\newdata
检测到单一文件夹结构，使用自定义数据集加载...
找到 6 张图片

开始预测...
  预测批次 1/1

预测结果已保存到 G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\prediction_results\prediction_results.csv

预测类别统计:
  - benign: 6 张 (100.0%)

开始处理恶性预测图片...
找到 0 张恶性预测图片（置信度 ≥ 0.5）
没有找到符合条件的恶性预测图片


C:\Users\lillian\OneDrive\ipykernel_20832\2146267974.py:242: UserWarning: Glyph 24694 (\N{CJK UNIFIED IDEOGRAPH-6076}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\lillian\OneDrive\ipykernel_20832\2146267974.py:242: UserWarning: Glyph 24615 (\N{CJK UNIFIED IDEOGRAPH-6027}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\lillian\OneDrive\ipykernel_20832\2146267974.py:242: UserWarning: Glyph 27010 (\N{CJK UNIFIED IDEOGRAPH-6982}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\lillian\OneDrive\ipykernel_20832\2146267974.py:242: UserWarning: Glyph 29575 (\N{CJK UNIFIED IDEOGRAPH-7387}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\lillian\OneDrive\ipykernel_20832\2146267974.py:242: UserWarning: Glyph 22270 (\N{CJK UNIFIED IDEOGRAPH-56FE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\lillian\OneDrive\ipykernel_20832\2146267974.py:242: UserWarning: Glyph 29255 (\N{CJK UNIFIED IDEOGRAPH-7247}) missing fro

预测概率分布图已保存到 G:\04PhD\02Project\03Urine\StudyTest\result\classification_model\UR_SingleCell\prediction_results\malignant_probability_distribution.png

✅ 预测完成！


## 推理代码（传参版本）

In [ ]:
#!/usr/bin/env python
"""
使用训练好的DenseNet161模型对新数据进行推理并为恶性预测添加置信度标记
- 预测判断：torch.max（等价于固定阈值0.5），二分类标准做法
- 文件名置信度：保留4位小数原始数值
- CONFIDENCE_THRESHOLD：只用于筛选哪些恶性图片值得复制
"""

import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import shutil
import argparse

from shilab_classifier.models.model_definitions import create_model
from shilab_classifier.training.preprocessing import pad_to_square_transform


# ============================================================
# 数据加载类
# ============================================================
class UnlabeledImageDataset(Dataset):
    """处理单文件夹结构的无标签图像数据集"""

    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform   = transform
        exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
        self.image_files = sorted([
            f for f in os.listdir(folder_path)
            if f.lower().endswith(exts)
        ])
        self.samples = [(os.path.join(folder_path, f), 0) for f in self.image_files]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.samples[idx][0]
        image    = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, 0


# ============================================================
# 模型权重加载（兼容多种保存格式）
# ============================================================
def load_model_weights(model, model_path, device):
    checkpoint = torch.load(model_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print("已加载 checkpoint['model_state_dict']")
    elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
        print("已加载 checkpoint['state_dict']")
    else:
        model.load_state_dict(checkpoint)
        print("已加载直接保存的 state_dict")
    return model


# ============================================================
# 处理恶性预测图片
# ============================================================
def process_malignant_predictions(results_df, malignant_output_dir, confidence_threshold, output_dir):
    """
    复制恶性预测图片到输出目录，文件名附带4位小数置信度
    筛选条件：predicted_class == 'malignant' 且 prob_malignant >= confidence_threshold
    """
    malignant_predictions = results_df[
        (results_df['predicted_class'] == 'malignant') &
        (results_df['prob_malignant'] >= confidence_threshold)
    ]

    print(f"找到 {len(malignant_predictions)} 张恶性预测图片（置信度 ≥ {confidence_threshold}）")

    if len(malignant_predictions) == 0:
        print("没有找到符合条件的恶性预测图片")
        return

    processed_count    = 0
    renamed_files_info = []

    for _, row in malignant_predictions.iterrows():
        original_path = row['image_path']
        confidence    = row['prob_malignant']

        name, ext = os.path.splitext(os.path.basename(original_path))

        # ✅ 保留4位小数原始数值，格式统一且可读
        # 示例：0.9521 → "cell_001_malignant_0.9521.jpg"
        #       1.0000 → "cell_002_malignant_1.0000.jpg"
        confidence_str = f"{confidence:.4f}"
        new_filename   = f"{name}_malignant_{confidence_str}{ext}"
        new_path       = os.path.join(malignant_output_dir, new_filename)

        try:
            shutil.copy2(original_path, new_path)
            processed_count += 1
            renamed_files_info.append({
                'original_path': original_path,
                'new_path'     : new_path,
                'confidence'   : confidence,
            })
        except Exception as e:
            print(f"  ⚠️  处理文件 {original_path} 时出错: {e}")

    if renamed_files_info:
        renamed_df   = pd.DataFrame(renamed_files_info)
        renamed_path = os.path.join(output_dir, 'malignant_images_with_confidence.csv')
        renamed_df.to_csv(renamed_path, index=False, encoding='utf-8-sig')
        print(f"已处理 {processed_count} 张恶性图片，详情保存到 {renamed_path}")
        print(f"带置信度的恶性图片已保存到 {malignant_output_dir}")


# ============================================================
# 主函数
# ============================================================
def predict_new_data(
    model_name="DenseNet161",
    model_path=None,
    input_dir=None,
    output_dir=None,
    batch_size=16,
    num_workers=0,
    confidence_threshold=0.5,
    device=None
):
    """对新收集的数据进行预测并为恶性预测添加置信度标记"""
    
    # 参数检查
    if model_path is None or input_dir is None or output_dir is None:
        raise ValueError("必须提供模型路径、输入目录和输出目录")
    
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    
    num_classes = 2
    idx_to_class = {0: "benign", 1: "malignant"}
    
    # 标准化参数（必须和训练时所确定模型折一致）
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    
    # 创建输出目录
    malignant_output_dir = os.path.join(output_dir, 'malignant_images')
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(malignant_output_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"使用 {model_name} 模型预测新数据")
    print(f"{'='*60}")
    print(f"模型路径: {model_path}")
    print(f"输入目录: {input_dir}")
    print(f"输出目录: {output_dir}")
    print(f"设备: {device}")
    print(f"批次大小: {batch_size}")
    print(f"置信度阈值: {confidence_threshold}")

    # ----------------------------------------------------------
    # 1. 加载模型
    # ----------------------------------------------------------
    print(f"加载模型权重: {model_path}")
    model = create_model(model_name, num_classes=num_classes)
    model = load_model_weights(model, model_path, device)
    model = model.to(device)
    model.eval()

    # ----------------------------------------------------------
    # 2. 数据预处理（必须和训练时完全一致）
    # ----------------------------------------------------------
    print(f"使用标准化参数 - 均值: {mean}, 标准差: {std}")
    transform = transforms.Compose([
        transforms.Lambda(pad_to_square_transform),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    # ----------------------------------------------------------
    # 3. 加载数据
    # ----------------------------------------------------------
    print(f"加载新数据: {input_dir}")

    # 过滤隐藏文件夹（如 .ipynb_checkpoints）再判断目录结构
    has_subfolders = any(
        os.path.isdir(os.path.join(input_dir, d))
        for d in os.listdir(input_dir)
        if not d.startswith('.')
    )

    if has_subfolders:
        print("检测到子文件夹结构，使用ImageFolder加载数据...")
        print("⚠️  注意：请确认下方打印的类别映射和训练时一致！")
        try:
            dataset    = datasets.ImageFolder(input_dir, transform=transform)
            dataloader = DataLoader(dataset, batch_size=batch_size,
                                    shuffle=False, num_workers=num_workers)
            print(f"找到 {len(dataset)} 张图片，分为 {len(dataset.class_to_idx)} 个类别")
            for class_name, idx in dataset.class_to_idx.items():
                print(f"  - 文件夹类别 {idx}: {class_name}")
        except Exception as e:
            print(f"加载数据时出错: {e}")
            return None
    else:
        print("检测到单一文件夹结构，使用自定义数据集加载...")
        dataset    = UnlabeledImageDataset(input_dir, transform=transform)
        dataloader = DataLoader(dataset, batch_size=batch_size,
                                shuffle=False, num_workers=num_workers)
        print(f"找到 {len(dataset)} 张图片")

    if len(dataset) == 0:
        print("错误：没有找到任何图片，请检查路径和文件格式。")
        return None

    # ----------------------------------------------------------
    # 4. 推理
    # ----------------------------------------------------------
    print("\n开始预测...")
    all_paths  = []
    all_logits = []
    all_probs  = []
    all_preds  = []

    with torch.no_grad():
        for batch_idx, (inputs, _) in enumerate(dataloader):
            if batch_idx % 10 == 0:
                print(f"  预测批次 {batch_idx + 1}/{len(dataloader)}")

            # 获取当前批次图像路径
            batch_size_actual = inputs.size(0)
            start_idx   = batch_idx * batch_size
            batch_paths = [
                dataset.samples[i][0]
                for i in range(start_idx, min(start_idx + batch_size_actual, len(dataset)))
            ]

            inputs  = inputs.to(device)
            outputs = model(inputs)

            # 兼容 Inception 等多输出模型
            if isinstance(outputs, tuple):
                outputs = outputs[0]

            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            all_paths.extend(batch_paths)
            all_logits.append(outputs.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_preds.append(predicted.cpu().numpy())

    # ----------------------------------------------------------
    # 5. 整理结果
    # ----------------------------------------------------------
    all_logits = np.concatenate(all_logits, axis=0)
    all_probs  = np.concatenate(all_probs,  axis=0)
    all_preds  = np.concatenate(all_preds,  axis=0)

    results_df = pd.DataFrame({
        'image_path'     : all_paths,
        'filename'       : [os.path.basename(p) for p in all_paths],
        'predicted_idx'  : all_preds,
        'predicted_class': [idx_to_class[int(idx)] for idx in all_preds],
        'logit_benign'   : all_logits[:, 0],
        'logit_malignant': all_logits[:, 1],
        'prob_benign'    : all_probs[:, 0],
        'prob_malignant' : all_probs[:, 1],
    })

    results_path = os.path.join(output_dir, 'prediction_results.csv')
    results_df.to_csv(results_path, index=False, encoding='utf-8-sig')
    print(f"\n预测结果已保存到 {results_path}")

    # 统计
    class_counts = results_df['predicted_class'].value_counts()
    print("\n预测类别统计:")
    for class_name, count in class_counts.items():
        print(f"  - {class_name}: {count} 张 ({count / len(results_df) * 100:.1f}%)")

    # ----------------------------------------------------------
    # 6. 处理恶性图片
    # ----------------------------------------------------------
    print("\n开始处理恶性预测图片...")
    process_malignant_predictions(results_df, malignant_output_dir, confidence_threshold, output_dir)

    # ----------------------------------------------------------
    # 7. 绘制概率分布图
    # ----------------------------------------------------------
    plt.figure(figsize=(10, 6))
    plt.hist(results_df['prob_malignant'], bins=20, alpha=0.7,
             color='steelblue', edgecolor='white')
    plt.axvline(x=confidence_threshold, color='red', linestyle='--',
                label=f'Threshold = {confidence_threshold}')
    plt.title('Malignant Prediction Probability Distribution')
    plt.xlabel('Malignant probability')
    plt.ylabel('Number of images')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    prob_hist_path = os.path.join(output_dir, 'malignant_probability_distribution.png')
    plt.savefig(prob_hist_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"预测概率分布图已保存到 {prob_hist_path}")

    print("\n✅ 预测完成！")
    return results_df


# ============================================================
# 命令行参数解析
# ============================================================
def parse_args():
    parser = argparse.ArgumentParser(description='使用训练好的模型对新数据进行推理')
    
    parser.add_argument('-m', '--model', type=str, default="DenseNet161",
                        help='模型名称 (默认: DenseNet161)')
    
    parser.add_argument('-w', '--weights', type=str, required=True,
                        help='模型权重文件路径 (.pth)')
    
    parser.add_argument('-i', '--input', type=str, required=True,
                        help='输入数据目录路径')
    
    parser.add_argument('-o', '--output', type=str, required=True,
                        help='输出结果目录路径')
    
    parser.add_argument('-b', '--batch-size', type=int, default=16,
                        help='批处理大小 (默认: 16)')
    
    parser.add_argument('-j', '--workers', type=int, default=0,
                        help='数据加载线程数 (默认: 0)')
    
    parser.add_argument('-t', '--threshold', type=float, default=0.5,
                        help='恶性预测置信度阈值 (默认: 0.5)')    
    
    return parser.parse_args()


# ============================================================
# 主程序入口
# ============================================================
if __name__ == "__main__":
    args = parse_args()
    
    # 设置设备
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 运行预测
    predict_new_data(
        model_name=args.model,
        model_path=args.weights,
        input_dir=args.input,
        output_dir=args.output,
        batch_size=args.batch_size,
        num_workers=args.workers,
        confidence_threshold=args.threshold,
        device=device
    )

### 每个病人进行推理

In [ ]:
#!/usr/bin/env python
"""
使用训练好的DenseNet161模型对新数据进行推理并为恶性预测添加置信度标记
- 预测判断：torch.max（等价于固定阈值0.5），二分类标准做法
- 文件名置信度：保留4位小数原始数值
- CONFIDENCE_THRESHOLD：只用于筛选哪些恶性图片值得复制
"""

import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import shutil
import argparse

from shilab_classifier.models.model_definitions import create_model
from shilab_classifier.training.preprocessing import pad_to_square_transform, pad_to_square_299_transform


# ============================================================
# 数据加载类
# ============================================================
class UnlabeledImageDataset(Dataset):
    """处理单文件夹结构的无标签图像数据集"""

    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform   = transform
        exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
        self.image_files = sorted([
            f for f in os.listdir(folder_path)
            if f.lower().endswith(exts)
        ])
        self.samples = [(os.path.join(folder_path, f), 0) for f in self.image_files]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.samples[idx][0]
        image    = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, 0


# ============================================================
# 模型权重加载（兼容多种保存格式）
# ============================================================
def load_model_weights(model, model_path, device):
    checkpoint = torch.load(model_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print("已加载 checkpoint['model_state_dict']")
    elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
        print("已加载 checkpoint['state_dict']")
    else:
        model.load_state_dict(checkpoint)
        print("已加载直接保存的 state_dict")
    return model


# ============================================================
# 处理恶性预测图片
# ============================================================
def process_malignant_predictions(results_df, malignant_output_dir, confidence_threshold, output_dir):
    """
    复制恶性预测图片到输出目录，文件名附带4位小数置信度
    筛选条件：predicted_class == 'malignant' 且 prob_malignant >= confidence_threshold
    """
    malignant_predictions = results_df[
        (results_df['predicted_class'] == 'malignant') &
        (results_df['prob_malignant'] >= confidence_threshold)
    ]

    print(f"找到 {len(malignant_predictions)} 张恶性预测图片（置信度 ≥ {confidence_threshold}）")

    if len(malignant_predictions) == 0:
        print("没有找到符合条件的恶性预测图片")
        return

    processed_count    = 0
    renamed_files_info = []

    for _, row in malignant_predictions.iterrows():
        original_path = row['image_path']
        confidence    = row['prob_malignant']

        name, ext = os.path.splitext(os.path.basename(original_path))

        # ✅ 保留4位小数原始数值，格式统一且可读
        # 示例：0.9521 → "cell_001_malignant_0.9521.jpg"
        #       1.0000 → "cell_002_malignant_1.0000.jpg"
        confidence_str = f"{confidence:.4f}"
        new_filename   = f"{name}_malignant_{confidence_str}{ext}"
        new_path       = os.path.join(malignant_output_dir, new_filename)

        try:
            shutil.copy2(original_path, new_path)
            processed_count += 1
            renamed_files_info.append({
                'original_path': original_path,
                'new_path'     : new_path,
                'confidence'   : confidence,
            })
        except Exception as e:
            print(f"  ⚠️  处理文件 {original_path} 时出错: {e}")

    if renamed_files_info:
        renamed_df   = pd.DataFrame(renamed_files_info)
        renamed_path = os.path.join(output_dir, 'malignant_images_with_confidence.csv')
        renamed_df.to_csv(renamed_path, index=False, encoding='utf-8-sig')
        print(f"已处理 {processed_count} 张恶性图片，详情保存到 {renamed_path}")
        print(f"带置信度的恶性图片已保存到 {malignant_output_dir}")


# ============================================================
# 主函数
# ============================================================
def predict_new_data(
    model_name="DenseNet161",
    model_path=None,
    input_dir=None,
    output_dir=None,
    batch_size=16,
    num_workers=0,
    confidence_threshold=0.5,
    device=None,
    mean=None,
    std=None
):
    """对新收集的数据进行预测并为恶性预测添加置信度标记"""
    
    # 参数检查
    if model_path is None or input_dir is None or output_dir is None:
        raise ValueError("必须提供模型路径、输入目录和输出目录")
    
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    
    num_classes = 2
    idx_to_class = {0: "benign", 1: "malignant"}
    
    # 标准化参数（如果未提供，则使用ImageNet默认值）
    if mean is None:
        mean = [0.485, 0.456, 0.406]  # ImageNet默认值
    
    if std is None:
        std = [0.229, 0.224, 0.225]  # ImageNet默认值
    
    # 创建输出目录
    malignant_output_dir = os.path.join(output_dir, 'malignant_images')
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(malignant_output_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"使用 {model_name} 模型预测新数据")
    print(f"{'='*60}")
    print(f"模型路径: {model_path}")
    print(f"输入目录: {input_dir}")
    print(f"输出目录: {output_dir}")
    print(f"设备: {device}")
    print(f"批次大小: {batch_size}")
    print(f"置信度阈值: {confidence_threshold}")

    # ----------------------------------------------------------
    # 1. 加载模型
    # ----------------------------------------------------------
    print(f"加载模型权重: {model_path}")
    model = create_model(model_name, num_classes=num_classes)
    model = load_model_weights(model, model_path, device)
    model = model.to(device)
    model.eval()

    # ----------------------------------------------------------
    # 2. 数据预处理（必须和训练时完全一致）
    # ----------------------------------------------------------
    print(f"使用标准化参数 - 均值: {mean}, 标准差: {std}")
    
    # 选择合适的填充变换
    if 'Inception' in model_name:
        pad_transform = pad_to_square_299_transform
    else:
        pad_transform = pad_to_square_transform
        
    transform = transforms.Compose([
        transforms.Lambda(pad_transform),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    # ----------------------------------------------------------
    # 3. 加载数据
    # ----------------------------------------------------------
    print(f"加载新数据: {input_dir}")

    # 过滤隐藏文件夹（如 .ipynb_checkpoints）再判断目录结构
    has_subfolders = any(
        os.path.isdir(os.path.join(input_dir, d))
        for d in os.listdir(input_dir)
        if not d.startswith('.')
    )

    if has_subfolders:
        print("检测到子文件夹结构，使用ImageFolder加载数据...")
        print("⚠️  注意：请确认下方打印的类别映射和训练时一致！")
        try:
            dataset    = datasets.ImageFolder(input_dir, transform=transform)
            dataloader = DataLoader(dataset, batch_size=batch_size,
                                    shuffle=False, num_workers=num_workers)
            print(f"找到 {len(dataset)} 张图片，分为 {len(dataset.class_to_idx)} 个类别")
            for class_name, idx in dataset.class_to_idx.items():
                print(f"  - 文件夹类别 {idx}: {class_name}")
        except Exception as e:
            print(f"加载数据时出错: {e}")
            return None
    else:
        print("检测到单一文件夹结构，使用自定义数据集加载...")
        dataset    = UnlabeledImageDataset(input_dir, transform=transform)
        dataloader = DataLoader(dataset, batch_size=batch_size,
                                shuffle=False, num_workers=num_workers)
        print(f"找到 {len(dataset)} 张图片")

    if len(dataset) == 0:
        print("错误：没有找到任何图片，请检查路径和文件格式。")
        return None

    # ----------------------------------------------------------
    # 4. 推理
    # ----------------------------------------------------------
    print("\n开始预测...")
    all_paths  = []
    all_logits = []
    all_probs  = []
    all_preds  = []

    with torch.no_grad():
        for batch_idx, (inputs, _) in enumerate(dataloader):
            if batch_idx % 10 == 0:
                print(f"  预测批次 {batch_idx + 1}/{len(dataloader)}")

            # 获取当前批次图像路径
            batch_size_actual = inputs.size(0)
            start_idx   = batch_idx * batch_size
            batch_paths = [
                dataset.samples[i][0]
                for i in range(start_idx, min(start_idx + batch_size_actual, len(dataset)))
            ]

            inputs  = inputs.to(device)
            outputs = model(inputs)

            # 兼容 Inception 等多输出模型
            if isinstance(outputs, tuple):
                outputs = outputs[0]

            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            all_paths.extend(batch_paths)
            all_logits.append(outputs.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_preds.append(predicted.cpu().numpy())

    # ----------------------------------------------------------
    # 5. 整理结果
    # ----------------------------------------------------------
    all_logits = np.concatenate(all_logits, axis=0)
    all_probs  = np.concatenate(all_probs,  axis=0)
    all_preds  = np.concatenate(all_preds,  axis=0)

    results_df = pd.DataFrame({
        'image_path'     : all_paths,
        'filename'       : [os.path.basename(p) for p in all_paths],
        'predicted_idx'  : all_preds,
        'predicted_class': [idx_to_class[int(idx)] for idx in all_preds],
        'logit_benign'   : all_logits[:, 0],
        'logit_malignant': all_logits[:, 1],
        'prob_benign'    : all_probs[:, 0],
        'prob_malignant' : all_probs[:, 1],
    })

    results_path = os.path.join(output_dir, 'prediction_results.csv')
    results_df.to_csv(results_path, index=False, encoding='utf-8-sig')
    print(f"\n预测结果已保存到 {results_path}")

    # 统计
    class_counts = results_df['predicted_class'].value_counts()
    print("\n预测类别统计:")
    for class_name, count in class_counts.items():
        print(f"  - {class_name}: {count} 张 ({count / len(results_df) * 100:.1f}%)")

    # ----------------------------------------------------------
    # 6. 处理恶性图片
    # ----------------------------------------------------------
    print("\n开始处理恶性预测图片...")
    process_malignant_predictions(results_df, malignant_output_dir, confidence_threshold, output_dir)

    # ----------------------------------------------------------
    # 7. 绘制概率分布图
    # ----------------------------------------------------------
    plt.figure(figsize=(10, 6))
    plt.hist(results_df['prob_malignant'], bins=20, alpha=0.7,
             color='steelblue', edgecolor='white')
    plt.axvline(x=confidence_threshold, color='red', linestyle='--',
                label=f'Threshold = {confidence_threshold}')
    plt.title('Malignant Prediction Probability Distribution')
    plt.xlabel('Malignant probability')
    plt.ylabel('Number of images')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    prob_hist_path = os.path.join(output_dir, 'malignant_probability_distribution.png')
    plt.savefig(prob_hist_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"预测概率分布图已保存到 {prob_hist_path}")

    print("\n✅ 预测完成！")
    return results_df


# ============================================================
# 命令行参数解析
# ============================================================
def parse_args():
    parser = argparse.ArgumentParser(description='使用训练好的模型对新数据进行推理')
    
    parser.add_argument('-m', '--model', type=str, default="DenseNet161",
                        help='模型名称 (默认: DenseNet161)')
    
    parser.add_argument('-w', '--weights', type=str, required=True,
                        help='模型权重文件路径 (.pth)')
    
    parser.add_argument('-i', '--input', type=str, required=True,
                        help='输入数据目录路径')
    
    parser.add_argument('-o', '--output', type=str, required=True,
                        help='输出结果目录路径')
    
    parser.add_argument('-b', '--batch-size', type=int, default=16,
                        help='批处理大小 (默认: 16)')
    
    parser.add_argument('-j', '--workers', type=int, default=0,
                        help='数据加载线程数 (默认: 0)')
    
    parser.add_argument('-t', '--threshold', type=float, default=0.5,
                        help='恶性预测置信度阈值 (默认: 0.5)')
    
    parser.add_argument('--mean', type=float, nargs=3, default=None,
                        help='标准化均值 [R G B] (默认: ImageNet均值 [0.485, 0.456, 0.406])')
    
    parser.add_argument('--std', type=float, nargs=3, default=None,
                        help='标准化标准差 [R G B] (默认: ImageNet标准差 [0.229, 0.224, 0.225])')
    
    return parser.parse_args()


# ============================================================
# 主程序入口
# ============================================================
if __name__ == "__main__":
    args = parse_args()
    
    # 设置设备
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 运行预测
    predict_new_data(
        model_name=args.model,
        model_path=args.weights,
        input_dir=args.input,
        output_dir=args.output,
        batch_size=args.batch_size,
        num_workers=args.workers,
        confidence_threshold=args.threshold,
        device=device,
        mean=args.mean,
        std=args.std
    )